<a href="https://colab.research.google.com/github/dinooooooi/dinooooi/blob/main/crawling_game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install requests beautifulsoup4 pandas tqdm lxml openpyxl

In [2]:
import re
import time
import os
import requests
import pandas as pd

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from urllib.parse import quote

In [ ]:
# =========================================
# PUBG Mobile Liquipedia 설정
# =========================================

GAME_SLUG = "pubgmobile"
GAME_LABEL = "PUBG Mobile"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

USER_AGENT = "KeynsgResearch/0.1 (eonconesg@gmail.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

# Liquipedia 무료 API 제한 고려
REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/pubgmobile
https://liquipedia.net/pubgmobile/api.php


In [ ]:
def clean_text(text):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def make_page_url(title):
    title = title or ""
    return f"{BASE_URL}/{quote(title.replace(' ', '_'))}"


def api_get(params, sleep=REQUEST_SLEEP):
    params = {
        **params,
        "format": "json"
    }

    r = requests.get(
        API_URL,
        params=params,
        headers=HEADERS,
        timeout=40
    )

    time.sleep(sleep)

    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    data = r.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data

In [ ]:
data = api_get({
    "action": "query",
    "meta": "siteinfo",
    "siprop": "general"
})

data["query"]["general"]["sitename"]

'Liquipedia PUBG Mobile Wiki'

In [ ]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    PUBG Mobile Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

#pubg 모바일부터

In [ ]:
PUBGM_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korea team",
    "Korean team",
    "Korea esports",
    "PUBG Mobile Korea",
    "PMPS Korea",
    "PUBG Mobile Pro Series Korea",
    "PMSC Korea",
    "PWS Korea",
    "DWG KIA",
    "Dplus KIA",
    "DRX",
    "Gen.G Esports",
    "Nongshim RedForce",
    "ROX",
    "T1",
]

In [ ]:
search_rows = []

for keyword in PUBGM_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

pubgm_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(pubgm_search_df))
display(pubgm_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: Korean / offset=150
검색: Korean / offset=200
검색: Korean / offset=250
검색: Korean / offset=300
검색: Korean / offset=350
검색: Korean / offset=400
검색: Korean / offset=450
검색: Korean / offset=500
검색: Korean / offset=550
검색: Korean / offset=600
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: South Korean / offset=100
검색: South Korean / offset=150
검색: South Korean / offset=200
검색: South Korean / offset=250
검색: South Korean /

,game_slug,game_label,title,snippet,pageid,url,search_keyword
0,pubgmobile,PUBG Mobile,India Korea Invitational/2023,Links Teams Number of teams: 16 India - Korea ...,88200,https://liquipedia.net/pubgmobile/India_Korea_...,korea
1,pubgmobile,PUBG Mobile,Dplus,Dplus Team Information Location: South Korea R...,78671,https://liquipedia.net/pubgmobile/Dplus,korea
2,pubgmobile,PUBG Mobile,PUBG Mobile Global Championship/2025/Points/Korea,Total points earned by teams in the Finals sta...,94176,https://liquipedia.net/pubgmobile/PUBG_Mobile_...,korea
3,pubgmobile,PUBG Mobile,KRAFTON,Company Information Location: South Korea Head...,80000,https://liquipedia.net/pubgmobile/KRAFTON,korea
4,pubgmobile,PUBG Mobile,PUBG Mobile Rivals Cup/2021,Rivals Cup: Korea vs Japan is a tournament tha...,78170,https://liquipedia.net/pubgmobile/PUBG_Mobile_...,korea
...,...,...,...,...,...,...,...
95,pubgmobile,PUBG Mobile,PUBG Mobile Pro Series/2022/Season 4,PUBG Mobile Pro Series 2022 Season 4 League In...,83025,https://liquipedia.net/pubgmobile/PUBG_Mobile_...,korea
96,pubgmobile,PUBG Mobile,Uriel,Player Information Name: 안수찬 Romanized Name: A...,86527,https://liquipedia.net/pubgmobile/Uriel,korea
97,pubgmobile,PUBG Mobile,PUBG Mobile Pro Series/2024/Season 0,ROX ANGRY Notes 1 ANGRY acquired the slot of O...,89273,https://liquipedia.net/pubgmobile/PUBG_Mobile_...,korea
98,pubgmobile,PUBG Mobile,PUBG Mobile Open Challenge/2021,Mobile Open Challenge 2021 League Information ...,76343,https://liquipedia.net/pubgmobile/PUBG_Mobile_...,korea


In [ ]:
def build_team_info_candidate_df(search_df):
    def is_team_info_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        # 선수 페이지 제외
        if "player information" in text:
            return False

        # 팀 페이지 조건
        if "team information" not in text:
            return False

        # 한국 팀 조건
        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
        ]

        if not any(term in text for term in korea_terms):
            return False

        return True

    df = search_df[
        search_df.apply(is_team_info_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Team Information 기반 한국 팀 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df


pubgm_team_candidate_df = build_team_info_candidate_df(pubgm_search_df)
display(pubgm_team_candidate_df[["title", "snippet", "search_keyword", "url"]].head(200))

Team Information 기반 한국 팀 후보 수: 20
예상 parse 소요 시간: 10.3 분
예상 parse 소요 시간: 0.17 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea R...,korea,https://liquipedia.net/pubgmobile/Dplus
1,ASA KOREA,ASA KOREA Team Information Location: South Kor...,korea,https://liquipedia.net/pubgmobile/ASA_KOREA
2,DUKSAN Esports,DUKSAN Esports Team Information Location: Sout...,korea,https://liquipedia.net/pubgmobile/DUKSAN_Esports
3,DRX,DRX Team Information Location: South Korea Reg...,korea,https://liquipedia.net/pubgmobile/DRX
4,Team Square,Team Square Team Information Location: South K...,korea,https://liquipedia.net/pubgmobile/Team_Square
5,Nongshim RedForce,Nongshim RedForce Team Information Location: S...,korea,https://liquipedia.net/pubgmobile/Nongshim_Red...
6,Zz,zz Team Information Location: South Korea Regi...,korea,https://liquipedia.net/pubgmobile/Zz
7,EmTek StormX,emTek StormX Team Information Location: South ...,korea,https://liquipedia.net/pubgmobile/EmTek_StormX
8,T1,T1 Team Information Location: South Korea Unit...,korea,https://liquipedia.net/pubgmobile/T1
9,Gen.G Esports,Gen.G Esports Team Information Location: South...,korea,https://liquipedia.net/pubgmobile/Gen.G_Esports


In [ ]:
def build_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "korea",
        "south korea",
        "dplus",
        "kia",
        "dwg",
        "drx",
        "gen.g",
        "nongshim",
        "redforce",
        "rox",
        "t1",
        "danawa",
        "eagle owls",
        "bnk",
        "freecs",
        "geng",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "split",
        "playoffs",
        "cup",
        "championship",
        "tournament",
        "qualifier",
        "showmatch",
        "match history",
        "results",
        "statistics",
        "roster",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "patch",
        "standings",
        "group stage",
        "regular season",
        "participants",
        "matches",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
        ]

        if not any(term in text for term in korea_terms):
            return False

        if not any(keyword in text for keyword in team_like_keywords):
            return False

        return True

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("완화 한국 팀 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df


pubgm_team_candidate_loose_df = build_team_candidate_loose_df(pubgm_search_df)
display(pubgm_team_candidate_loose_df[["title", "snippet", "search_keyword", "url"]].head(200))

완화 한국 팀 후보 수: 142
예상 parse 소요 시간: 73.4 분
예상 parse 소요 시간: 1.22 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea R...,korea,https://liquipedia.net/pubgmobile/Dplus
1,KRAFTON,Company Information Location: South Korea Head...,korea,https://liquipedia.net/pubgmobile/KRAFTON
2,PUBG Mobile,Information Developer: Lightspeed &amp; Quantu...,korea,https://liquipedia.net/pubgmobile/PUBG_Mobile
3,ASA KOREA,ASA KOREA Team Information Location: South Kor...,korea,https://liquipedia.net/pubgmobile/ASA_KOREA
4,DUKSAN Esports,DUKSAN Esports Team Information Location: Sout...,korea,https://liquipedia.net/pubgmobile/DUKSAN_Esports
...,...,...,...,...
137,WEST Challenge,19 Russia 1 / 81 (1%) KnowMe 19 Slovenia 1 / 8...,korea,https://liquipedia.net/pubgmobile/WEST_Challenge
138,Aesor,19esports 2022-02-04 2022-08-30 Boring Protoco...,korea,https://liquipedia.net/pubgmobile/Aesor
139,Joey,2021-03-23 XSET 2021-07-08 2022-06-21 Lazarus ...,korea,https://liquipedia.net/pubgmobile/Joey
140,Peacekeeper Elite,"Streamer Championship S1 Sep 15–18, 2020 $8,86...",korea,https://liquipedia.net/pubgmobile/Peacekeeper_...


In [ ]:
# =========================================
# PUBG Mobile 완화 후보에서 팀 후보만 재정제
# source: pubgm_team_candidate_loose_df
# output: pubgm_dedup_df2
# 페이지 불러오기 X, API 호출 X
# =========================================

source_df = pubgm_team_candidate_loose_df.copy()

pubgm_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "clan",

    # 한국/국내 PUBG Mobile 팀명성 키워드
    "dplus",
    "kia",
    "dwg",
    "dk",
    "drx",
    "gen.g",
    "geng",
    "nongshim",
    "redforce",
    "rox",
    "t1",
    "danawa",
    "freecs",
    "eagle owls",
    "bnk",
    "fearx",
    "zz",
    "zzang",
    "emtek",
    "emtek stormx",
    "stormx",
    "ds gaming",
    "game pt",
    "sga",
    "onside",
    "sentinel",
    "gnl",
    "slasher",
    "mir",
    "old ocean",
    "azla",
    "pentagram",
    "busan",
    "bespa",
    "daegu",
    "daejeon",
    "gyeonggi",
    "gyeongnam",
    "jecheon",
    "yangju",
]

pubgm_bad_title_patterns = [
    "/",                         # 하위 문서 제거
    r"\b20\d{2}\b",              # 2024, 2025 등 시즌/대회 문서 제거
    "season",
    "split",
    "playoffs",
    "cup",
    "championship",
    "tournament",
    "qualifier",
    "showmatch",
    "match history",
    "results",
    "statistics",
    "roster",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "patch",
    "standings",
    "group stage",
    "regular season",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
    "weekly",
    "super weekend",
]

pubgm_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

pubgm_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
]


def is_refined_pubgm_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    # 1. 선수/대회/리그성 페이지 제외
    if any(marker in text for marker in pubgm_bad_page_markers):
        return False

    # 2. 제목에 시즌/대회/하위문서성 패턴 있으면 제외
    for pattern in pubgm_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    # 3. 한국 관련 조건
    if not any(term in text for term in pubgm_korea_terms):
        return False

    # 4. Team Information이 있으면 강한 팀 후보로 통과
    if "team information" in text:
        return True

    # 5. Team Information이 없는 경우에는 팀명성 키워드가 있어야 통과
    if any(keyword in text for keyword in pubgm_team_name_keywords):
        return True

    return False


pubgm_dedup_df2 = source_df[
    source_df.apply(is_refined_pubgm_team_candidate, axis=1)
].copy().reset_index(drop=True)

# 제목 정리
pubgm_dedup_df2["title_clean"] = (
    pubgm_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

# 완전 중복 제거
pubgm_dedup_df2 = pubgm_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("PUBG Mobile 완화 후보 수:", len(pubgm_team_candidate_loose_df))
print("재정제 후 pubgm_dedup_df2 후보 수:", len(pubgm_dedup_df2))
print("예상 parse 소요 시간:", round(len(pubgm_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(pubgm_dedup_df2) * 31 / 3600, 2), "시간")

display(pubgm_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

PUBG Mobile 완화 후보 수: 142
재정제 후 pubgm_dedup_df2 후보 수: 64
예상 parse 소요 시간: 33.1 분
예상 parse 소요 시간: 0.55 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea R...,korea,https://liquipedia.net/pubgmobile/Dplus
1,ASA KOREA,ASA KOREA Team Information Location: South Kor...,korea,https://liquipedia.net/pubgmobile/ASA_KOREA
2,DUKSAN Esports,DUKSAN Esports Team Information Location: Sout...,korea,https://liquipedia.net/pubgmobile/DUKSAN_Esports
3,DRX,DRX Team Information Location: South Korea Reg...,korea,https://liquipedia.net/pubgmobile/DRX
4,PUBG Mobile DS League,tournament organized by DS Gaming which featur...,korea,https://liquipedia.net/pubgmobile/PUBG_Mobile_...
...,...,...,...,...
59,NOVA Monster Shield,Hong Kong esports organization. Known for thei...,korea,https://liquipedia.net/pubgmobile/NOVA_Monster...
60,Naoto,@SLZOZISAN TAMR4 / @koiking02 Naoto / @RtyG_ S...,korea,https://liquipedia.net/pubgmobile/Naoto
61,NEW STATE MOBILE Open Challenge,KP Gaming NsRG NEW RAVE Hons Team TOP ALV Lege...,korea,https://liquipedia.net/pubgmobile/NEW_STATE_MO...
62,Aesor,19esports 2022-02-04 2022-08-30 Boring Protoco...,korea,https://liquipedia.net/pubgmobile/Aesor


In [ ]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [ ]:
def extract_pubg_players_from_team_html_v1(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    roster_keywords = [
        "Active Roster",
        "Current Roster",
        "Player Roster",
        "Roster",
        "Players",
        "Former Players",
        "Former Roster",
        "Organization",
    ]

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page",
    ]

    def get_section_heading(el):
        prev = el.find_previous(["h2", "h3", "h4"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def extract_country_from_cell_or_row(el):
        countries = []
        for img in el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if value and value not in countries:
                    if not value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                        countries.append(value)
        return ", ".join(countries)

    def clean_date(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    for table_idx, table in enumerate(soup.find_all("table")):
        heading = get_section_heading(table)
        table_text = clean_text(table.get_text(" ", strip=True))
        combined = f"{heading} {table_text}".lower()

        if not any(k.lower() in combined for k in roster_keywords):
            continue

        trs = table.find_all("tr")
        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            header_hits = [
                "id", "player", "name", "role", "position",
                "join date", "joined", "leave date", "left",
                "nationality", "country"
            ]

            if any(h in lowered for h in header_hits):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = []
        for h in headers:
            h_low = h.lower()
            if h_low in ["id", "player"]:
                normalized_headers.append("player_nickname")
            elif h_low == "name":
                normalized_headers.append("player_real_name")
            elif h_low in ["position", "role"]:
                normalized_headers.append("position_or_role")
            elif h_low in ["join date", "joined"]:
                normalized_headers.append("join_date")
            elif h_low in ["leave date", "left"]:
                normalized_headers.append("leave_date")
            elif h_low in ["country", "nationality"]:
                normalized_headers.append("nationality")
            else:
                normalized_headers.append(h_low.replace(" ", "_"))

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])
            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))
            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date"]:
                    value = clean_date(value)

                row_data[col] = value

            player_links = []

            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                player_name = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if not player_name:
                    continue

                if player_name.lower() in bad_link_keywords:
                    continue

                if any(x in href for x in ["/File:", "/Category:", "/Special:", "/Help:", "/Template:"]):
                    continue

                if len(player_name) > 40:
                    continue

                if re.search(r"(season|playoffs|championship|cup|tournament|league|series)", player_name, flags=re.I):
                    continue

                player_links.append({
                    "player_nickname_from_link": player_name,
                    "player_page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "player_url": "https://liquipedia.net" + href,
                })

            if player_links:
                main_link = player_links[0]
                player_nickname = row_data.get("player_nickname", "") or main_link["player_nickname_from_link"]
                player_page = main_link["player_page"]
                player_url = main_link["player_url"]
            else:
                player_nickname = row_data.get("player_nickname", "")
                player_page = ""
                player_url = ""

            if not player_nickname:
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_country_from_cell_or_row(tr)
            )

            rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": heading,
                "table_index": table_idx,
                "player_nickname": player_nickname,
                "player_real_name": row_data.get("player_real_name", ""),
                "position_or_role": row_data.get("position_or_role", ""),
                "join_date": row_data.get("join_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "nationality": nationality,
                "player_page": player_page,
                "player_url": player_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=["team_title", "section_heading", "player_nickname", "player_real_name", "join_date"]
    ).reset_index(drop=True)

    return df

In [ ]:
# =========================================
# 1. 한 팀 테스트
# =========================================

test_team = pubgm_dedup_df2["title"].iloc[0]
page = get_page_html(test_team)

test_players_df = extract_pubg_players_from_team_html_v1(test_team, page["html"])

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("추출 선수 row 수:", len(test_players_df))

display(test_players_df.head(50))

테스트 팀: Dplus
페이지 존재 여부: True
추출 선수 row 수: 0


""


In [ ]:
GAME_SLUG = "pubgmobile"
GAME_LABEL = "PUBG Mobile"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

print(BASE_URL)
print(API_URL)

https://liquipedia.net/pubgmobile
https://liquipedia.net/pubgmobile/api.php


In [ ]:
def extract_pubgmobile_players_from_team_html_v3(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    # 선수 섹션만 대상으로 함. Organization/Staff는 일단 제외.
    player_section_keywords = [
        "Player Roster",
        "Active",
        "Former",
        "Former Players",
        "Current Roster",
        "Active Roster",
        "Players",
    ]

    exclude_section_keywords = [
        "Organization",
        "Staff",
        "Former Organization",
        "Achievements",
        "Awards",
        "Statistics",
        "Gallery",
        "References",
    ]

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "pubg mobile", "liquipedia",
    ]

    def get_section_heading(el):
        prev = el.find_previous(["h2", "h3", "h4"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_country_from_element(el):
        countries = []
        for img in el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if value and value not in countries:
                    if not value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                        countries.append(value)
        return ", ".join(countries)

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "player_nickname"
        if h == "name":
            return "player_real_name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position_or_role"

        return h.replace(" ", "_")

    def is_bad_player_link(player_name, href):
        name = clean_text(player_name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/File:", "/Category:", "/Special:", "/Help:", "/Template:",
            "/Portal:", "/Liquipedia:", "/User:"
        ]):
            return True

        if len(name) > 40:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|pmgc|pmps|kel)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    for table_idx, table in enumerate(soup.find_all("table")):
        heading = get_section_heading(table)
        table_text = clean_text(table.get_text(" ", strip=True))
        combined = f"{heading} {table_text}".lower()

        # Organization / Staff 표는 제외
        if any(k.lower() in combined for k in exclude_section_keywords):
            # 단, Former Players라는 단어가 있으면 선수 표일 수 있으므로 살림
            if "former players" not in combined and "player roster" not in combined:
                continue

        # 선수 관련 표만
        if not any(k.lower() in combined for k in player_section_keywords):
            continue

        trs = table.find_all("tr")
        if not trs:
            continue

        headers = []
        header_row_idx = None

        # 헤더 찾기: ID / Name / Join Date / Leave Date 등
        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])
            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))
            if not row_text:
                continue

            # 연도 구분행, Show All 같은 행 제외
            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            # 선수 링크 추출
            player_links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                player_name = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_player_link(player_name, href):
                    continue

                player_links.append({
                    "player_nickname_from_link": player_name,
                    "player_page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "player_url": "https://liquipedia.net" + href,
                })

            # 첫 번째 유효 링크를 선수 링크로 사용
            if player_links:
                main_link = player_links[0]
                player_nickname = row_data.get("player_nickname", "") or main_link["player_nickname_from_link"]
                player_page = main_link["player_page"]
                player_url = main_link["player_url"]
            else:
                player_nickname = row_data.get("player_nickname", "")
                player_page = ""
                player_url = ""

            if not player_nickname:
                continue

            # 팀 자기 자신 링크 제거
            if player_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_country_from_element(tr)
            )

            rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": heading,
                "table_index": table_idx,
                "player_nickname": player_nickname,
                "player_real_name": row_data.get("player_real_name", ""),
                "position_or_role": row_data.get("position_or_role", ""),
                "join_date": row_data.get("join_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "player_page": player_page,
                "player_url": player_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "section_heading",
            "player_nickname",
            "player_real_name",
            "join_date",
            "leave_date",
        ]
    ).reset_index(drop=True)

    return df

In [ ]:
test_team = "Dplus"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_players_df = extract_pubgmobile_players_from_team_html_v3(
        test_team,
        page["html"]
    )

    print("추출 선수 row 수:", len(test_players_df))

    if len(test_players_df) > 0:
        display(test_players_df[[
            "team_title",
            "section_heading",
            "player_nickname",
            "player_real_name",
            "join_date",
            "leave_date",
            "new_team",
            "nationality",
            "player_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 선수 row가 추출되지 않았습니다.")

테스트 팀: Dplus
페이지 존재 여부: True
에러: None
추출 선수 row 수: 24


,team_title,section_heading,player_nickname,player_real_name,join_date,leave_date,new_team,nationality,player_url
0,Dplus,Active,Nolbu,Song Soo-an,2023-07-28 [ 11 ],,,South Korea,https://liquipedia.net/pubgmobile/Nolbu
1,Dplus,Active,chpz,Jung Yoo-chan,2024-01-13 [ 14 ],,,South Korea,https://liquipedia.net/pubgmobile/Chpz
2,Dplus,Active,Auto,Hong Samuel,2026-01-17 [ 23 ],,,South Korea,https://liquipedia.net/pubgmobile/Auto
3,Dplus,Active,Cyxae,Choi Young-jae,2026-01-17 [ 23 ],,,South Korea,https://liquipedia.net/pubgmobile/Cyxae
4,Dplus,Active,Porico,Kim Si-hyun,2026-01-17 [ 23 ],,,South Korea,https://liquipedia.net/pubgmobile/Porico
5,Dplus,Active,FAVIAN,Park Sang-cheol,2026-04-12 [ 25 ],,,South Korea,https://liquipedia.net/pubgmobile/FAVIAN
6,Dplus,Former,ONBA,Oh Seung-ju,2022-06-28 [ 3 ],2022-09-16 [ 5 ],Team Square,"South Korea, Team Square",https://liquipedia.net/pubgmobile/ONBA
7,Dplus,Former,Sayden,Jeon Min-jae,2023-01-06 [ 7 ],2023-11-10 [ 12 ],Gen.G Esports,"South Korea, Gen.G Esports",https://liquipedia.net/pubgmobile/Sayden
8,Dplus,Former,Porico,Kim Si-hyun,2023-01-30 [ 9 ],2023-05-26 [ 10 ],Dplus,"South Korea, Dplus",https://liquipedia.net/pubgmobile/Porico
9,Dplus,Former,JUNI,Kim Kyung-jun,2021-11-17 [ 2 ],2023-01-27 [ 8 ],vanquish,"South Korea, vanquish",https://liquipedia.net/pubgmobile/JUNI


In [ ]:
def crawl_pubgmobile_teams_and_players_v3(candidate_df, output_prefix, team_type):
    team_rows = []
    player_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 선수 추출 v3"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:south korea" in infobox_lower.replace(" ", "")
                or "region:south koreakorea" in infobox_lower.replace(" ", "")
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "game_slug": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                players_df = extract_pubgmobile_players_from_team_html_v3(
                    page["title"],
                    page["html"]
                )

                if len(players_df) > 0:
                    players_df["team_display_title"] = page["display_title"]
                    players_df["team_url"] = page["url"]
                    players_df["team_type"] = team_type
                    player_dfs.append(players_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if player_dfs:
        players_raw_df = pd.concat(player_dfs, ignore_index=True)
    else:
        players_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 선수 row 수:", len(players_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    players_raw_path = f"/content/{output_prefix}_players_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    players_raw_df.to_csv(players_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(players_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, players_raw_df, errors_df

In [ ]:
pubgm_teams_checked_df, pubgm_teams_final_df, pubgm_players_raw_df, pubgm_errors_df = crawl_pubgmobile_teams_and_players_v3(
    candidate_df=pubgm_dedup_df2,
    output_prefix="pubgmobile_korean_mainteam_refined_v3",
    team_type="main_team_refined"
)

display(pubgm_teams_final_df.head(100))
display(pubgm_players_raw_df.head(100))

크롤링 대상 후보 수: 64
예상 소요 시간: 33.1 분
예상 소요 시간: 0.55 시간


pubgmobile_korean_mainteam_refined_v3 팀 검증 + 선수 추출 v3:   0%|          | 0/64 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 25
추출 선수 row 수: 417
에러 수: 0
저장 완료
/content/pubgmobile_korean_mainteam_refined_v3_teams_checked.csv
/content/pubgmobile_korean_mainteam_refined_v3_teams_final.csv
/content/pubgmobile_korean_mainteam_refined_v3_players_raw.csv
/content/pubgmobile_korean_mainteam_refined_v3_errors.csv


,game_slug,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,pubgmobile,PUBG Mobile,Dplus,Dplus,https://liquipedia.net/pubgmobile/Dplus,True,True,[ e ][ h ] Dplus Team Information Location: So...,None,main_team_refined
1,pubgmobile,PUBG Mobile,ASA KOREA,ASA KOREA,https://liquipedia.net/pubgmobile/ASA_KOREA,True,True,[ e ][ h ] ASA KOREA Team Information Location...,None,main_team_refined
2,pubgmobile,PUBG Mobile,DUKSAN Esports,DUKSAN Esports,https://liquipedia.net/pubgmobile/DUKSAN_Esports,True,True,[ e ][ h ] DUKSAN Esports Team Information Loc...,None,main_team_refined
3,pubgmobile,PUBG Mobile,DRX,DRX,https://liquipedia.net/pubgmobile/DRX,True,True,[ e ][ h ] DRX Team Information Location: Sout...,None,main_team_refined
4,pubgmobile,PUBG Mobile,Team Square,Team Square,https://liquipedia.net/pubgmobile/Team_Square,True,True,[ e ][ h ] Team Square Team Information Locati...,None,main_team_refined
5,pubgmobile,PUBG Mobile,Nongshim RedForce,Nongshim RedForce,https://liquipedia.net/pubgmobile/Nongshim_Red...,True,True,[ e ][ h ] Nongshim RedForce Team Information ...,None,main_team_refined
6,pubgmobile,PUBG Mobile,Zz,zz,https://liquipedia.net/pubgmobile/Zz,True,True,[ e ][ h ] zz Team Information Location: South...,None,main_team_refined
7,pubgmobile,PUBG Mobile,EmTek StormX,emTek StormX,https://liquipedia.net/pubgmobile/EmTek_StormX,True,True,[ e ][ h ] emTek StormX Team Information Locat...,None,main_team_refined
8,pubgmobile,PUBG Mobile,T1,T1,https://liquipedia.net/pubgmobile/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
9,pubgmobile,PUBG Mobile,Gen.G Esports,Gen.G Esports,https://liquipedia.net/pubgmobile/Gen.G_Esports,True,True,[ e ][ h ] Gen.G Esports Team Information Loca...,None,main_team_refined


,game_slug,game_label,team_title,team_url,section_heading,table_index,player_nickname,player_real_name,position_or_role,join_date,leave_date,new_team,nationality,player_page,player_url,row_text,team_display_title,team_type
0,pubgmobile,PUBG Mobile,Dplus,https://liquipedia.net/pubgmobile/Dplus,Active,1,Nolbu,Song Soo-an,,2023-07-28 [ 11 ],,,South Korea,Nolbu,https://liquipedia.net/pubgmobile/Nolbu,Nolbu Song Soo-an 2023-07-28 [ 11 ],Dplus,main_team_refined
1,pubgmobile,PUBG Mobile,Dplus,https://liquipedia.net/pubgmobile/Dplus,Active,1,chpz,Jung Yoo-chan,,2024-01-13 [ 14 ],,,South Korea,Chpz,https://liquipedia.net/pubgmobile/Chpz,chpz Jung Yoo-chan 2024-01-13 [ 14 ],Dplus,main_team_refined
2,pubgmobile,PUBG Mobile,Dplus,https://liquipedia.net/pubgmobile/Dplus,Active,1,Auto,Hong Samuel,,2026-01-17 [ 23 ],,,South Korea,Auto,https://liquipedia.net/pubgmobile/Auto,Auto Hong Samuel 2026-01-17 [ 23 ],Dplus,main_team_refined
3,pubgmobile,PUBG Mobile,Dplus,https://liquipedia.net/pubgmobile/Dplus,Active,1,Cyxae,Choi Young-jae,,2026-01-17 [ 23 ],,,South Korea,Cyxae,https://liquipedia.net/pubgmobile/Cyxae,Cyxae Choi Young-jae 2026-01-17 [ 23 ],Dplus,main_team_refined
4,pubgmobile,PUBG Mobile,Dplus,https://liquipedia.net/pubgmobile/Dplus,Active,1,Porico,Kim Si-hyun,,2026-01-17 [ 23 ],,,South Korea,Porico,https://liquipedia.net/pubgmobile/Porico,Porico Kim Si-hyun 2026-01-17 [ 23 ],Dplus,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,pubgmobile,PUBG Mobile,DRX,https://liquipedia.net/pubgmobile/DRX,Former,3,SOEZ,Song Ho-jin,,2024-06-05 [ 1 ],2026-01-07 [ 10 ],Nongshim RedForce,"South Korea, Nongshim RedForce",SOEZ,https://liquipedia.net/pubgmobile/SOEZ,SOEZ Song Ho-jin 2024-06-05 [ 1 ] 2026-01-07 [...,DRX,main_team_refined
96,pubgmobile,PUBG Mobile,DRX,https://liquipedia.net/pubgmobile/DRX,Active,4,Buddha,Tsuneaki Takeda,,2022-01-07,,,Japan,index.php?title=Buddha&action=edit&redlink=1,https://liquipedia.net/pubgmobile/index.php?ti...,Buddha Tsuneaki Takeda Director of Business De...,DRX,main_team_refined
97,pubgmobile,PUBG Mobile,DRX,https://liquipedia.net/pubgmobile/DRX,Active,4,Can,Yang Sun-il,,2022-01-07,,,South Korea,index.php?title=Can&action=edit&redlink=1,https://liquipedia.net/pubgmobile/index.php?ti...,Can Yang Sun-il CEO 2022-01-07,DRX,main_team_refined
98,pubgmobile,PUBG Mobile,DRX,https://liquipedia.net/pubgmobile/DRX,Active,4,Dopani,Lim Hyun-seok,,2022-01-07,,,South Korea,index.php?title=Dopani&action=edit&redlink=1,https://liquipedia.net/pubgmobile/index.php?ti...,Dopani Lim Hyun-seok COO 2022-01-07,DRX,main_team_refined


In [ ]:
def make_unique_players(players_raw_df):
    if len(players_raw_df) == 0:
        return pd.DataFrame()

    df = players_raw_df.copy()

    if "player_url" not in df.columns:
        df["player_url"] = ""

    if "player_nickname" not in df.columns:
        df["player_nickname"] = ""

    df["player_unique_key"] = (
        df["player_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["player_unique_key"] == ""

    df.loc[empty_url_mask, "player_unique_key"] = (
        df.loc[empty_url_mask, "player_nickname"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    unique_df = df.drop_duplicates(
        subset=["player_unique_key"]
    ).reset_index(drop=True)

    return unique_df

In [ ]:
pubgm_players_unique_df = make_unique_players(pubgm_players_raw_df)

print("PUBG Mobile 최종 한국 팀 수:", len(pubgm_teams_final_df))
print("PUBG Mobile 선수 raw row 수:", len(pubgm_players_raw_df))
print("PUBG Mobile 고유 선수 수:", len(pubgm_players_unique_df))

display(pubgm_players_unique_df[[
    "team_title",
    "player_nickname",
    "player_real_name",
    "join_date",
    "leave_date",
    "new_team",
    "nationality",
    "player_url"
]].head(100))

PUBG Mobile 최종 한국 팀 수: 25
PUBG Mobile 선수 raw row 수: 417
PUBG Mobile 고유 선수 수: 236


,team_title,player_nickname,player_real_name,join_date,leave_date,new_team,nationality,player_url
0,Dplus,Nolbu,Song Soo-an,2023-07-28 [ 11 ],,,South Korea,https://liquipedia.net/pubgmobile/Nolbu
1,Dplus,chpz,Jung Yoo-chan,2024-01-13 [ 14 ],,,South Korea,https://liquipedia.net/pubgmobile/Chpz
2,Dplus,Auto,Hong Samuel,2026-01-17 [ 23 ],,,South Korea,https://liquipedia.net/pubgmobile/Auto
3,Dplus,Cyxae,Choi Young-jae,2026-01-17 [ 23 ],,,South Korea,https://liquipedia.net/pubgmobile/Cyxae
4,Dplus,Porico,Kim Si-hyun,2026-01-17 [ 23 ],,,South Korea,https://liquipedia.net/pubgmobile/Porico
...,...,...,...,...,...,...,...,...
95,EmTek StormX,OGG,,2023-05-24 [ 4 ],2023-08-11 [ 5 ],,South Korea,https://liquipedia.net/pubgmobile/index.php?ti...
96,EmTek StormX,Yeon,Lim Dong-gun,2023-02-21 [ 3 ],2024-02-22 [ 7 ],,South Korea,https://liquipedia.net/pubgmobile/index.php?ti...
97,EmTek StormX,Cat7,,2024-09-11 [ 8 ],2025-02-06 [ 9 ],GNL Esports,"South Korea, GNL Esports",https://liquipedia.net/pubgmobile/index.php?ti...
98,EmTek StormX,Clutch,Yoon Sung-hyeon,2023-02-21 [ 3 ],2025-02-06 [ 9 ],GAME PT,"South Korea, GAME PT",https://liquipedia.net/pubgmobile/Clutch


In [ ]:
pubgm_teams_final_df.to_csv(
    "/content/pubgmobile_korean_teams_final_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_players_raw_df.to_csv(
    "/content/pubgmobile_korean_players_raw_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_players_unique_df.to_csv(
    "/content/pubgmobile_korean_players_unique_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_errors_df.to_csv(
    "/content/pubgmobile_korean_extract_errors_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/pubgmobile_korean_teams_final_v3.csv")
print("/content/pubgmobile_korean_players_raw_v3.csv")
print("/content/pubgmobile_korean_players_unique_v3.csv")
print("/content/pubgmobile_korean_extract_errors_v3.csv")

저장 완료
/content/pubgmobile_korean_teams_final_v3.csv
/content/pubgmobile_korean_players_raw_v3.csv
/content/pubgmobile_korean_players_unique_v3.csv
/content/pubgmobile_korean_extract_errors_v3.csv


In [ ]:
# =========================================
# 4. 고유 선수 목록 생성 함수
# =========================================

def make_unique_players(players_raw_df):
    if len(players_raw_df) == 0:
        return pd.DataFrame()

    df = players_raw_df.copy()

    if "player_url" not in df.columns:
        df["player_url"] = ""

    if "player_nickname" not in df.columns:
        df["player_nickname"] = ""

    df["player_unique_key"] = (
        df["player_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["player_unique_key"] == ""

    df.loc[empty_url_mask, "player_unique_key"] = (
        df.loc[empty_url_mask, "player_nickname"]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    unique_df = df.drop_duplicates(
        subset=["player_unique_key"]
    ).reset_index(drop=True)

    return unique_df

In [ ]:
# =========================================
# 4-2. PUBG Mobile 고유 선수 목록 생성
# =========================================

pubgm_players_unique_df = make_unique_players(pubgm_players_raw_df)

print("PUBG Mobile 최종 한국 팀 수:", len(pubgm_teams_final_df))
print("PUBG Mobile 선수 raw row 수:", len(pubgm_players_raw_df))
print("PUBG Mobile 고유 선수 수:", len(pubgm_players_unique_df))

display(pubgm_players_unique_df.head(100))

PUBG Mobile 최종 한국 팀 수: 25
PUBG Mobile 선수 raw row 수: 0
PUBG Mobile 고유 선수 수: 0


""


In [ ]:
# =========================================
# 5. 최종 저장
# =========================================

pubgm_dedup_df2.to_csv(
    "/content/pubgmobile_korean_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_teams_checked_df.to_csv(
    "/content/pubgmobile_korean_teams_checked.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_teams_final_df.to_csv(
    "/content/pubgmobile_korean_teams_final.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_players_raw_df.to_csv(
    "/content/pubgmobile_korean_players_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_players_unique_df.to_csv(
    "/content/pubgmobile_korean_players_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

pubgm_errors_df.to_csv(
    "/content/pubgmobile_korean_extract_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/pubgmobile_korean_team_candidates_refined.csv")
print("/content/pubgmobile_korean_teams_checked.csv")
print("/content/pubgmobile_korean_teams_final.csv")
print("/content/pubgmobile_korean_players_raw.csv")
print("/content/pubgmobile_korean_players_unique.csv")
print("/content/pubgmobile_korean_extract_errors.csv")

저장 완료
/content/pubgmobile_korean_team_candidates_refined.csv
/content/pubgmobile_korean_teams_checked.csv
/content/pubgmobile_korean_teams_final.csv
/content/pubgmobile_korean_players_raw.csv
/content/pubgmobile_korean_players_unique.csv
/content/pubgmobile_korean_extract_errors.csv


#PUBG pc 버전 시작

In [ ]:
# =========================================
# PUBG PC Liquipedia 설정
# =========================================

GAME_SLUG = "pubg"
GAME_LABEL = "PUBG PC"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KeynsgResearch/0.1 (eonconesg@gmail.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

# Liquipedia API 제한 고려
REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/pubg
https://liquipedia.net/pubg/api.php


In [ ]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    PUBG PC Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [ ]:
PUBG_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korea team",
    "Korean team",
    "Korea esports",
    "PUBG Korea",
    "PWS Korea",
    "PUBG Weekly Series Korea",
    "PUBG Nations Cup Korea",

    # 한국 PUBG PC 팀명/조직 후보
    "Gen.G Esports",
    "Danawa e-sports",
    "Dplus KIA",
    "Dplus",
    "DWG KIA",
    "DRX",
    "DN FREECS",
    "Afreeca Freecs",
    "Kwangdong Freecs",
    "T1",
    "VSG",
    "Griffin",
    "OP.GG",
    "GHIBLI Esports",
    "GNL ESPORTS",
    "Eagle Owls",
    "KX Gaming",
    "ROX",
    "emTek StormX",
    "OZ Gaming",
    "Genocide",
    "Maru Gaming",
    "Team Quadro",
    "GPS GHIBLI",
    "PENTAGRAM",
    "Divine",
]

In [ ]:
search_rows = []

for keyword in PUBG_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

pubg_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(pubg_search_df))
display(pubg_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: korea / offset=650
검색: korea / offset=700
검색: korea / offset=750
검색: korea / offset=800
검색: korea / offset=850
검색: korea / offset=900
검색: korea / offset=950
max_total 도달: 1000
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: South Korea / offset=350
검색: South Korea / offset=400
검색: South Korea / offset=450
검색: South Korea / offset=500
검색: South Korea / offset=550
검색: South Korea / offset=600
검색: South Korea / offset=650
검색: South Korea / offset=700
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: Korean / offset=150

,game_slug,game_label,title,snippet,pageid,url,search_keyword
0,pubg,PUBG PC,Qualifier Tournaments,"Feb 28, 2026 South Korea 16 participants TBD T...",4444,https://liquipedia.net/pubg/Qualifier_Tournaments,korea
1,pubg,PUBG PC,C-Tier Tournaments,2024 $755 South Korea 16 participants SUPERNOV...,66100,https://liquipedia.net/pubg/C-Tier_Tournaments,korea
2,pubg,PUBG PC,A-Tier Tournaments,Team skipnho Team TGLTN PUBG WEEKLY SERIES: KO...,3291,https://liquipedia.net/pubg/A-Tier_Tournaments,korea
3,pubg,PUBG PC,B-Tier Tournaments,Prize Pool Location P# Winner Runner-up PUBG R...,4387,https://liquipedia.net/pubg/B-Tier_Tournaments,korea
4,pubg,PUBG PC,PUBG WEEKLY SERIES/2024/KOREA/Phase 1,3 by PUBG Korea of PUBG Korea at PWS Phase 1 2...,89017,https://liquipedia.net/pubg/PUBG_WEEKLY_SERIES...,korea
...,...,...,...,...,...,...,...
95,pubg,PUBG PC,AfreecaTV PUBG League/2018/Season 1,PUBG League - Season 1 League Information Seri...,9672,https://liquipedia.net/pubg/AfreecaTV_PUBG_Lea...,korea
96,pubg,PUBG PC,Baegopa,South Korea Region: Asia Approx. Total Winning...,87772,https://liquipedia.net/pubg/Baegopa,korea
97,pubg,PUBG PC,ROCCAT INV,Information Location: South Korea Approx. Tota...,11670,https://liquipedia.net/pubg/ROCCAT_INV,korea
98,pubg,PUBG PC,Yureka,Name: Park Gyu-tae Nationality: South Korea Bo...,70103,https://liquipedia.net/pubg/Yureka,korea


In [ ]:
def build_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "clan",

        # 한국/국내 PUBG PC 팀명성 키워드
        "gen.g",
        "geng",
        "danawa",
        "dplus",
        "kia",
        "dwg",
        "dk",
        "drx",
        "freecs",
        "afreeca",
        "kwangdong",
        "dn freecs",
        "t1",
        "vsg",
        "griffin",
        "op.gg",
        "ghibli",
        "gnl",
        "eagle owls",
        "kx",
        "rox",
        "stormx",
        "emtek",
        "oz",
        "genocide",
        "maru",
        "quadro",
        "gps",
        "gca",
        "gamecoach",
        "beyond stratos",
        "bsg",
        "pentagram",
        "azla",
        "onside",
        "sentinel",
        "mir",
        "old ocean",
        "from",
        "divine",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "split",
        "playoffs",
        "cup",
        "championship",
        "tournament",
        "qualifier",
        "showmatch",
        "match history",
        "results",
        "statistics",
        "roster",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "patch",
        "standings",
        "group stage",
        "regular season",
        "participants",
        "matches",
        "rankings",
        "awards",
        "finals",
        "weekly",
        "series",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
        ]

        if not any(term in text for term in korea_terms):
            return False

        if not any(keyword in text for keyword in team_like_keywords):
            return False

        return True

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("PUBG PC 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [ ]:
pubg_team_candidate_loose_df = build_team_candidate_loose_df(pubg_search_df)

display(pubg_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

PUBG PC 완화 후보 수: 259
예상 parse 소요 시간: 133.8 분
예상 parse 소요 시간: 2.23 시간


,title,snippet,search_keyword,url
0,South Korea,South Korea Team Information Location: South K...,korea,https://liquipedia.net/pubg/South_Korea
1,PUBG Korea League,Afreeca Freecs Fatal PUBG Korea League 2019 - ...,korea,https://liquipedia.net/pubg/PUBG_Korea_League
2,DeToNator.KOREA,DeToNator.KOREA Team Information Location: Sou...,korea,https://liquipedia.net/pubg/DeToNator.KOREA
3,Prince (Korea Team),Prince Team Information Location: South Korea ...,korea,https://liquipedia.net/pubg/Prince_%28Korea_Te...
4,Gen.G Esports,"September, 4th in PUBG Korea League Season 2 F...",korea,https://liquipedia.net/pubg/Gen.G_Esports
...,...,...,...,...
195,Team Square,Team Square Team Information Location: South K...,korea,https://liquipedia.net/pubg/Team_Square
196,Griffin White,Information Location: South Korea Links Histor...,korea,https://liquipedia.net/pubg/Griffin_White
197,Hexa,Sung-yoon Nationality: South Korea Status: Ret...,korea,https://liquipedia.net/pubg/Hexa
198,Qurate,Information Name: 박성주 Romanized Name: Park Seo...,korea,https://liquipedia.net/pubg/Qurate


In [ ]:
# =========================================
# PUBG PC 완화 후보에서 팀 후보만 재정제
# source: pubg_team_candidate_loose_df
# output: pubg_dedup_df2
# 페이지 불러오기 X, API 호출 X
# =========================================

source_df = pubg_team_candidate_loose_df.copy()

pubg_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "clan",

    "gen.g",
    "geng",
    "danawa",
    "dplus",
    "kia",
    "dwg",
    "dk",
    "drx",
    "freecs",
    "afreeca",
    "kwangdong",
    "dn freecs",
    "t1",
    "vsg",
    "griffin",
    "op.gg",
    "ghibli",
    "gnl",
    "eagle owls",
    "kx",
    "rox",
    "stormx",
    "emtek",
    "oz",
    "genocide",
    "maru",
    "quadro",
    "gps",
    "gca",
    "gamecoach",
    "beyond stratos",
    "bsg",
    "pentagram",
    "azla",
    "onside",
    "sentinel",
    "mir",
    "old ocean",
    "from",
    "divine",
]

pubg_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "split",
    "playoffs",
    "cup",
    "championship",
    "tournament",
    "qualifier",
    "showmatch",
    "match history",
    "results",
    "statistics",
    "roster",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "patch",
    "standings",
    "group stage",
    "regular season",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
    "weekly",
    "series",
]

pubg_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

pubg_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
]


def is_refined_pubg_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    # 1. 선수/대회/리그성 페이지 제외
    if any(marker in text for marker in pubg_bad_page_markers):
        return False

    # 2. 제목에 시즌/대회/하위문서성 패턴 있으면 제외
    for pattern in pubg_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    # 3. 한국 관련 조건
    if not any(term in text for term in pubg_korea_terms):
        return False

    # 4. Team Information이 있으면 강한 팀 후보로 통과
    if "team information" in text:
        return True

    # 5. Team Information이 없는 경우에는 팀명성 키워드가 있어야 통과
    if any(keyword in text for keyword in pubg_team_name_keywords):
        return True

    return False


pubg_dedup_df2 = source_df[
    source_df.apply(is_refined_pubg_team_candidate, axis=1)
].copy().reset_index(drop=True)

pubg_dedup_df2["title_clean"] = (
    pubg_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

pubg_dedup_df2 = pubg_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("PUBG PC 완화 후보 수:", len(pubg_team_candidate_loose_df))
print("재정제 후 pubg_dedup_df2 후보 수:", len(pubg_dedup_df2))
print("예상 parse 소요 시간:", round(len(pubg_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(pubg_dedup_df2) * 31 / 3600, 2), "시간")

display(pubg_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

PUBG PC 완화 후보 수: 259
재정제 후 pubg_dedup_df2 후보 수: 259
예상 parse 소요 시간: 133.8 분
예상 parse 소요 시간: 2.23 시간


,title,snippet,search_keyword,url
0,South Korea,South Korea Team Information Location: South K...,korea,https://liquipedia.net/pubg/South_Korea
1,PUBG Korea League,Afreeca Freecs Fatal PUBG Korea League 2019 - ...,korea,https://liquipedia.net/pubg/PUBG_Korea_League
2,DeToNator.KOREA,DeToNator.KOREA Team Information Location: Sou...,korea,https://liquipedia.net/pubg/DeToNator.KOREA
3,Prince (Korea Team),Prince Team Information Location: South Korea ...,korea,https://liquipedia.net/pubg/Prince_%28Korea_Te...
4,Gen.G Esports,"September, 4th in PUBG Korea League Season 2 F...",korea,https://liquipedia.net/pubg/Gen.G_Esports
...,...,...,...,...
195,Team Square,Team Square Team Information Location: South K...,korea,https://liquipedia.net/pubg/Team_Square
196,Griffin White,Information Location: South Korea Links Histor...,korea,https://liquipedia.net/pubg/Griffin_White
197,Hexa,Sung-yoon Nationality: South Korea Status: Ret...,korea,https://liquipedia.net/pubg/Hexa
198,Qurate,Information Name: 박성주 Romanized Name: Park Seo...,korea,https://liquipedia.net/pubg/Qurate


In [ ]:
def extract_pubg_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    player_section_keywords = [
        "Player Roster",
        "Active",
        "Former",
        "Former Players",
        "Current Roster",
        "Active Roster",
        "Players",
    ]

    staff_section_keywords = [
        "Organization",
        "Staff",
        "Former Organization",
    ]

    hard_exclude_keywords = [
        "Achievements",
        "Awards",
        "Statistics",
        "Gallery",
        "References",
        "Earnings Chart",
        "Results",
    ]

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "pubg", "liquipedia",
    ]

    def get_section_heading(el):
        prev = el.find_previous(["h2", "h3", "h4"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_country_from_element(el):
        countries = []
        for img in el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if value and value not in countries:
                    if not value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                        countries.append(value)
        return ", ".join(countries)

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "nickname_or_id"
        if h == "name":
            return "real_name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position_or_role"

        # Liquipedia Organization 표는 빈 헤더가 역할 컬럼인 경우가 있음
        if h == "":
            return "position_or_role"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/File:", "/Category:", "/Special:", "/Help:", "/Template:",
            "/Portal:", "/Liquipedia:", "/User:"
        ]):
            return True

        if len(name) > 60:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|pws|pgc|pgs|pnc|apl|pkl)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def classify_table(heading, table_text):
        combined = f"{heading} {table_text}".lower()

        if any(k.lower() in combined for k in hard_exclude_keywords):
            if not any(k.lower() in combined for k in player_section_keywords + staff_section_keywords):
                return None

        if "organization" in combined or "staff" in combined:
            return "staff"

        if any(k.lower() in combined for k in player_section_keywords):
            return "player"

        return None

    for table_idx, table in enumerate(soup.find_all("table")):
        heading = get_section_heading(table)
        table_text = clean_text(table.get_text(" ", strip=True))

        person_group = classify_table(heading, table_text)
        if person_group is None:
            continue

        trs = table.find_all("tr")
        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        # Organization 표 헤더 예시: ID / Name / [빈칸] / Join Date
        # 빈칸 헤더가 날아가면 역할 컬럼이 누락될 수 있어서 보정
        if person_group == "staff":
            if "position_or_role" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position_or_role"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])
            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))
            if not row_text:
                continue

            # 연도 구분행, Show All 같은 행 제외
            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                nickname_or_id = row_data.get("nickname_or_id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                nickname_or_id = row_data.get("nickname_or_id", "")
                person_page = ""
                person_url = ""

            if not nickname_or_id:
                continue

            # 팀 자기 자신 링크 제거
            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_country_from_element(tr)
            )

            rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": heading,
                "table_index": table_idx,
                "person_group": person_group,  # player / staff
                "nickname_or_id": nickname_or_id,
                "real_name": row_data.get("real_name", ""),
                "position_or_role": row_data.get("position_or_role", ""),
                "join_date": row_data.get("join_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "person_group",
            "section_heading",
            "nickname_or_id",
            "real_name",
            "position_or_role",
            "join_date",
            "leave_date",
        ]
    ).reset_index(drop=True)

    return df

In [ ]:
test_team = "Gen.G Esports"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_pubg_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 인원 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "person_group",
            "section_heading",
            "nickname_or_id",
            "real_name",
            "position_or_role",
            "join_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 인원 row가 추출되지 않았습니다.")

테스트 팀: Gen.G Esports
페이지 존재 여부: True
에러: None
추출 인원 row 수: 58


,team_title,person_group,section_heading,nickname_or_id,real_name,position_or_role,join_date,leave_date,new_team,nationality,person_url
0,Gen.G Esports,player,Active,BeaN,Oh Won-bin,,2026-01-07 [ 57 ],,,South Korea,https://liquipedia.net/pubg/BeaN
1,Gen.G Esports,player,Active,Salute,Woo Je-hyeon,,2026-01-07 [ 55 ] [ 56 ],,,South Korea,https://liquipedia.net/pubg/Salute
2,Gen.G Esports,player,Active,seoul,Cho Gi-yeol,,2026-01-07 [ 58 ],,,South Korea,https://liquipedia.net/pubg/Seoul
3,Gen.G Esports,player,Active,diyy,Daniel Noh,,2026-01-24 [ 59 ],,,United States,https://liquipedia.net/pubg/Diyy
4,Gen.G Esports,player,Former,WICK2D,Kim Jin-hyung,,2018-08-16 [ 4 ],2018-12-19 [ 9 ],DeToNator.KOREA,"South Korea, DeToNator.KOREA",https://liquipedia.net/pubg/WICK2D
5,Gen.G Esports,player,Former,SimSn,Sim Young-hoon,,2018-08-16 [ 4 ],2018-10-29 [ 7 ],,South Korea,https://liquipedia.net/pubg/SimSn
6,Gen.G Esports,player,Former,EscA,Kim In-jae,Inactive,2018-08-16 [ 4 ],2018-08-20 [ 6 ],OP Gaming Rangers (Substitute),"South Korea, OP Gaming Rangers",https://liquipedia.net/pubg/EscA
7,Gen.G Esports,player,Former,YoonRoot,Yoon Hyun-woo,,2018-08-16 [ 4 ],2018-08-17 [ 5 ],Gen.G Esports (Streamer),"South Korea, Gen.G Esports",https://liquipedia.net/pubg/YoonRoot
8,Gen.G Esports,player,Former,Chelator,Kim Min-ki,,2018-08-16 [ 4 ],2019-12-11 [ 17 ],Gen.G Esports (Coach),"South Korea, Gen.G Esports",https://liquipedia.net/pubg/Chelator
9,Gen.G Esports,player,Former,Esther,Go Jeong-wan,,2018-08-16 [ 4 ],2019-12-11,Gen.G Esports (Content Creator),"South Korea, Gen.G Esports",https://liquipedia.net/pubg/Esther


In [ ]:
def crawl_pubg_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "game_slug": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_pubg_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [ ]:
pubg_teams_checked_df, pubg_teams_final_df, pubg_people_raw_df, pubg_errors_df = crawl_pubg_teams_and_people_v4(
    candidate_df=pubg_dedup_df2,
    output_prefix="pubg_pc_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(pubg_teams_final_df.head(100))
display(pubg_people_raw_df.head(100))

크롤링 대상 후보 수: 259
예상 소요 시간: 133.8 분
예상 소요 시간: 2.23 시간


pubg_pc_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/259 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 74
추출 인원 row 수: 1193
에러 수: 0
저장 완료
/content/pubg_pc_korean_mainteam_refined_v4_teams_checked.csv
/content/pubg_pc_korean_mainteam_refined_v4_teams_final.csv
/content/pubg_pc_korean_mainteam_refined_v4_people_raw.csv
/content/pubg_pc_korean_mainteam_refined_v4_errors.csv


,game_slug,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,pubg,PUBG PC,South Korea,South Korea,https://liquipedia.net/pubg/South_Korea,True,True,[ e ][ h ] South Korea Team Information Locati...,None,main_team_refined
1,pubg,PUBG PC,DeToNator.KOREA,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,True,True,[ e ][ h ] DeToNator.KOREA Team Information Lo...,None,main_team_refined
2,pubg,PUBG PC,Prince (Korea Team),Prince,https://liquipedia.net/pubg/Prince_%28Korea_Te...,True,True,[ e ][ h ] Prince Team Information Location: S...,None,main_team_refined
3,pubg,PUBG PC,Gen.G Esports,Gen.G Esports,https://liquipedia.net/pubg/Gen.G_Esports,True,True,[ e ][ h ] Gen.G Esports Team Information Loca...,None,main_team_refined
4,pubg,PUBG PC,LAVEGA Esports,LAVEGA Esports,https://liquipedia.net/pubg/LAVEGA_Esports,True,True,[ e ][ h ] LAVEGA Esports Team Information Loc...,None,main_team_refined
...,...,...,...,...,...,...,...,...,...,...
69,pubg,PUBG PC,ROG Centurion,ROG Centurion,https://liquipedia.net/pubg/ROG_Centurion,True,True,[ e ][ h ] ASUS ROG Centurion Team Information...,None,main_team_refined
70,pubg,PUBG PC,WeGirls,WeGirls,https://liquipedia.net/pubg/WeGirls,True,True,[ e ][ h ] WeGirls Team Information Location: ...,None,main_team_refined
71,pubg,PUBG PC,Maxtill VIP,Maxtill VIP,https://liquipedia.net/pubg/Maxtill_VIP,True,True,[ e ][ h ] Maxtill VIP Team Information Locati...,None,main_team_refined
72,pubg,PUBG PC,ROG Maximus,ROG Maximus,https://liquipedia.net/pubg/ROG_Maximus,True,True,[ e ][ h ] ASUS ROG Maximus Team Information L...,None,main_team_refined


,game_slug,game_label,team_title,team_url,section_heading,table_index,person_group,nickname_or_id,real_name,position_or_role,join_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,WICK2D,Jin-hyung Kim,IGL,2018-12-24 [ 6 ],2020-03-22 [ 11 ],Team VSG,"South Korea, Team VSG",WICK2D,https://liquipedia.net/pubg/WICK2D,WICK2D Jin-hyung Kim IGL 2018-12-24 [ 6 ] 2020...,DeToNator.KOREA,main_team_refined
1,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,Mickey,Da-hyeon Kim,Co-IGL,2019-12-13 [ 10 ],2020-03-22 [ 11 ],DAMWON Gaming DAMWON Gaming,South Korea,Mick9y,https://liquipedia.net/pubg/Mick9y,Mickey Da-hyeon Kim Co-IGL 2019-12-13 [ 10 ] 2...,DeToNator.KOREA,main_team_refined
2,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,Ohjungje,Jeong-je Oh,Fragger,2019-12-13 [ 10 ],2020-03-22 [ 11 ],,South Korea,Ohjungje,https://liquipedia.net/pubg/Ohjungje,Ohjungje Jeong-je Oh Fragger 2019-12-13 [ 10 ]...,DeToNator.KOREA,main_team_refined
3,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,Joy,Hye-min Park,Fragger,2019-12-13 [ 10 ],2020-03-22 [ 11 ],,South Korea,Joy,https://liquipedia.net/pubg/Joy,Joy Hye-min Park Fragger 2019-12-13 [ 10 ] 202...,DeToNator.KOREA,main_team_refined
4,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,1,player,ISCO,Ho-jin je,Starter,2018-09-06 [ 4 ],2019-12-10 [ 8 ],,South Korea,ISCO,https://liquipedia.net/pubg/ISCO,ISCO Ho-jin je Starter 2018-09-06 [ 4 ] 2019-1...,DeToNator.KOREA,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,pubg,PUBG PC,Gen.G Esports,https://liquipedia.net/pubg/Gen.G_Esports,Former,13,player,Esther,Go Jeong-wan,Content Creator,2019-12-11,2020-07-10 [ 20 ],Gen.G Esports,"South Korea, Gen.G Esports",Esther,https://liquipedia.net/pubg/Esther,Esther Go Jeong-wan Content Creator 2019-12-11...,Gen.G Esports,main_team_refined
96,pubg,PUBG PC,Gen.G Esports,https://liquipedia.net/pubg/Gen.G_Esports,Former,14,player,WatchinU,Bae Seong-hu,Head Coach,2019-12-11 [ 17 ],2022-09-13 [ 35 ],,South Korea,WatchinU,https://liquipedia.net/pubg/WatchinU,WatchinU Bae Seong-hu Head Coach 2019-12-11 [ ...,Gen.G Esports,main_team_refined
97,pubg,PUBG PC,Gen.G Esports,https://liquipedia.net/pubg/Gen.G_Esports,Former,14,player,Chelator,Kim Min-ki,Coach,2019-12-11 [ 17 ],2022-01-02 [ 27 ],,South Korea,Chelator,https://liquipedia.net/pubg/Chelator,Chelator Kim Min-ki Coach 2019-12-11 [ 17 ] 20...,Gen.G Esports,main_team_refined
98,pubg,PUBG PC,Gen.G Esports,https://liquipedia.net/pubg/Gen.G_Esports,Former,15,player,Pio,Cha Seung-hoon,Streamer,2022-01-27 [ 32 ],2023-02-14 [ 37 ],Gen.G Esports,"South Korea, Gen.G Esports",Pio,https://liquipedia.net/pubg/Pio,Pio Cha Seung-hoon Streamer 2022-01-27 [ 32 ] ...,Gen.G Esports,main_team_refined


In [ ]:
def make_unique_people(people_raw_df):
    if len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    if "person_url" not in df.columns:
        df["person_url"] = ""

    if "nickname_or_id" not in df.columns:
        df["nickname_or_id"] = ""

    if "real_name" not in df.columns:
        df["real_name"] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "nickname_or_id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "real_name"].fillna("").astype(str).str.lower().str.strip()
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key", "person_group"]
    ).reset_index(drop=True)

    return unique_df

In [ ]:
pubg_people_unique_df = make_unique_people(pubg_people_raw_df)


print("전체 고유 인원 수:", len(pubg_people_unique_df))

display(pubg_people_unique_df.head(100))

전체 고유 인원 수: 505


,game_slug,game_label,team_title,team_url,section_heading,table_index,person_group,nickname_or_id,real_name,position_or_role,join_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type,person_unique_key
0,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,WICK2D,Jin-hyung Kim,IGL,2018-12-24 [ 6 ],2020-03-22 [ 11 ],Team VSG,"South Korea, Team VSG",WICK2D,https://liquipedia.net/pubg/WICK2D,WICK2D Jin-hyung Kim IGL 2018-12-24 [ 6 ] 2020...,DeToNator.KOREA,main_team_refined,https://liquipedia.net/pubg/WICK2D
1,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,Mickey,Da-hyeon Kim,Co-IGL,2019-12-13 [ 10 ],2020-03-22 [ 11 ],DAMWON Gaming DAMWON Gaming,South Korea,Mick9y,https://liquipedia.net/pubg/Mick9y,Mickey Da-hyeon Kim Co-IGL 2019-12-13 [ 10 ] 2...,DeToNator.KOREA,main_team_refined,https://liquipedia.net/pubg/Mick9y
2,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,Ohjungje,Jeong-je Oh,Fragger,2019-12-13 [ 10 ],2020-03-22 [ 11 ],,South Korea,Ohjungje,https://liquipedia.net/pubg/Ohjungje,Ohjungje Jeong-je Oh Fragger 2019-12-13 [ 10 ]...,DeToNator.KOREA,main_team_refined,https://liquipedia.net/pubg/Ohjungje
3,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,0,player,Joy,Hye-min Park,Fragger,2019-12-13 [ 10 ],2020-03-22 [ 11 ],,South Korea,Joy,https://liquipedia.net/pubg/Joy,Joy Hye-min Park Fragger 2019-12-13 [ 10 ] 202...,DeToNator.KOREA,main_team_refined,https://liquipedia.net/pubg/Joy
4,pubg,PUBG PC,DeToNator.KOREA,https://liquipedia.net/pubg/DeToNator.KOREA,Former,1,player,ISCO,Ho-jin je,Starter,2018-09-06 [ 4 ],2019-12-10 [ 8 ],,South Korea,ISCO,https://liquipedia.net/pubg/ISCO,ISCO Ho-jin je Starter 2018-09-06 [ 4 ] 2019-1...,DeToNator.KOREA,main_team_refined,https://liquipedia.net/pubg/ISCO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,pubg,PUBG PC,LAVEGA Esports,https://liquipedia.net/pubg/LAVEGA_Esports,Former,2,player,NOmer3y,Jin Hyun-min,,2023-02-16 [ 14 ],2023-05-03 [ 15 ],Veronica7,"South Korea, Veronica7",NOmer3y,https://liquipedia.net/pubg/NOmer3y,NOmer3y Jin Hyun-min 2023-02-16 [ 14 ] 2023-05...,LAVEGA Esports,main_team_refined,https://liquipedia.net/pubg/NOmer3y
96,pubg,PUBG PC,LAVEGA Esports,https://liquipedia.net/pubg/LAVEGA_Esports,Former,2,player,Ahcay,Jung Hyeon-wook,,2022-12-31 [ 12 ],2023-02-16 [ 14 ],PENTAGRAM,"South Korea, PENTAGRAM",Ahcay,https://liquipedia.net/pubg/Ahcay,Ahcay Jung Hyeon-wook 2022-12-31 [ 12 ] 2023-0...,LAVEGA Esports,main_team_refined,https://liquipedia.net/pubg/Ahcay
97,pubg,PUBG PC,LAVEGA Esports,https://liquipedia.net/pubg/LAVEGA_Esports,Former,2,player,Hwan2da,Jang Hwan,,2022-12-31 [ 12 ],2023-02-16 [ 14 ],,South Korea,Hwan2da,https://liquipedia.net/pubg/Hwan2da,Hwan2da Jang Hwan 2022-12-31 [ 12 ] 2023-02-16...,LAVEGA Esports,main_team_refined,https://liquipedia.net/pubg/Hwan2da
98,pubg,PUBG PC,LAVEGA Esports,https://liquipedia.net/pubg/LAVEGA_Esports,Active,3,player,GIO,,Head Coach,2022-12-31 [ 12 ],,,South Korea,index.php?title=GIO&action=edit&redlink=1,https://liquipedia.net/pubg/index.php?title=GI...,GIO Head Coach 2022-12-31 [ 12 ],LAVEGA Esports,main_team_refined,https://liquipedia.net/pubg/index.php?title=GI...


In [ ]:
pubg_dedup_df2.to_csv(
    "/content/pubg_pc_korean_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

pubg_teams_final_df.to_csv(
    "/content/pubg_pc_korean_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

pubg_people_raw_df.to_csv(
    "/content/pubg_pc_korean_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

pubg_people_unique_df.to_csv(
    "/content/pubg_pc_korean_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

pubg_errors_df.to_csv(
    "/content/pubg_pc_korean_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/pubg_pc_korean_teams_final_v4.csv")
print("/content/pubg_pc_korean_people_raw_v4.csv")
print("/content/pubg_pc_korean_people_unique_v4.csv")
print("/content/pubg_pc_korean_extract_errors_v4.csv")

저장 완료
/content/pubg_pc_korean_teams_final_v4.csv
/content/pubg_pc_korean_people_raw_v4.csv
/content/pubg_pc_korean_people_unique_v4.csv
/content/pubg_pc_korean_extract_errors_v4.csv


#발로란트 시작

In [ ]:
GAME_SLUG = "valorant"
GAME_LABEL = "VALORANT"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

In [ ]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    VALORANT Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [ ]:
VALORANT_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korea team",
    "Korean team",
    "Korea esports",
    "VCT Korea",
    "VCT Pacific Korea",
    "VALORANT Korea",
    "VALORANT Challengers Korea",
    "Challengers Korea",
    "Game Changers Korea",

    # 대표 한국 팀/조직/관련 팀명
    "DRX",
    "DRX Academy",
    "DRX Changers",
    "T1",
    "Gen.G Esports",
    "Gen.G Global Academy",
    "Nongshim RedForce",
    "Dplus KIA",
    "Dplus",
    "DWG KIA",
    "FearX",
    "BNK FearX",
    "On Sla2ers",
    "ONSIDE GAMING",
    "SLT",
    "Maru Gaming",
    "World Game Star",
    "WGS",
    "CNJ esports",
    "SuperFect",
    "F4Q",
    "Vision Strikers",
    "Rio Company",
    "Team MUYAHO",
    "Gwangju Shadow",
    "Kwangdong Freecs",
    "Afreeca Freecs",
]

In [ ]:
search_rows = []

for keyword in VALORANT_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

valorant_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(valorant_search_df))
display(valorant_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: korea / offset=650
검색: korea / offset=700
검색: korea / offset=750
검색: korea / offset=800
검색: korea / offset=850
검색: korea / offset=900
검색: korea / offset=950
max_total 도달: 1000
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: South Korea / offset=350
검색: South Korea / offset=400
검색: South Korea / offset=450
검색: South Korea / offset=500
검색: South Korea / offset=550
검색: South Korea / offset=600
검색: South Korea / offset=650
검색: South Korea / offset=700
검색: South Korea / offset=750
검색: South Korea / offset=800
검색: South Korea / offset=850
검색: S

,game_slug,game_label,title,snippet,pageid,url,search_keyword
0,valorant,VALORANT,Jett,Jett General Information Real Name: Sunwoo Han...,514,https://liquipedia.net/valorant/Jett,korea
1,valorant,VALORANT,A-Tier Tournaments,Bren Esports Paper Rex VCT 2021: Korea Stage 3...,1161,https://liquipedia.net/valorant/A-Tier_Tournam...,korea
2,valorant,VALORANT,Dplus,Dplus Team Information Location: South Korea R...,16539,https://liquipedia.net/valorant/Dplus,korea
3,valorant,VALORANT,Gen.G Esports,Gen.G Esports Team Information Location: South...,2771,https://liquipedia.net/valorant/Gen.G_Esports,korea
4,valorant,VALORANT,DRX,DRX Team Information Location: South Korea Reg...,29769,https://liquipedia.net/valorant/DRX,korea
...,...,...,...,...,...,...,...
95,valorant,VALORANT,Persia,2021: Korea Stage 2 Challengers with TUBEPLE G...,21234,https://liquipedia.net/valorant/Persia,korea
96,valorant,VALORANT,Quantum Strikers,Quantum Strikers Team Information Location: So...,5532,https://liquipedia.net/valorant/Quantum_Strikers,korea
97,valorant,VALORANT,IAM,Dec 14 - 17:00 KST Predator League Korea 2026-...,58435,https://liquipedia.net/valorant/IAM,korea
98,valorant,VALORANT,VCT/2024/Game Changers/Korea/Stage 2/Open Qual...,Changers Korea Stage 2 - Open Qualifier League...,57005,https://liquipedia.net/valorant/VCT/2024/Game_...,korea


In [ ]:
def build_valorant_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "academy",
        "changers",
        "challengers",
        "prospects",

        # 한국 발로란트 팀명성 키워드
        "drx",
        "t1",
        "gen.g",
        "geng",
        "global academy",
        "nongshim",
        "redforce",
        "dplus",
        "kia",
        "dwg",
        "fearx",
        "bnk",
        "on sla2ers",
        "onside",
        "slt",
        "maru",
        "wgs",
        "world game star",
        "cnj",
        "superfect",
        "f4q",
        "vision strikers",
        "rio company",
        "muyaho",
        "gwangju",
        "shadow",
        "kwangdong",
        "freecs",
        "afreeca",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "split",
        "playoffs",
        "cup",
        "championship",
        "tournament",
        "qualifier",
        "showmatch",
        "match history",
        "results",
        "statistics",
        "roster",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "patch",
        "standings",
        "group stage",
        "regular season",
        "participants",
        "matches",
        "rankings",
        "awards",
        "finals",
        "league",
        "stage",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        if not any(term in text for term in korea_terms):
            return False

        if not any(keyword in text for keyword in team_like_keywords):
            return False

        return True

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("VALORANT 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [ ]:
valorant_team_candidate_loose_df = build_valorant_team_candidate_loose_df(valorant_search_df)

display(valorant_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

VALORANT 완화 후보 수: 189
예상 parse 소요 시간: 97.7 분
예상 parse 소요 시간: 1.63 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea R...,korea,https://liquipedia.net/valorant/Dplus
1,Gen.G Esports,Gen.G Esports Team Information Location: South...,korea,https://liquipedia.net/valorant/Gen.G_Esports
2,DRX,DRX Team Information Location: South Korea Reg...,korea,https://liquipedia.net/valorant/DRX
3,T1,T1 Team Information Location: South Korea Unit...,korea,https://liquipedia.net/valorant/T1
4,Nongshim RedForce,Nongshim RedForce Team Information Location: S...,korea,https://liquipedia.net/valorant/Nongshim_RedForce
...,...,...,...,...
184,NSG x Renegades Invitational,Xp3 3 Mexico 1 / 80 (1%) Dcop 3 Mongolia 1 / 8...,korea,https://liquipedia.net/valorant/NSG_x_Renegade...
185,Valorant Kevin Scrim,Information Series: Kevin Scrim Organizer: Afr...,South Korea,https://liquipedia.net/valorant/Valorant_Kevin...
186,Top Esports,Sylvan (2024-01-02). &quot;Lft / FA&quot;. Xiv...,Korean,https://liquipedia.net/valorant/Top_Esports
187,Global Esports,Chiu Director 2023-??-?? UnknowN Minseong Kim ...,Korean,https://liquipedia.net/valorant/Global_Esports


In [ ]:
# =========================================
# VALORANT 완화 후보에서 팀 후보만 재정제
# source: valorant_team_candidate_loose_df
# output: valorant_dedup_df2
# 페이지 불러오기 X, API 호출 X
# =========================================

source_df = valorant_team_candidate_loose_df.copy()

valorant_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "academy",
    "changers",
    "challengers",
    "prospects",

    "drx",
    "t1",
    "gen.g",
    "geng",
    "global academy",
    "nongshim",
    "redforce",
    "dplus",
    "kia",
    "dwg",
    "fearx",
    "bnk",
    "on sla2ers",
    "onside",
    "slt",
    "maru",
    "wgs",
    "world game star",
    "cnj",
    "superfect",
    "f4q",
    "vision strikers",
    "rio company",
    "muyaho",
    "gwangju",
    "shadow",
    "kwangdong",
    "freecs",
    "afreeca",
]

valorant_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "split",
    "playoffs",
    "cup",
    "championship",
    "tournament",
    "qualifier",
    "showmatch",
    "match history",
    "results",
    "statistics",
    "roster",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "patch",
    "standings",
    "group stage",
    "regular season",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
    "league",
    "stage",
]

valorant_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

valorant_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_valorant_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in valorant_bad_page_markers):
        return False

    for pattern in valorant_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in valorant_korea_terms):
        return False

    # Team Information이 있으면 강한 팀 후보
    if "team information" in text:
        return True

    # Team Information이 없는 경우에는 팀명성 키워드가 있어야 통과
    if any(keyword in text for keyword in valorant_team_name_keywords):
        return True

    return False


valorant_dedup_df2 = source_df[
    source_df.apply(is_refined_valorant_team_candidate, axis=1)
].copy().reset_index(drop=True)

valorant_dedup_df2["title_clean"] = (
    valorant_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

valorant_dedup_df2 = valorant_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("VALORANT 완화 후보 수:", len(valorant_team_candidate_loose_df))
print("재정제 후 valorant_dedup_df2 후보 수:", len(valorant_dedup_df2))
print("예상 parse 소요 시간:", round(len(valorant_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(valorant_dedup_df2) * 31 / 3600, 2), "시간")

display(valorant_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

VALORANT 완화 후보 수: 189
재정제 후 valorant_dedup_df2 후보 수: 189
예상 parse 소요 시간: 97.7 분
예상 parse 소요 시간: 1.63 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea R...,korea,https://liquipedia.net/valorant/Dplus
1,Gen.G Esports,Gen.G Esports Team Information Location: South...,korea,https://liquipedia.net/valorant/Gen.G_Esports
2,DRX,DRX Team Information Location: South Korea Reg...,korea,https://liquipedia.net/valorant/DRX
3,T1,T1 Team Information Location: South Korea Unit...,korea,https://liquipedia.net/valorant/T1
4,Nongshim RedForce,Nongshim RedForce Team Information Location: S...,korea,https://liquipedia.net/valorant/Nongshim_RedForce
...,...,...,...,...
184,NSG x Renegades Invitational,Xp3 3 Mexico 1 / 80 (1%) Dcop 3 Mongolia 1 / 8...,korea,https://liquipedia.net/valorant/NSG_x_Renegade...
185,Valorant Kevin Scrim,Information Series: Kevin Scrim Organizer: Afr...,South Korea,https://liquipedia.net/valorant/Valorant_Kevin...
186,Top Esports,Sylvan (2024-01-02). &quot;Lft / FA&quot;. Xiv...,Korean,https://liquipedia.net/valorant/Top_Esports
187,Global Esports,Chiu Director 2023-??-?? UnknowN Minseong Kim ...,Korean,https://liquipedia.net/valorant/Global_Esports


In [ ]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [ ]:
def extract_valorant_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    hard_exclude_keywords = [
        "Results",
        "Data",
        "Gallery",
        "Documentaries",
        "References",
        "Achievements",
        "Awards",
        "Statistics",
        "Upcoming Matches",
        "Upcoming Tournaments",
    ]

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "valorant", "liquipedia",
    ]

    def get_section_heading(el):
        prev = el.find_previous(["h2", "h3", "h4"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_previous_subheading(el):
        prev = el.find_previous(["h3", "h4", "h5"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_country_from_element(el):
        countries = []
        for img in el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if value and value not in countries:
                    if not value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                        countries.append(value)
        return ", ".join(countries)

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h in ["inactive date"]:
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["replacing"]:
            return "replacing"
        if h in ["tournament(s)", "tournaments"]:
            return "tournaments"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"

        # Organization 표는 ID / Name / 역할 / Join Date 구조인데 역할 헤더가 빈칸일 수 있음
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/File:", "/Category:", "/Special:", "/Help:", "/Template:",
            "/Portal:", "/Liquipedia:", "/User:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|vct|masters|champions)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def classify_table(heading, subheading, table_text):
        combined = f"{heading} {subheading} {table_text}".lower()

        if any(k.lower() in combined for k in hard_exclude_keywords):
            if "player roster" not in combined and "organization" not in combined and "stand-in" not in combined:
                return None, None

        is_org = "organization" in combined or "former organization" in combined
        is_player = (
            "player roster" in combined
            or "former players" in combined
            or "inactive players" in combined
            or "temporary stand-ins" in combined
            or "stand-in" in combined
        )

        if is_org:
            if "former" in combined:
                return "organization", "Former"
            return "organization", "Active"

        if is_player:
            if "inactive" in combined:
                return "player_roster", "Inactive"
            if "former" in combined:
                return "player_roster", "Former"
            if "stand-in" in combined or "temporary stand-ins" in combined:
                return "player_roster", "Stand-in"
            return "player_roster", "Active"

        return None, None

    def default_position(table_type, status, row_data):
        # Organization이면 표의 역할 컬럼을 position으로 사용
        if table_type == "organization":
            pos = row_data.get("position", "")
            if pos:
                return pos
            if status == "Former":
                return "Former Staff"
            return "Staff"

        # Player Roster면 역할이 없으므로 상태 기반으로 position 채움
        if table_type == "player_roster":
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            if status == "Stand-in":
                return "Stand-in"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        heading = get_section_heading(table)
        subheading = get_previous_subheading(table)
        table_text = clean_text(table.get_text(" ", strip=True))

        table_type, status = classify_table(heading, subheading, table_text)

        if table_type is None:
            continue

        trs = table.find_all("tr")
        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
                or "tournament(s)" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        # Organization 표에서 빈 역할 헤더 보정
        if table_type == "organization":
            if "position" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])
            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))
            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_country_from_element(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": heading,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "replacing": row_data.get("replacing", ""),
                "tournaments": row_data.get("tournaments", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
        ]
    ).reset_index(drop=True)

    return df

In [ ]:
test_team = "DRX"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_valorant_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: DRX
페이지 존재 여부: True
에러: None
추출 row 수: 3


,team_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,DRX,player_roster,Stand-in,Rb,Goo Sang-min,Stand-in,,,,,"South Korea, DRX Prospects, VALORANT Champions...",https://liquipedia.net/valorant/Flashback
1,DRX,player_roster,Stand-in,Athan,,Stand-in,,,,,"South Korea, DRX Prospects, VALORANT Champions...",https://liquipedia.net/valorant/Athan
2,DRX,player_roster,Stand-in,HYUNMIN,Song Hyun-min,Stand-in,,,,,"South Korea, DRX Prospects",https://liquipedia.net/valorant/ApeX_(Korean_p...


#테스트 보고 발로란트 수정

In [ ]:
def extract_valorant_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "valorant", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Data",
        "Gallery",
        "Documentaries",
        "References",
        "Achievements",
        "Awards",
        "Statistics",
        "Upcoming Matches",
        "Upcoming Tournaments",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        """
        가장 가까운 h2를 기준으로 큰 섹션 판단.
        예: Player Roster / Organization
        """
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        """
        가장 가까운 h3/h4를 기준으로 상태 판단.
        예: Active / Inactive / Former / Stand-in
        """
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                # 팀 로고/대회 로고처럼 보이는 것 제외용 최소 필터
                if value.lower() in ["valorant", "vct", "edit"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h == "replacing":
            return "replacing"
        if h in ["tournament(s)", "tournaments"]:
            return "tournaments"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|vct|masters|champions)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)
        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low:
            table_type = "player_roster"

            if "inactive" in sub_low:
                status = "Inactive"
            elif "former" in sub_low:
                status = "Former"
            elif "stand" in sub_low:
                status = "Stand-in"
            else:
                status = "Active"

            return table_type, status, main_section

        if "organization" in main_low:
            table_type = "organization"

            if "former" in sub_low:
                status = "Former"
            else:
                status = "Active"

            return table_type, status, main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        if table_type == "organization":
            pos = row_data.get("position", "")
            if pos:
                return pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            if status == "Stand-in":
                return "Stand-in"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
                or "tournament(s)" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        # Organization 표: ID / Name / [역할] / Join Date 구조 보정
        if table_type == "organization":
            if "position" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            # 연도 구분행, Show All 같은 단일 셀 행 제외
            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                # Stand-in 표는 ID/Name이 두 세트라 뒤에서 덮일 수 있음
                # 그래서 이미 있는 id/name은 덮어쓰지 않음
                if status == "Stand-in" and col in ["id", "name"] and col in row_data:
                    continue

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                # Stand-in도 첫 번째 링크가 stand-in 인물이라 첫 링크 사용
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            # 팀 자기 자신 링크 제거
            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "replacing": row_data.get("replacing", ""),
                "tournaments": row_data.get("tournaments", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [ ]:
test_team = "DRX"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_valorant_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: DRX
페이지 존재 여부: True
에러: None
추출 row 수: 49


,team_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,DRX,player_roster,Active,MaKo,Kim Myeong-kwan,Player,2022-01-07 [ 8 ],,,,South Korea,https://liquipedia.net/valorant/MaKo
1,DRX,player_roster,Active,free1ng,No Ha-jun,Player,2024-10-11 [ 30 ],,,,South Korea,https://liquipedia.net/valorant/Freeing
2,DRX,player_roster,Active,HYUNMIN,Song Hyun-min,Player,2024-10-11 [ 30 ],,,,South Korea,https://liquipedia.net/valorant/HYUNMIN
3,DRX,player_roster,Active,BeYN,Kang Ha-bin,Player,2025-01-05 [ 38 ],,,,South Korea,https://liquipedia.net/valorant/BeYN
4,DRX,player_roster,Active,Hermes,Ahn Byeong-wook,Player,2026-01-08 [ 43 ],,,,South Korea,https://liquipedia.net/valorant/Hermes
5,DRX,player_roster,Active,Yong,Kim Ho-yong,Player,2026-03-17 [ 45 ],,,,South Korea,https://liquipedia.net/valorant/Yong
6,DRX,player_roster,Inactive,Flicker,Yoon Tae-hee,Inactive Player,2025-07-07 [ 41 ],2026-01-20 [ 44 ],,,South Korea,https://liquipedia.net/valorant/Flicker
7,DRX,player_roster,Former,Lakia,Kim Jong-min,Former Player,2022-01-10 [ 9 ],,2022-01-23 [ 10 ],IGZIST,South Korea,https://liquipedia.net/valorant/Lakia
8,DRX,player_roster,Former,Lakia,Kim Jong-min,Former Player,2022-01-07 [ 8 ],,2022-01-10 [ 9 ],DRX (Inactive),South Korea,https://liquipedia.net/valorant/Lakia
9,DRX,player_roster,Former,Rb,Goo Sang-min,Former Player,2022-01-07 [ 8 ],,2023-12-12 [ 19 ],DRX (Inactive),South Korea,https://liquipedia.net/valorant/Rb


In [ ]:
# =========================================
# VALORANT 팀 검증 + 인원 추출 함수 v4
# 먼저 이 셀을 실행해야 함
# =========================================

def crawl_valorant_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "game_slug": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_valorant_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [ ]:
valorant_teams_checked_df, valorant_teams_final_df, valorant_people_raw_df, valorant_errors_df = crawl_valorant_teams_and_people_v4(
    candidate_df=valorant_dedup_df2,
    output_prefix="valorant_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(valorant_teams_final_df.head(100))
display(valorant_people_raw_df.head(100))

크롤링 대상 후보 수: 189
예상 소요 시간: 97.7 분
예상 소요 시간: 1.63 시간


valorant_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/189 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 45
추출 인원 row 수: 1107
에러 수: 0
저장 완료
/content/valorant_korean_mainteam_refined_v4_teams_checked.csv
/content/valorant_korean_mainteam_refined_v4_teams_final.csv
/content/valorant_korean_mainteam_refined_v4_people_raw.csv
/content/valorant_korean_mainteam_refined_v4_errors.csv


,game_slug,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,valorant,VALORANT,Dplus,Dplus,https://liquipedia.net/valorant/Dplus,True,True,[ e ][ h ] Dplus Team Information Location: So...,None,main_team_refined
1,valorant,VALORANT,Gen.G Esports,Gen.G Esports,https://liquipedia.net/valorant/Gen.G_Esports,True,True,[ e ][ h ] Gen.G Esports Team Information Loca...,None,main_team_refined
2,valorant,VALORANT,DRX,DRX,https://liquipedia.net/valorant/DRX,True,True,[ e ][ h ] DRX Team Information Location: Sout...,None,main_team_refined
3,valorant,VALORANT,T1,T1,https://liquipedia.net/valorant/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
4,valorant,VALORANT,Nongshim RedForce,Nongshim RedForce,https://liquipedia.net/valorant/Nongshim_RedForce,True,True,[ e ][ h ] Nongshim RedForce Team Information ...,None,main_team_refined
5,valorant,VALORANT,SLT,SLT,https://liquipedia.net/valorant/SLT,True,True,[ e ][ h ] SLT Team Information Location: Sout...,None,main_team_refined
6,valorant,VALORANT,DRX Academy,DRX Academy,https://liquipedia.net/valorant/DRX_Academy,True,True,[ e ][ h ] DRX Academy Team Information Locati...,None,main_team_refined
7,valorant,VALORANT,T1 Academy,T1 Academy,https://liquipedia.net/valorant/T1_Academy,True,True,[ e ][ h ] T1 Esports Academy Team Information...,None,main_team_refined
8,valorant,VALORANT,FEARX,FEARX,https://liquipedia.net/valorant/FEARX,True,True,[ e ][ h ] FearX Team Information Location: So...,None,main_team_refined
9,valorant,VALORANT,Gen.G Global Academy,Gen.G Global Academy,https://liquipedia.net/valorant/Gen.G_Global_A...,True,True,[ e ][ h ] Gen.G Global Academy Team Informati...,None,main_team_refined


,game_slug,game_label,team_title,team_url,section_heading,subheading,table_type,status,table_index,id,...,leave_date,new_team,replacing,tournaments,nationality,person_page,person_url,row_text,team_display_title,team_type
0,valorant,VALORANT,Dplus,https://liquipedia.net/valorant/Dplus,Player Roster,Active,player_roster,Active,0,CabezA,...,,,,,South Korea,CabezA,https://liquipedia.net/valorant/CabezA,CabezA Roh Geon-hyeong 2025-11-30 [ 44 ],Dplus,main_team_refined
1,valorant,VALORANT,Dplus,https://liquipedia.net/valorant/Dplus,Player Roster,Active,player_roster,Active,0,JaXe,...,,,,,South Korea,JaXe,https://liquipedia.net/valorant/JaXe,JaXe Shin Jae-young 2025-11-30 [ 45 ],Dplus,main_team_refined
2,valorant,VALORANT,Dplus,https://liquipedia.net/valorant/Dplus,Player Roster,Active,player_roster,Active,0,Kally,...,,,,,South Korea,Kally,https://liquipedia.net/valorant/Kally,Kally Kim Dong-uk 2025-11-30 [ 45 ],Dplus,main_team_refined
3,valorant,VALORANT,Dplus,https://liquipedia.net/valorant/Dplus,Player Roster,Active,player_roster,Active,0,Ramgi,...,,,,,South Korea,Ramgi,https://liquipedia.net/valorant/Ramgi,Ramgi Woo Jun-hyuk 2025-11-30 [ 45 ],Dplus,main_team_refined
4,valorant,VALORANT,Dplus,https://liquipedia.net/valorant/Dplus,Player Roster,Active,player_roster,Active,0,shu,...,,,,,South Korea,Shu (Korean player),https://liquipedia.net/valorant/Shu_(Korean_pl...,shu Yun Si-hu 2025-11-30 [ 45 ],Dplus,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,valorant,VALORANT,Gen.G Esports,https://liquipedia.net/valorant/Gen.G_Esports,Player Roster,Stand-in,player_roster,Stand-in,9,miLLe,...,,,,Immortals First Light (3rd Place Match vs. Imm...,Canada,MiLLe,https://liquipedia.net/valorant/MiLLe,miLLe Michael St-Pierre MkaeL Michael De Luca ...,Gen.G Esports,main_team_refined
96,valorant,VALORANT,Gen.G Esports,https://liquipedia.net/valorant/Gen.G_Esports,Player Roster,Stand-in,player_roster,Stand-in,9,ZynX,...,,,None,,South Korea,ZynX,https://liquipedia.net/valorant/ZynX,ZynX Kim Dong-ha None SOOP VALORANT LEAGUE 202...,Gen.G Esports,main_team_refined
97,valorant,VALORANT,Gen.G Esports,https://liquipedia.net/valorant/Gen.G_Esports,Organization,Active,organization,Active,10,solo,...,,,,,South Korea,Solo,https://liquipedia.net/valorant/Solo,solo Kang Keun-chul Head Coach 2025-10-23 [ 75 ],Gen.G Esports,main_team_refined
98,valorant,VALORANT,Gen.G Esports,https://liquipedia.net/valorant/Gen.G_Esports,Organization,Active,organization,Active,10,HSK,...,,,,,South Korea,HSK,https://liquipedia.net/valorant/HSK,HSK Kim Hae-seong Coach 2025-10-23 [ 76 ],Gen.G Esports,main_team_refined


In [ ]:
# =========================================
# 고유 인원 목록 만들기
# =========================================

def make_unique_people(people_raw_df):
    if len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    if "person_url" not in df.columns:
        df["person_url"] = ""

    if "id" not in df.columns:
        df["id"] = ""

    if "name" not in df.columns:
        df["name"] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "position"].fillna("").astype(str).str.lower().str.strip()
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key", "position"]
    ).reset_index(drop=True)

    return unique_df


valorant_people_unique_df = make_unique_people(valorant_people_raw_df)

print("전체 row 수:", len(valorant_people_raw_df))
print("고유 인원 수:", len(valorant_people_unique_df))

display(valorant_people_unique_df[[
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

전체 row 수: 1107
고유 인원 수: 701


,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Dplus,Active,CabezA,Roh Geon-hyeong,Player,2025-11-30 [ 44 ],,,,South Korea,https://liquipedia.net/valorant/CabezA
1,Dplus,Active,JaXe,Shin Jae-young,Player,2025-11-30 [ 45 ],,,,South Korea,https://liquipedia.net/valorant/JaXe
2,Dplus,Active,Kally,Kim Dong-uk,Player,2025-11-30 [ 45 ],,,,South Korea,https://liquipedia.net/valorant/Kally
3,Dplus,Active,Ramgi,Woo Jun-hyuk,Player,2025-11-30 [ 45 ],,,,South Korea,https://liquipedia.net/valorant/Ramgi
4,Dplus,Active,shu,Yun Si-hu,Player,2025-11-30 [ 45 ],,,,South Korea,https://liquipedia.net/valorant/Shu_(Korean_pl...
...,...,...,...,...,...,...,...,...,...,...,...
95,Gen.G Esports,Former,ChunChun,,Streamer,2023-02-10 [ 47 ],,2023-07-19 [ 51 ],,South Korea,https://liquipedia.net/valorant/index.php?titl...
96,Gen.G Esports,Former,HSK,Kim Hae-seong,Head Coach,2023-10-10 [ 54 ],,2024-01-30 [ 89 ],Gen.G Esports (Head of Strategy),South Korea,https://liquipedia.net/valorant/HSK
97,Gen.G Esports,Former,HSK,Kim Hae-seong,Head of Strategy,2024-01-30 [ 89 ],,2025-10-21 [ 74 ],Gen.G Esports (Coach),South Korea,https://liquipedia.net/valorant/HSK
98,Gen.G Esports,Former,peri,Jung Beom-gi,Assistant Coach,2024-12-18 [ 67 ],,2025-10-21 [ 74 ],Gen.G Esports (Coach),South Korea,https://liquipedia.net/valorant/Peri


In [ ]:
valorant_dedup_df2.to_csv(
    "/content/valorant_korean_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

valorant_teams_checked_df.to_csv(
    "/content/valorant_korean_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

valorant_teams_final_df.to_csv(
    "/content/valorant_korean_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

valorant_people_raw_df.to_csv(
    "/content/valorant_korean_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

valorant_people_unique_df.to_csv(
    "/content/valorant_korean_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

valorant_errors_df.to_csv(
    "/content/valorant_korean_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/valorant_korean_teams_final_v4.csv")
print("/content/valorant_korean_people_unique_v4.csv")

저장 완료
/content/valorant_korean_teams_final_v4.csv
/content/valorant_korean_people_unique_v4.csv


In [ ]:
GAME_SLUG = "tft"
GAME_LABEL = "Teamfight Tactics"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"


In [ ]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    TFT Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [ ]:
TFT_SEARCH_KEYWORDS = [
    # 국가/지역 기반
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korea team",
    "Korean team",
    "Korea esports",
    "TFT Korea",
    "Teamfight Tactics Korea",
    "Korea Teamfight Tactics",
    "APAC Korea",
    "Asia-Pacific Korea",

    # 대표 한국 TFT 팀/조직/관련 팀명
    "FN Esports",
    "MIRAEN SEJONG",
    "Miraen Sejong",
    "Future N",
    "T1",
    "Gen.G",
    "DRX",
    "Kwangdong Freecs",
    "Afreeca Freecs",
    "Nongshim RedForce",
    "Dplus KIA",
    "Dplus",
    "BNK FearX",
    "FearX",
    "Team GP",
    "Sengoku Gaming Korea",
    "OP.GG",
    "Shadow Corporation",
    "Teamfight Tactics Korean team",
]

In [ ]:
search_rows = []

for keyword in TFT_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

tft_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(tft_search_df))
display(tft_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: Korean / offset=150
검색: Korean / offset=200
검색: Korean / offset=250
검색: Korean / offset=300
검색: Korean / offset=350
검색: Korean / offset=400
검색: Korean / offset=450
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: South Korean / offset=100
검색: South Korean / offset=150
검색: South Korean / offset=200
검색: South Korean / offset=250
검색: South Korean / offset=300
검색: Korea team / offset=0
검색: Korea team / offset=50
검색: Korea team / offset=100
검색: Korea team / offset=150
검색: Korean team / of

,game_slug,game_label,title,snippet,pageid,url,search_keyword
0,tft,Teamfight Tactics,Qualifiers Tournaments,"Qualifier Jan 24–25, 2026 South Korea 32 TheDe...",81,https://liquipedia.net/tft/Qualifiers_Tournaments,korea
1,tft,Teamfight Tactics,B-Tier Tournaments,World South Korea 12 DongGeul Tept CN TFT Set ...,74,https://liquipedia.net/tft/B-Tier_Tournaments,korea
2,tft,Teamfight Tactics,TFT APAC Esports,"#2 May 24–25, 2025 South Korea 112 엉망진창 메시 화내면...",5782,https://liquipedia.net/tft/TFT_APAC_Esports,korea
3,tft,Teamfight Tactics,Cyber City,"#2 May 24–25, 2025 South Korea 112 엉망진창 메시 화내면...",9484,https://liquipedia.net/tft/Cyber_City,korea
4,tft,Teamfight Tactics,C-Tier Tournaments,1 SOOP TFT Series: Lore &amp; Legends - Finals...,2983,https://liquipedia.net/tft/C-Tier_Tournaments,korea
...,...,...,...,...,...,...,...
95,tft,Teamfight Tactics,Fates/Korea/ATS/Finals/2,Teamfight Tactics Patch: 11.4 Game Mode: Solos...,5490,https://liquipedia.net/tft/Fates/Korea/ATS/Fin...,korea
96,tft,Teamfight Tactics,Reckoning/Korea/ATS/Finals/2,Teamfight Tactics Patch: 11.12 Game Mode: Solo...,5486,https://liquipedia.net/tft/Reckoning/Korea/ATS...,korea
97,tft,Teamfight Tactics,Geunman,South Korea Region: Korea Status: Active Alter...,7485,https://liquipedia.net/tft/Geunman,korea
98,tft,Teamfight Tactics,Remix Rumble/Korea/AF Open Invitational/Season 2,Teamfight Tactics Patch: 13.24b Game Mode: Sol...,5178,https://liquipedia.net/tft/Remix_Rumble/Korea/...,korea


In [ ]:
def build_tft_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "roster",

        # 한국 TFT 팀명성 키워드
        "fn esports",
        "miraen",
        "sejong",
        "future n",
        "t1",
        "gen.g",
        "geng",
        "drx",
        "kwangdong",
        "freecs",
        "afreeca",
        "nongshim",
        "redforce",
        "dplus",
        "kia",
        "fearx",
        "bnk",
        "op.gg",
        "shadow",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "split",
        "playoffs",
        "cup",
        "championship",
        "tournament",
        "qualifier",
        "showmatch",
        "match history",
        "results",
        "statistics",
        "roster",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "patch",
        "standings",
        "group stage",
        "regular season",
        "participants",
        "matches",
        "rankings",
        "awards",
        "finals",
        "league",
        "stage",
        "tactician",
        "golden spatula",
        "regional finals",
        "pro circuit",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
        "cosmetic item",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "south korea",
            "korea",
            "korean",
        ]

        if not any(term in text for term in korea_terms):
            return False

        if not any(keyword in text for keyword in team_like_keywords):
            return False

        return True

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("TFT 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * PARSE_SLEEP / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * PARSE_SLEEP / 3600, 2), "시간")

    return df

In [ ]:
tft_team_candidate_loose_df = build_tft_team_candidate_loose_df(tft_search_df)

display(tft_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

TFT 완화 후보 수: 94
예상 parse 소요 시간: 48.6 분
예상 parse 소요 시간: 0.81 시간


,title,snippet,search_keyword,url
0,TFT APAC Esports,"#2 May 24–25, 2025 South Korea 112 엉망진창 메시 화내면...",korea,https://liquipedia.net/tft/TFT_APAC_Esports
1,Binteum,Romanized Name: Kang Seong-jun Nationality: So...,korea,https://liquipedia.net/tft/Binteum
2,Ssang Yeop,Romanized Name: Seo Sung-won Nationality: Sout...,korea,https://liquipedia.net/tft/Ssang_Yeop
3,SCsC,Romanized Name: Kim Seung-chul Nationality: So...,korea,https://liquipedia.net/tft/SCsC
4,Dunizuni,조준희 Romanized Name: Cho Jun-hee Nationality: S...,korea,https://liquipedia.net/tft/Dunizuni
...,...,...,...,...
89,Ikura,Gwang-won Nationality: South Korea Region: Kor...,korea,https://liquipedia.net/tft/Ikura
90,LeVition,Information Nationality: South Korea Region: K...,korea,https://liquipedia.net/tft/LeVition
91,Chabo Kim,Nationality: South Korea Region: Korea Status:...,korea,https://liquipedia.net/tft/Chabo_Kim
92,Yummy chidon,Information Nationality: South Korea Region: K...,korea,https://liquipedia.net/tft/Yummy_chidon


In [ ]:
# =========================================
# TFT 완화 후보에서 팀 후보만 재정제
# source: tft_team_candidate_loose_df
# output: tft_dedup_df2
# 페이지 불러오기 X, API 호출 X
# =========================================

source_df = tft_team_candidate_loose_df.copy()

tft_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "roster",

    "fn esports",
    "miraen",
    "sejong",
    "future n",
    "t1",
    "gen.g",
    "geng",
    "drx",
    "kwangdong",
    "freecs",
    "afreeca",
    "nongshim",
    "redforce",
    "dplus",
    "kia",
    "fearx",
    "bnk",
    "op.gg",
    "shadow",
]

tft_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "split",
    "playoffs",
    "cup",
    "championship",
    "tournament",
    "qualifier",
    "showmatch",
    "match history",
    "results",
    "statistics",
    "roster",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "patch",
    "standings",
    "group stage",
    "regular season",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
    "league",
    "stage",
    "tactician",
    "golden spatula",
    "regional finals",
    "pro circuit",
]

tft_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
    "cosmetic item",
]

tft_korea_terms = [
    "location: south korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_tft_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in tft_bad_page_markers):
        return False

    for pattern in tft_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in tft_korea_terms):
        return False

    # Team Information이 있으면 강한 팀 후보
    if "team information" in text:
        return True

    # Organization Overview 같은 팀 연결 페이지도 일부 통과
    if "organization overview" in text:
        return True

    # Team Information이 없는 경우에는 팀명성 키워드가 있어야 통과
    if any(keyword in text for keyword in tft_team_name_keywords):
        return True

    return False


tft_dedup_df2 = source_df[
    source_df.apply(is_refined_tft_team_candidate, axis=1)
].copy().reset_index(drop=True)

tft_dedup_df2["title_clean"] = (
    tft_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

tft_dedup_df2 = tft_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("TFT 완화 후보 수:", len(tft_team_candidate_loose_df))
print("재정제 후 tft_dedup_df2 후보 수:", len(tft_dedup_df2))
print("예상 parse 소요 시간:", round(len(tft_dedup_df2) * PARSE_SLEEP / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(tft_dedup_df2) * PARSE_SLEEP / 3600, 2), "시간")

display(tft_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

TFT 완화 후보 수: 94
재정제 후 tft_dedup_df2 후보 수: 94
예상 parse 소요 시간: 48.6 분
예상 parse 소요 시간: 0.81 시간


,title,snippet,search_keyword,url
0,TFT APAC Esports,"#2 May 24–25, 2025 South Korea 112 엉망진창 메시 화내면...",korea,https://liquipedia.net/tft/TFT_APAC_Esports
1,Binteum,Romanized Name: Kang Seong-jun Nationality: So...,korea,https://liquipedia.net/tft/Binteum
2,Ssang Yeop,Romanized Name: Seo Sung-won Nationality: Sout...,korea,https://liquipedia.net/tft/Ssang_Yeop
3,SCsC,Romanized Name: Kim Seung-chul Nationality: So...,korea,https://liquipedia.net/tft/SCsC
4,Dunizuni,조준희 Romanized Name: Cho Jun-hee Nationality: S...,korea,https://liquipedia.net/tft/Dunizuni
...,...,...,...,...
89,Ikura,Gwang-won Nationality: South Korea Region: Kor...,korea,https://liquipedia.net/tft/Ikura
90,LeVition,Information Nationality: South Korea Region: K...,korea,https://liquipedia.net/tft/LeVition
91,Chabo Kim,Nationality: South Korea Region: Korea Status:...,korea,https://liquipedia.net/tft/Chabo_Kim
92,Yummy chidon,Information Nationality: South Korea Region: K...,korea,https://liquipedia.net/tft/Yummy_chidon


In [ ]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [ ]:
def extract_tft_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    hard_exclude_keywords = [
        "Results",
        "Data",
        "Gallery",
        "Documentaries",
        "References",
        "Achievements",
        "Awards",
        "Statistics",
        "Upcoming Matches",
        "Upcoming Tournaments",
        "Recent Matches",
        "Trivia",
        "Logos",
        "Rosters",
    ]

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "tft", "teamfight tactics", "liquipedia",
        "show all", "achievements", "recent matches",
        "individual", "team",
    ]

    def get_section_heading(el):
        prev = el.find_previous(["h2", "h3", "h4"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_previous_subheading(el):
        prev = el.find_previous(["h3", "h4", "h5"])
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_country_from_element(el):
        countries = []
        for img in el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if value and value not in countries:
                    if not value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                        countries.append(value)
        return ", ".join(countries)

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player", "handle"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["real name", "real_name"]:
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h in ["inactive date"]:
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["replacing"]:
            return "replacing"
        if h in ["tournament(s)", "tournaments"]:
            return "tournaments"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"

        # Organization 표는 ID / Name / 역할 / Join Date 구조인데 역할 헤더가 빈칸일 수 있음
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/File:", "/Category:", "/Special:", "/Help:", "/Template:",
            "/Portal:", "/Liquipedia:", "/User:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|regional|tactician|golden spatula|pro circuit|coliseum)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def classify_table(heading, subheading, table_text):
        combined = f"{heading} {subheading} {table_text}".lower()

        if any(k.lower() in combined for k in hard_exclude_keywords):
            if (
                "player roster" not in combined
                and "organization" not in combined
                and "former players" not in combined
                and "active players" not in combined
                and "former organization" not in combined
            ):
                return None, None

        is_org = (
            "organization" in combined
            or "former organization" in combined
            or "staff" in combined
            or "management" in combined
        )

        is_player = (
            "player roster" in combined
            or "active players" in combined
            or "former players" in combined
            or "inactive players" in combined
            or "temporary stand-ins" in combined
            or "stand-in" in combined
        )

        if is_org:
            if "former" in combined:
                return "organization", "Former"
            return "organization", "Active"

        if is_player:
            if "inactive" in combined:
                return "player_roster", "Inactive"
            if "former" in combined:
                return "player_roster", "Former"
            if "stand-in" in combined or "temporary stand-ins" in combined:
                return "player_roster", "Stand-in"
            return "player_roster", "Active"

        return None, None

    def default_position(table_type, status, row_data):
        # Organization이면 표의 역할 컬럼을 position으로 사용
        if table_type == "organization":
            pos = row_data.get("position", "")
            if pos:
                return pos
            if status == "Former":
                return "Former Staff"
            return "Staff"

        # Player Roster면 역할이 없으므로 상태 기반으로 position 채움
        if table_type == "player_roster":
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            if status == "Stand-in":
                return "Stand-in"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        heading = get_section_heading(table)
        subheading = get_previous_subheading(table)
        table_text = clean_text(table.get_text(" ", strip=True))

        table_type, status = classify_table(heading, subheading, table_text)

        if table_type is None:
            continue

        trs = table.find_all("tr")
        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:12]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
                or "tournament(s)" in lowered
                or "role" in lowered
                or "position" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        # Organization 표에서 빈 역할 헤더 보정
        if table_type == "organization":
            if "position" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])
            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))
            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_country_from_element(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": heading,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "replacing": row_data.get("replacing", ""),
                "tournaments": row_data.get("tournaments", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
        ]
    ).reset_index(drop=True)

    return df

In [ ]:
test_team = "FN Esports"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_tft_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url",
            "row_text"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: FN Esports
페이지 존재 여부: True
에러: None
추출 row 수: 0
페이지는 열렸지만 row가 추출되지 않았습니다.


In [ ]:
def extract_tft_people_from_team_html_v5(team_title, html, debug=False):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_country_from_element(el):
        countries = []
        for img in el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if value and value not in countries:
                    if not value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                        countries.append(value)
        return ", ".join(countries)

    def get_context_text(el):
        """
        TFT 페이지는 표 바로 위 heading이 h2/h3로 깔끔하게 안 잡히는 경우가 있어서
        앞쪽 텍스트를 넓게 가져와 Player Roster / Organization / Active / Former 판단에 사용.
        """
        context_parts = []

        cur = el
        count = 0

        while cur and count < 18:
            cur = cur.find_previous()
            if not cur:
                break

            if cur.name in ["h1", "h2", "h3", "h4", "h5", "span", "div", "p"]:
                txt = clean_text(cur.get_text(" ", strip=True)).replace("[edit]", "")
                if txt and len(txt) < 300:
                    context_parts.append(txt)
                    count += 1

        context = " ".join(reversed(context_parts))
        return clean_text(context)

    def normalize_header(h):
        h = clean_text(h).lower()
        h = h.replace("\xa0", " ")

        if h in ["id", "player", "handle"]:
            return "id"
        if h in ["name", "real name", "real_name"]:
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h in ["inactive date"]:
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h in ["tournament(s)", "tournaments"]:
            return "tournaments"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def classify_from_context(context, table_text):
        combined = f"{context} {table_text}".lower()

        # 너무 넓게 잡힌 achievements/results/data 쪽 제외
        if any(x in combined for x in [
            "date place tier tournament player prize",
            "achievements",
            "individual",
            "team date place tier",
            "gallery",
            "references",
            "results",
            "data",
            "logos",
        ]):
            # 단, roster/organization 키워드가 명확하면 살림
            if not any(y in combined for y in [
                "player roster",
                "former players",
                "active players",
                "organization",
                "former organization",
            ]):
                return None, None

        is_org = any(x in combined for x in [
            "organization",
            "former organization",
            "active organization",
        ])

        is_player = any(x in combined for x in [
            "player roster",
            "former players",
            "active players",
            "inactive players",
            "temporary stand-ins",
            "stand-in",
        ])

        if is_org:
            if "former organization" in combined or " former " in combined:
                return "organization", "Former"
            return "organization", "Active"

        if is_player:
            if "inactive" in combined:
                return "player_roster", "Inactive"
            if "former players" in combined or " former " in combined:
                return "player_roster", "Former"
            if "stand-in" in combined or "temporary stand-ins" in combined:
                return "player_roster", "Stand-in"
            return "player_roster", "Active"

        return None, None

    def default_position(table_type, status, row_data):
        if table_type == "organization":
            pos = row_data.get("position", "")
            if pos:
                return pos
            if status == "Former":
                return "Former Staff"
            return "Staff"

        if table_type == "player_roster":
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            if status == "Stand-in":
                return "Stand-in"
            return "Player"

        return ""

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        bad_exact = [
            "edit", "view", "history", "team", "teams", "player", "players",
            "tournament", "tournaments", "overview", "results", "matches",
            "statistics", "schedule", "vods", "references", "news",
            "main page", "tft", "teamfight tactics", "liquipedia",
            "show all", "achievements", "recent matches",
            "individual",
        ]

        if name_lower in bad_exact:
            return True

        if any(x.lower() in href_lower for x in [
            "/File:", "/Category:", "/Special:", "/Help:", "/Template:",
            "/Portal:", "/Liquipedia:", "/User:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|regional|tactician|golden spatula|pro circuit|coliseum)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def extract_person_link_from_row(tr):
        links = []

        for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
            link_text = clean_text(a.get_text(" ", strip=True))
            href = a.get("href", "")

            if is_bad_person_link(link_text, href):
                continue

            links.append({
                "name_from_link": link_text,
                "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                "url": "https://liquipedia.net" + href,
            })

        if links:
            return links[0]

        return None

    def parse_standard_table(table, table_idx):
        context = get_context_text(table)
        table_text = clean_text(table.get_text(" ", strip=True))
        table_type, status = classify_from_context(context, table_text)

        if table_type is None:
            return []

        trs = table.find_all("tr")
        if not trs:
            return []

        parsed_rows = []

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:15]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if any(x in lowered for x in [
                "id", "player", "name", "join date", "leave date",
                "inactive date", "new team", "role", "position"
            ]):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            return []

        normalized_headers = [normalize_header(h) for h in headers]

        # Organization 표: ID / Name / 역할 / Join Date 구조인데 역할 헤더가 비어 있거나 누락될 수 있음
        if table_type == "organization":
            if "position" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])
            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))
            if not row_text or len(cells) == 1:
                continue

            # 연도 탭 / Show All 같은 줄 제외
            if row_text.lower() in ["show all"] or re.fullmatch(r"20\d{2}", row_text):
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            main_link = extract_person_link_from_row(tr)

            if main_link:
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = row_data.get("nationality", "") or extract_country_from_element(tr)
            position = default_position(table_type, status, row_data)

            parsed_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": context,
                "subheading": "",
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "replacing": row_data.get("replacing", ""),
                "tournaments": row_data.get("tournaments", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

        return parsed_rows

    # 1차: 일반 table 기반 추출
    for table_idx, table in enumerate(soup.find_all("table")):
        extracted = parse_standard_table(table, table_idx)
        rows.extend(extracted)

    # 2차 fallback:
    # 일부 TFT 페이지에서 표 context 판정이 꼬일 경우,
    # ID/Name/Join Date가 있는 table만 다시 직접 추출
    if not rows:
        if debug:
            print("1차 table 추출 실패. fallback 진입")

        for table_idx, table in enumerate(soup.find_all("table")):
            table_text = clean_text(table.get_text(" ", strip=True))
            table_text_lower = table_text.lower()

            if not (
                "id" in table_text_lower
                and "name" in table_text_lower
                and "join date" in table_text_lower
            ):
                continue

            if "date place tier tournament player prize" in table_text_lower:
                continue

            context = get_context_text(table)
            context_lower = context.lower()

            # fallback에서는 table 내용으로도 강제 판정
            forced_context = f"{context} {table_text[:300]}"

            table_type, status = classify_from_context(forced_context, table_text)

            if table_type is None:
                # FN Esports 같은 경우 Former Players / Organization 텍스트가 붙어 있을 수 있음
                if "former players" in table_text_lower:
                    table_type, status = "player_roster", "Former"
                elif "organization" in table_text_lower or "general manager" in table_text_lower or "ceo" in table_text_lower:
                    table_type, status = "organization", "Active"
                elif "former organization" in table_text_lower or "analyst" in table_text_lower:
                    table_type, status = "organization", "Former"
                else:
                    continue

            trs = table.find_all("tr")
            if not trs:
                continue

            headers = []
            header_row_idx = None

            for i, tr in enumerate(trs[:15]):
                cells = tr.find_all(["th", "td"])
                cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
                lowered = [c.lower() for c in cell_texts]

                if "id" in lowered and "name" in lowered:
                    headers = cell_texts
                    header_row_idx = i
                    break

            if not headers:
                continue

            normalized_headers = [normalize_header(h) for h in headers]

            if table_type == "organization":
                if "position" not in normalized_headers and len(normalized_headers) >= 4:
                    normalized_headers[2] = "position"

            for tr in trs[(header_row_idx + 1):]:
                cells = tr.find_all(["td", "th"])
                if not cells:
                    continue

                row_text = clean_text(tr.get_text(" ", strip=True))
                if not row_text or len(cells) == 1:
                    continue

                if row_text.lower() in ["show all"] or re.fullmatch(r"20\d{2}", row_text):
                    continue

                row_data = {}

                for idx, cell in enumerate(cells):
                    if idx >= len(normalized_headers):
                        continue

                    col = normalized_headers[idx]
                    value = clean_text(cell.get_text(" ", strip=True))

                    if col in ["join_date", "leave_date", "inactive_date"]:
                        value = clean_ref_text(value)

                    row_data[col] = value

                main_link = extract_person_link_from_row(tr)

                if main_link:
                    person_id = row_data.get("id", "") or main_link["name_from_link"]
                    person_page = main_link["page"]
                    person_url = main_link["url"]
                else:
                    person_id = row_data.get("id", "")
                    person_page = ""
                    person_url = ""

                if not person_id:
                    continue

                nationality = row_data.get("nationality", "") or extract_country_from_element(tr)
                position = default_position(table_type, status, row_data)

                rows.append({
                    "game_slug": GAME_SLUG,
                    "game_label": GAME_LABEL,
                    "team_title": team_title,
                    "team_url": make_page_url(team_title),
                    "section_heading": context,
                    "subheading": "",
                    "table_type": table_type,
                    "status": status,
                    "table_index": table_idx,
                    "id": person_id,
                    "name": row_data.get("name", ""),
                    "position": position,
                    "join_date": row_data.get("join_date", ""),
                    "inactive_date": row_data.get("inactive_date", ""),
                    "leave_date": row_data.get("leave_date", ""),
                    "new_team": row_data.get("new_team", ""),
                    "replacing": row_data.get("replacing", ""),
                    "tournaments": row_data.get("tournaments", ""),
                    "nationality": nationality,
                    "person_page": person_page,
                    "person_url": person_url,
                    "row_text": row_text,
                })

    if debug:
        print("최종 추출 rows:", len(rows))

        if len(rows) == 0:
            print("디버그: table 개수:", len(soup.find_all("table")))
            for i, table in enumerate(soup.find_all("table")[:20]):
                txt = clean_text(table.get_text(" ", strip=True))
                print("=" * 80)
                print("table index:", i)
                print(txt[:500])

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
        ]
    ).reset_index(drop=True)

    return df

In [ ]:
test_team = "FN Esports"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_tft_people_from_team_html_v5(
        team_title=test_team,
        html=page["html"],
        debug=True
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url",
            "row_text"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: FN Esports
페이지 존재 여부: True
에러: None
최종 추출 rows: 20
추출 row 수: 19


,team_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url,row_text
0,FN Esports,player_roster,Former,Panda,Kim Se-jin,Former Player,2024-01-09 [ 1 ],,2024-12-30 [ 9 ],ROC Esports,"South Korea, ROC Esports",https://liquipedia.net/tft/Panda_(Kim_Se-jin),Panda Kim Se-jin 2024-01-09 [ 1 ] 2024-12-30 [...
1,FN Esports,player_roster,Former,Elmumu,Na Daniel,Former Player,2024-05-31 [ 3 ],,2024-09-10 [ 7 ],,South Korea,https://liquipedia.net/tft/Elmumu,Elmumu Na Daniel 2024-05-31 [ 3 ] 2024-09-10 [...
2,FN Esports,player_roster,Former,Dr OH,Oh Se-jin,Former Player,2024-01-09 [ 1 ],,2024-05-09 [ 2 ],ROC Esports,"South Korea, ROC Esports",https://liquipedia.net/tft/Dr_OH,Dr OH Oh Se-jin 2024-01-09 [ 1 ] 2024-05-09 [ ...
3,FN Esports,player_roster,Former,Ahn,Ahn Jung-hyun,Former Player,2024-01-09 [ 1 ],,2025-04-08 [ 11 ],ONIC Esports,"South Korea, ONIC Esports",https://liquipedia.net/tft/Ahn,Ahn Ahn Jung-hyun 2024-01-09 [ 1 ] 2025-04-08 ...
4,FN Esports,player_roster,Former,Cosmo,Ki Hyeon-oh,Former Player,2024-01-09 [ 1 ],,2025-03-02 [ 10 ],ONIC Esports,"South Korea, ONIC Esports",https://liquipedia.net/tft/Cosmo,Cosmo Ki Hyeon-oh 2024-01-09 [ 1 ] 2025-03-02 ...
5,FN Esports,player_roster,Former,Ssiel,Kim Chan-yeong,Former Player,2024-09-11 [ 8 ],,2026-01-09 [ 12 ],,South Korea,https://liquipedia.net/tft/Ssiel,Ssiel Kim Chan-yeong 2024-09-11 [ 8 ] 2026-01-...
6,FN Esports,organization,Active,Ma3Str0,Oh Jong-yong,General Manager,2024-01-09,,,,South Korea,https://liquipedia.net/tft/index.php?title=Ma3...,Ma3Str0 Oh Jong-yong General Manager 2024-01-09
7,FN Esports,organization,Active,윤창환,Yun Chang-hwan,CEO & Owner,2024-01-09,,,,South Korea,https://liquipedia.net/tft/index.php?title=%EC...,윤창환 Yun Chang-hwan CEO & Owner 2024-01-09
8,FN Esports,organization,Former,SanChess,Jeong Hui-hun,Coach,2024-07-?? [ 4 ],,2024-07-20 [ 6 ],ROC Esports,"South Korea, ROC Esports",https://liquipedia.net/tft/SanChess,SanChess Jeong Hui-hun Coach 2024-07-?? [ 4 ] ...
9,FN Esports,organization,Former,Dori,Kim Jong-woong,Analyst,2024-07-06 [ 5 ],,2025-03-02 [ 10 ],T1 (Coach),"South Korea, T1",https://liquipedia.net/tft/Dori,Dori Kim Jong-woong Analyst 2024-07-06 [ 5 ] 2...


In [ ]:
def crawl_tft_teams_and_people_v5(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * PARSE_SLEEP / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * PARSE_SLEEP / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v5"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "game_slug": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        # TFT 예시 FN Esports처럼 Location: South Korea / Region: Asia-Pacific 구조
        # 따라서 region:korea가 아니라 location:southkorea / south korea 중심으로 검증
        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "game_slug": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_tft_people_from_team_html_v5(
                    team_title=page["title"],
                    html=page["html"],
                    debug=False
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [ ]:
tft_teams_checked_df, tft_teams_final_df, tft_people_raw_df, tft_errors_df = crawl_tft_teams_and_people_v5(
    candidate_df=tft_dedup_df2,
    output_prefix="tft_korean_mainteam_refined_v5",
    team_type="main_team_refined"
)

display(tft_teams_final_df.head(100))
display(tft_people_raw_df.head(100))

크롤링 대상 후보 수: 94
예상 소요 시간: 48.6 분
예상 소요 시간: 0.81 시간


tft_korean_mainteam_refined_v5 팀 검증 + 인원 추출 v5:   0%|          | 0/94 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 4
추출 인원 row 수: 47
에러 수: 0
저장 완료
/content/tft_korean_mainteam_refined_v5_teams_checked.csv
/content/tft_korean_mainteam_refined_v5_teams_final.csv
/content/tft_korean_mainteam_refined_v5_people_raw.csv
/content/tft_korean_mainteam_refined_v5_errors.csv


,game_slug,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,tft,Teamfight Tactics,T1,T1,https://liquipedia.net/tft/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
1,tft,Teamfight Tactics,FN Esports,FN Esports,https://liquipedia.net/tft/FN_Esports,True,True,[ e ][ h ] FN Esports Team Information Locatio...,None,main_team_refined
2,tft,Teamfight Tactics,DH.CNJ esports,DH.CNJ esports,https://liquipedia.net/tft/DH.CNJ_esports,True,True,[ e ][ h ] DH.CNJ esports Team Information Loc...,None,main_team_refined
3,tft,Teamfight Tactics,Nongshim RedForce,Nongshim RedForce,https://liquipedia.net/tft/Nongshim_RedForce,True,True,[ e ][ h ] Nongshim RedForce Team Information ...,None,main_team_refined


,game_slug,game_label,team_title,team_url,section_heading,subheading,table_type,status,table_index,id,...,leave_date,new_team,replacing,tournaments,nationality,person_page,person_url,row_text,team_display_title,team_type
0,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,[ ] [ ] [ ] Player Roster [ edit ] Player Rost...,,player_roster,Active,0,Binteum,...,,,,,South Korea,Binteum,https://liquipedia.net/tft/Binteum,Binteum Kang Seong-jun 2024-05-08 [ 4 ],T1,main_team_refined
1,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,[ ] [ ] [ ] Player Roster [ edit ] Player Rost...,,player_roster,Active,0,sCsC,...,,,,,South Korea,SCsC,https://liquipedia.net/tft/SCsC,sCsC Kim Seung-chul 2024-05-08 [ 4 ],T1,main_team_refined
2,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,[ ] [ ] [ ] Player Roster [ edit ] Player Rost...,,player_roster,Active,0,dunizuni,...,,,,,South Korea,Dunizuni,https://liquipedia.net/tft/Dunizuni,dunizuni Cho Jun-hee 2024-06-03 [ 5 ],T1,main_team_refined
3,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,[ ] [ ] [ ] Player Roster [ edit ] Player Rost...,,player_roster,Active,0,Ssang Yeop,...,,,,,South Korea,Ssang Yeop,https://liquipedia.net/tft/Ssang_Yeop,Ssang Yeop Seo Sung-won 2026-01-15 [ 11 ],T1,main_team_refined
4,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,[ ] [ ] [ ] Player Roster [ edit ] Player Rost...,,player_roster,Active,0,CrazyMoving,...,,,,,South Korea,CrazyMoving,https://liquipedia.net/tft/CrazyMoving,CrazyMoving Han Ki-soo 2026-03-23 [ 12 ],T1,main_team_refined
5,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,[ ] Former [ edit ] Former [ edit ] [ ] 2025 2...,,player_roster,Former,1,bobae,...,2025-02-25 [ 7 ],,,,South Korea,Bobae,https://liquipedia.net/tft/Bobae,bobae Kim Hwi-gang 2024-06-03 [ 5 ] 2025-02-25...,T1,main_team_refined
6,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,2025 2026 Show All Former Players ID Name Join...,,player_roster,Former,2,CrazyMoving,...,2026-01-07 [ 10 ],T1,,,"South Korea, T1",CrazyMoving,https://liquipedia.net/tft/CrazyMoving,CrazyMoving Han Ki-soo 2025-04-01 [ 9 ] 2026-0...,T1,main_team_refined
7,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,] [ ] T1 T1 Organization [ edit ] Organization...,,organization,Active,3,Joe,...,,,,,United States,index.php?title=Joe&action=edit&redlink=1,https://liquipedia.net/tft/index.php?title=Joe...,Joe Joseph Patrik Marsh CEO 2019-10-08,T1,main_team_refined
8,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,] [ ] T1 T1 Organization [ edit ] Organization...,,organization,Active,3,Faker,...,,,,,South Korea,,,Faker Lee Sang-hyeok Part-Owner 2019-10-08,T1,main_team_refined
9,tft,Teamfight Tactics,T1,https://liquipedia.net/tft/T1,] [ ] T1 T1 Organization [ edit ] Organization...,,organization,Active,3,BoxeR,...,,,,,South Korea,,,BoxeR Lim Yo-hwan Founder & Streamer 2019-10-08,T1,main_team_refined


In [ ]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    # -----------------------------------------
    # 1. 사람 unique key 생성
    # -----------------------------------------
    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    # URL이 없으면 id + name 기준
    # position은 제외: 같은 사람이 Player였다가 Coach가 되어도 사람 단위로 1명만 남기기 위해서
    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    # id/name도 비어 있는 이상치 제거
    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    # -----------------------------------------
    # 2. 날짜 파싱
    # -----------------------------------------
    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    # join_date가 없으면 inactive_date, 그것도 없으면 leave_date를 최신성 판단 날짜로 사용
    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    # -----------------------------------------
    # 3. status 우선순위
    # -----------------------------------------
    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    # position에도 Stand-in / Inactive가 적혀 있으면 우선순위 보정
    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    # -----------------------------------------
    # 4. 최신 이력이 위로 오도록 정렬
    # -----------------------------------------
    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    # -----------------------------------------
    # 5. 사람당 1행만 남김
    # -----------------------------------------
    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    # 보조 컬럼 제거
    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [ ]:
tft_people_unique_df = make_unique_people_latest_from_raw(tft_people_raw_df)

print("raw row 수:", len(tft_people_raw_df))
print("최신 기준 unique 인원 수:", len(tft_people_unique_df))

display(tft_people_unique_df[[
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 47
최신 기준 unique 인원 수: 37


,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,T1,Active,BoxeR,Lim Yo-hwan,Founder & Streamer,2019-10-08,,,,South Korea,
1,T1,Active,Faker,Lee Sang-hyeok,Part-Owner,2019-10-08,,,,South Korea,
2,FN Esports,Former,K.O. Coliseum: APAC Regional Finals,,A-Tier,,,,,"K.O. Coliseum: APAC Regional Finals, South Korea",https://liquipedia.net/tft/A-Tier_Tournaments
3,FN Esports,Former,Ahn,Ahn Jung-hyun,Former Player,2024-01-09 [ 1 ],,2025-04-08 [ 11 ],ONIC Esports,"South Korea, ONIC Esports",https://liquipedia.net/tft/Ahn
4,DH.CNJ esports,Active,TFT Champions Cup: CN 4th Year Invitational,,B-Tier,,,,,"TFT Champions Cup: CN 4th Year Invitational, S...",https://liquipedia.net/tft/B-Tier_Tournaments
5,T1,Former,Bebe872,Kim Kyu-yeon,Streamer,2021-01-01 [ 2 ],,2022-03-31 [ 3 ],,South Korea,https://liquipedia.net/tft/Bebe872
6,T1,Active,Binteum,Kang Seong-jun,Player,2024-05-08 [ 4 ],,,,South Korea,https://liquipedia.net/tft/Binteum
7,T1,Former,bobae,Kim Hwi-gang,Former Player,2024-06-03 [ 5 ],,2025-02-25 [ 7 ],,South Korea,https://liquipedia.net/tft/Bobae
8,FN Esports,Former,SOOP TFT Series: Into the Arcane - Finals,,C-Tier,,,,,"SOOP TFT Series: Into the Arcane - Finals, Sou...",https://liquipedia.net/tft/C-Tier_Tournaments
9,Nongshim RedForce,Former,Cassigod,Hwang Yun-jin,Coach,2026-05-08 [ 1 ],,,,South Korea,https://liquipedia.net/tft/Cassigod


In [ ]:
tft_teams_final_df.to_csv(
    "/content/tft_korean_teams_final.csv",
    index=False,
    encoding="utf-8-sig"
)

tft_people_unique_df.to_csv(
    "/content/tft_korean_people_unique.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/tft_korean_teams_final.csv")
print("/content/tft_korean_people_unique.csv")

저장 완료
/content/tft_korean_teams_final.csv
/content/tft_korean_people_unique.csv


#스매시브라더스

In [3]:
# =========================================
# Smash Liquipedia 설정
# =========================================

GAME_SLUG = "smash"
GAME_LABEL = "Super Smash Bros."

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanSmashEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/smash
https://liquipedia.net/smash/api.php


In [4]:
def clean_text(text):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def make_page_url(title):
    title = title or ""
    return f"{BASE_URL}/{quote(title.replace(' ', '_'))}"


def api_get(params, sleep=REQUEST_SLEEP):
    params = {
        **params,
        "format": "json"
    }

    r = requests.get(
        API_URL,
        params=params,
        headers=HEADERS,
        timeout=40
    )

    time.sleep(sleep)

    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    data = r.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data

In [5]:
data = api_get({
    "action": "query",
    "meta": "siteinfo",
    "siprop": "general"
})

data["query"]["general"]["sitename"]

'Liquipedia Smash Wiki'

In [6]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Smash Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [7]:
SMASH_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean player",
    "Korean team",
    "Korea esports",

    "Super Smash Bros Korea",
    "Super Smash Bros Korean",
    "Smash Korea",
    "Smash Korean",
    "Melee Korea",
    "Ultimate Korea",

    # 한국/아시아권 관련 후보
    "CJ eSports",
    "CaptainJack",
    "aMSa",
    "Abu",
    "Sanne",
    "Hero",
    "BUZZ e-sports",
    "DetonatioN",
    "DRX",
    "T1",
    "Gen.G",
]

In [8]:
search_rows = []

for keyword in SMASH_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

smash_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(smash_search_df))
display(smash_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: Korean / offset=150
검색: Korean / offset=200
검색: Korean / offset=250
검색: Korean / offset=300
검색: Korean / offset=350
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: Korean player / offset=0
검색: Korean player / offset=50
검색: Korean player / offset=100
검색: Korean player / offset=150
검색: Korean team / offset=0
검색: Korean team / offset=50
검색: Korean team / offset=100
검색: Korea esports / offset=0
검색: Korea esports / offset=50
검색: Super Smash Bros Korea / offset=0
검색: Super Smash Bros Korea / offset=50
검색: Super Smash Bros Korea / offset=100
검색: Super Smash Bros Korean / offset=0
검색: Super Smash Bros Korean / offset=50
검색: Super Smash Bros Korean / offset=100
검색: Smash Korea /

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,smash,Super Smash Bros.,South Korea Power Rankings,gg/rankings/super-smash-bros-ultimate/series/s...,28860,https://liquipedia.net/smash/South_Korea_Power...,korea
1,smash,Super Smash Bros.,Asia & Oceania Rankings,Maldives Mongolia Myanmar Nepal North Korea Om...,44646,https://liquipedia.net/smash/Asia_%26_Oceania_...,korea
2,smash,Super Smash Bros.,JJROCKETS,currently living in South Korea. He was ranked...,18171,https://liquipedia.net/smash/JJROCKETS,korea
3,smash,Super Smash Bros.,Team Liquid,beta in 2010 and became one of the most succes...,39,https://liquipedia.net/smash/Team_Liquid,korea
4,smash,Super Smash Bros.,Schima,&quot;Schima&quot; is a Melee Fox player from ...,42880,https://liquipedia.net/smash/Schima,korea
...,...,...,...,...,...,...,...
95,smash,Super Smash Bros.,KoreanDJ/Doubles Results,R725 - 32nd Apex 2015 KoreanDJ Ken L Axe Tagle...,32157,https://liquipedia.net/smash/KoreanDJ/Doubles_...,korea
96,smash,Super Smash Bros.,KoreanDJ/Notable Wins,Rank 26 - Chillindude 2014-01-19 | W : L at Ap...,49965,https://liquipedia.net/smash/KoreanDJ/Notable_...,korea
97,smash,Super Smash Bros.,Nouns Bowl/2025/Melee/Singles Pools/B2,YungWaff 3 Wink 0 SwebBy 3 SwebBy korean maher...,63633,https://liquipedia.net/smash/Nouns_Bowl/2025/M...,korea
98,smash,Super Smash Bros.,Mass Madness/11a/Singles Bracket,"Winners KoreanDJ 0 KoreanDJ Hax February 22, 2...",40164,https://liquipedia.net/smash/Mass_Madness/11a/...,korea


In [9]:
def build_smash_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "manager",

        "cj esports",
        "buzz",
        "detonation",
        "drx",
        "t1",
        "gen.g",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "matches",
        "rankings",
        "awards",
        "finals",
    ]

    bad_markers = [
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        # 팀 후보 또는 한국 관련 선수/팀 후보
        korea_terms = [
            "south korea",
            "korea",
            "korean",
            "japan",
            "japanese",
        ]

        has_korea_or_region = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text:
            return True

        if has_korea_or_region and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Smash 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [10]:
smash_team_candidate_loose_df = build_smash_team_candidate_loose_df(smash_search_df)

display(smash_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Smash 완화 후보 수: 13
예상 parse 소요 시간: 6.7 분
예상 parse 소요 시간: 0.11 시간


,title,snippet,search_keyword,url
0,Team Liquid,beta in 2010 and became one of the most succes...,korea,https://liquipedia.net/smash/Team_Liquid
1,T1,T1 Team Information Location: South Korea Unit...,korea,https://liquipedia.net/smash/T1
2,KoreanDJ,Melee: History Daniel &quot;KoreanDJ&quot; Jun...,korea,https://liquipedia.net/smash/KoreanDJ
3,Gen.G Esports,Gen.G Esports Team Information Location: Unite...,korea,https://liquipedia.net/smash/Gen.G_Esports
4,Ken (Melee player),"based on entrants. On March 18, 2014, Team Liq...",korea,https://liquipedia.net/smash/Ken_%28Melee_play...
5,Hero (Japanese player),ranked 106th in UltRank 2025. He joined CJ eSp...,CJ eSports,https://liquipedia.net/smash/Hero_%28Japanese_...
6,Sanne,History Sanne is a Melee Falco main from Japan...,CJ eSports,https://liquipedia.net/smash/Sanne
7,JAPAN 24,MASA Masha 9th-12th - KEN SBI e-Sports Omuatsu...,CJ eSports,https://liquipedia.net/smash/JAPAN_24
8,BUZZ e-sports,2023-04-29 BUZZ e-sports is a Japanese esports...,Hero,https://liquipedia.net/smash/BUZZ_e-sports
9,Neo (Japanese player),"currently ranked 28th in UltRank 2025, He join...",BUZZ e-sports,https://liquipedia.net/smash/Neo_%28Japanese_p...


In [11]:
# =========================================
# Smash 완화 후보에서 팀 후보만 재정제
# output: smash_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = smash_team_candidate_loose_df.copy()

smash_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "cj esports",
    "buzz",
    "detonation",
    "drx",
    "t1",
    "gen.g",
]

smash_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
]

smash_bad_page_markers = [
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]


def is_refined_smash_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in smash_bad_page_markers):
        return False

    for pattern in smash_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in smash_team_name_keywords):
        return True

    return False


smash_dedup_df2 = source_df[
    source_df.apply(is_refined_smash_team_candidate, axis=1)
].copy().reset_index(drop=True)

smash_dedup_df2["title_clean"] = (
    smash_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

smash_dedup_df2 = smash_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Smash 완화 후보 수:", len(smash_team_candidate_loose_df))
print("재정제 후 smash_dedup_df2 후보 수:", len(smash_dedup_df2))
print("예상 parse 소요 시간:", round(len(smash_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(smash_dedup_df2) * 31 / 3600, 2), "시간")

display(smash_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Smash 완화 후보 수: 13
재정제 후 smash_dedup_df2 후보 수: 13
예상 parse 소요 시간: 6.7 분
예상 parse 소요 시간: 0.11 시간


,title,snippet,search_keyword,url
0,Team Liquid,beta in 2010 and became one of the most succes...,korea,https://liquipedia.net/smash/Team_Liquid
1,T1,T1 Team Information Location: South Korea Unit...,korea,https://liquipedia.net/smash/T1
2,KoreanDJ,Melee: History Daniel &quot;KoreanDJ&quot; Jun...,korea,https://liquipedia.net/smash/KoreanDJ
3,Gen.G Esports,Gen.G Esports Team Information Location: Unite...,korea,https://liquipedia.net/smash/Gen.G_Esports
4,Ken (Melee player),"based on entrants. On March 18, 2014, Team Liq...",korea,https://liquipedia.net/smash/Ken_%28Melee_play...
5,Hero (Japanese player),ranked 106th in UltRank 2025. He joined CJ eSp...,CJ eSports,https://liquipedia.net/smash/Hero_%28Japanese_...
6,Sanne,History Sanne is a Melee Falco main from Japan...,CJ eSports,https://liquipedia.net/smash/Sanne
7,JAPAN 24,MASA Masha 9th-12th - KEN SBI e-Sports Omuatsu...,CJ eSports,https://liquipedia.net/smash/JAPAN_24
8,BUZZ e-sports,2023-04-29 BUZZ e-sports is a Japanese esports...,Hero,https://liquipedia.net/smash/BUZZ_e-sports
9,Neo (Japanese player),"currently ranked 28th in UltRank 2025, He join...",BUZZ e-sports,https://liquipedia.net/smash/Neo_%28Japanese_p...


In [12]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [13]:
def infer_smash_game_title(text):
    text = clean_text(text).lower()

    if "ultimate" in text:
        return "Super Smash Bros. Ultimate"

    if "melee" in text:
        return "Super Smash Bros. Melee"

    if "brawl" in text:
        return "Super Smash Bros. Brawl"

    if "smash 4" in text or "wii u" in text:
        return "Super Smash Bros. for Wii U"

    if "64" in text:
        return "Super Smash Bros. 64"

    return "Super Smash Bros."

In [14]:
def infer_smash_game_title(text):
    text = clean_text(text).lower()

    if "ultimate" in text:
        return "Super Smash Bros. Ultimate"

    if "melee" in text:
        return "Super Smash Bros. Melee"

    if "brawl" in text:
        return "Super Smash Bros. Brawl"

    if "smash 4" in text or "wii u" in text:
        return "Super Smash Bros. for Wii U"

    if "64" in text:
        return "Super Smash Bros. 64"

    return "Super Smash Bros."

In [15]:
def extract_field_from_infobox_text(infobox_text, field_names):
    text = clean_text(infobox_text)

    all_fields = [
        "Location",
        "Manager",
        "Coach",
        "Owner",
        "Captain",
        "Region",
        "Approx. Total Winnings",
        "Links",
        "History",
        "Created",
    ]

    for field in field_names:
        pattern = (
            rf"{re.escape(field)}:\s*(.*?)\s*(?="
            + "|".join([re.escape(f) + ":" for f in all_fields if f != field])
            + r"|$)"
        )

        m = re.search(pattern, text, flags=re.IGNORECASE)

        if m:
            return clean_text(m.group(1))

    return ""


def extract_infobox_links(html):
    soup = BeautifulSoup(html, "lxml")
    infobox = None

    for selector in [".fo-nttax-infobox", ".fo-nttax-infobox-wrapper", ".infobox", "table.infobox"]:
        infobox = soup.select_one(selector)
        if infobox:
            break

    links = []

    if not infobox:
        return links

    for a in infobox.select(f'a[href^="/{GAME_SLUG}/"]'):
        link_text = clean_text(a.get_text(" ", strip=True))
        href = a.get("href", "")

        if not link_text:
            continue

        links.append({
            "text": link_text,
            "url": "https://liquipedia.net" + href,
            "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
        })

    return links

In [16]:
def extract_smash_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    infobox_text = extract_infobox_text(html)
    location = extract_field_from_infobox_text(infobox_text, ["Location"])
    manager_text = extract_field_from_infobox_text(infobox_text, ["Manager"])
    winnings = extract_field_from_infobox_text(infobox_text, ["Approx. Total Winnings"])

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "smash", "liquipedia",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player", "tag"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)
        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if "player roster" not in main_low:
            return None, None, None

        if "former" in sub_low:
            status = "Former"
            position = "Former Player"
        else:
            status = "Active"
            position = "Player"

        game_title = infer_smash_game_title(sub_section)

        return status, position, game_title

    # -----------------------------------------
    # 1. Infobox Manager를 staff로 추가
    # -----------------------------------------
    infobox_links = extract_infobox_links(html)

    if manager_text:
        manager_link = None

        for link in infobox_links:
            if link["text"].lower() in manager_text.lower() or manager_text.lower() in link["text"].lower():
                manager_link = link
                break

        if manager_link:
            manager_id = manager_link["text"]
            person_page = manager_link["page"]
            person_url = manager_link["url"]
        else:
            manager_id = manager_text
            person_page = ""
            person_url = ""

        rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "game_title": "Super Smash Bros.",
            "team_title": team_title,
            "team_url": make_page_url(team_title),
            "section_heading": "Team Information",
            "subheading": "Manager",
            "status": "Active",
            "id": manager_id,
            "name": "",
            "position": "Manager",
            "join_date": "",
            "inactive_date": "",
            "leave_date": "",
            "new_team": "",
            "nationality": "",
            "person_page": person_page,
            "person_url": person_url,
            "row_text": manager_text,
            "infobox_text": infobox_text,
        })

    # -----------------------------------------
    # 2. Player Roster tables 추출
    # -----------------------------------------
    for table_idx, table in enumerate(soup.find_all("table")):
        status, position, game_title = table_context(table)

        if status is None:
            continue

        main_section = get_main_section(table)
        subheading = get_sub_section(table)

        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "tag" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []

            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": game_title,
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "status": status,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": "",
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
                "infobox_text": infobox_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "game_title",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [17]:
test_team = "CJ eSports"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_smash_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: CJ eSports
페이지 존재 여부: True
에러: None
추출 row 수: 8


,team_title,game_title,status,id,name,position,join_date,leave_date,new_team,nationality,person_url
0,CJ eSports,Super Smash Bros.,Active,h,,Manager,,,,,https://liquipedia.net/smash/Template:Infobox_...
1,CJ eSports,Super Smash Bros. Melee,Active,CaptainJack,,Player,2019-07-07,,,Japan,https://liquipedia.net/smash/Captain_Jack
2,CJ eSports,Super Smash Bros. Ultimate,Active,CaptainJack,,Player,2019-07-07,,,Japan,https://liquipedia.net/smash/Captain_Jack
3,CJ eSports,Super Smash Bros. Ultimate,Active,Abu,,Player,2019-07-07,,,Japan,https://liquipedia.net/smash/index.php?title=A...
4,CJ eSports,Super Smash Bros. Ultimate,Active,Kojiro,,Player,2019-08-11,,,Japan,https://liquipedia.net/smash/index.php?title=K...
5,CJ eSports,Super Smash Bros.,Former,aMSa,,Former Player,2019-07-07,2020-07-06,Golden Guardians,Japan,https://liquipedia.net/smash/AMSa
6,CJ eSports,Super Smash Bros.,Former,Sanne,,Former Player,2019-07-07,2020-07-06,,Japan,https://liquipedia.net/smash/Sanne
7,CJ eSports,Super Smash Bros.,Former,Hero,,Former Player,2021-01-23,2023-06-23,BUZZ e-sports,Japan,https://liquipedia.net/smash/Hero_(Japanese_pl...


In [18]:
def crawl_smash_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_team_candidate = (
            is_team_page
            and not is_player_page
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_team_candidate": is_team_candidate,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_team_candidate:
            try:
                people_df = extract_smash_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [19]:
smash_teams_checked_df, smash_teams_final_df, smash_people_raw_df, smash_errors_df = crawl_smash_teams_and_people_v4(
    candidate_df=smash_dedup_df2,
    output_prefix="smash_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(smash_teams_final_df.head(100))
display(smash_people_raw_df.head(100))

크롤링 대상 후보 수: 13
예상 소요 시간: 6.7 분
예상 소요 시간: 0.11 시간


smash_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/13 [00:00<?, ?it/s]

검증 완료
최종 팀 수: 5
추출 인원 row 수: 26
에러 수: 0
저장 완료
/content/smash_korean_mainteam_refined_v4_teams_checked.csv
/content/smash_korean_mainteam_refined_v4_teams_final.csv
/content/smash_korean_mainteam_refined_v4_people_raw.csv
/content/smash_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_team_candidate,infobox_text,error,team_type
0,smash,Super Smash Bros.,Team Liquid,Team Liquid,https://liquipedia.net/smash/Team_Liquid,True,True,[ e ][ h ] Team Liquid Team Information Locati...,None,main_team_refined
1,smash,Super Smash Bros.,T1,T1,https://liquipedia.net/smash/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
2,smash,Super Smash Bros.,Gen.G Esports,Gen.G Esports,https://liquipedia.net/smash/Gen.G_Esports,True,True,[ e ][ h ] Gen.G Esports Team Information Loca...,None,main_team_refined
3,smash,Super Smash Bros.,BUZZ e-sports,BUZZ e-sports,https://liquipedia.net/smash/BUZZ_e-sports,True,True,[ e ][ h ] BUZZ e-sports Team Information Loca...,None,main_team_refined
4,smash,Super Smash Bros.,DetonatioN FocusMe,DetonatioN FocusMe,https://liquipedia.net/smash/DetonatioN_FocusMe,True,True,[ e ][ h ] DetonatioN FocusMe Team Information...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,status,id,name,...,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,infobox_text,team_display_title,team_type
0,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Team Information,Manager,Active,h,,...,,,,,Template:Infobox team,https://liquipedia.net/smash/Template:Infobox_...,"Ryan "" L4st "" Krichbaum Links History",[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
1,smash,Super Smash Bros.,Super Smash Bros. Melee,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Melee Players,Active,Hungrybox,,...,,,,United States,Hungrybox,https://liquipedia.net/smash/Hungrybox,Hungrybox 2015-01-06,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
2,smash,Super Smash Bros.,Super Smash Bros. Melee,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Melee Players,Active,Crunch,,...,,,,United States,Crunch,https://liquipedia.net/smash/Crunch,Crunch 2016-07-09,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
3,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,KoreanDJ,,...,,2015-09-28,,United States,KoreanDJ,https://liquipedia.net/smash/KoreanDJ,KoreanDJ 2014-03-18 2015-09-28,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
4,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,Nairo,,...,,2016-08-16,,United States,Nairo,https://liquipedia.net/smash/Nairo,Nairo 2015-08-12 2016-08-16,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
5,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,Salem,,...,,2019-02-27,,United States,Salem,https://liquipedia.net/smash/Salem,Salem 2018-02-09 2019-02-27,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
6,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,Atelier,,...,,2023-07-15,NORTHEPTION,Japan,Atelier,https://liquipedia.net/smash/Atelier,Atelier 2021-09-01 2023-07-15 NORTHEPTION,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
7,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,Dabuz,,...,,2025-04-15,,United States,Dabuz,https://liquipedia.net/smash/Dabuz,Dabuz 2019-03-07 2025-04-15,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
8,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,Riddles,,...,,2025-04-15,,Canada,Riddles,https://liquipedia.net/smash/Riddles,Riddles 2022-05-25 2025-04-15,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined
9,smash,Super Smash Bros.,Super Smash Bros.,Team Liquid,https://liquipedia.net/smash/Team_Liquid,Player Roster,Former,Former,Ken,,...,,2026-01-10,,United States,Ken (Melee player),https://liquipedia.net/smash/Ken_(Melee_player),Ken 2014-03-18 2026-01-10,[ e ][ h ] Team Liquid Team Information Locati...,Team Liquid,main_team_refined


In [20]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [21]:
smash_people_unique_df = make_unique_people_latest_from_raw(smash_people_raw_df)

print("raw row 수:", len(smash_people_raw_df))
print("최신 기준 unique 인원 수:", len(smash_people_unique_df))

display(smash_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 26
최신 기준 unique 인원 수: 25


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Super Smash Bros.,T1,Former,ANTi,,Former Player,2019-04-17,,2020-07-02,,United States,https://liquipedia.net/smash/ANTi
1,Super Smash Bros.,Team Liquid,Former,Atelier,,Former Player,2021-09-01,,2023-07-15,NORTHEPTION,Japan,https://liquipedia.net/smash/Atelier
2,Super Smash Bros.,Team Liquid,Former,Chillindude,,Former Player,2015-01-06,,2026-01-10,,United States,https://liquipedia.net/smash/Chillindude
3,Super Smash Bros.,Team Liquid,Former,ChuDat,,Former Player,2017-06-14,,2026-01-10,,United States,https://liquipedia.net/smash/ChuDat
4,Super Smash Bros. Melee,Team Liquid,Active,Crunch,,Player,2016-07-09,,,,United States,https://liquipedia.net/smash/Crunch
5,Super Smash Bros.,Team Liquid,Former,Dabuz,,Former Player,2019-03-07,,2025-04-15,,United States,https://liquipedia.net/smash/Dabuz
6,Super Smash Bros.,BUZZ e-sports,Active,Hero,,Player,2023-06-23,,,,Japan,https://liquipedia.net/smash/Hero_(Japanese_pl...
7,Super Smash Bros. Melee,Team Liquid,Active,Hungrybox,,Player,2015-01-06,,,,United States,https://liquipedia.net/smash/Hungrybox
8,Super Smash Bros.,DetonatioN FocusMe,Former,Kameme,,Former Player,2016-09-30,,2018-10-04,RayRoad Gaming,Japan,https://liquipedia.net/smash/Kameme
9,Super Smash Bros.,Team Liquid,Former,Ken,,Former Player,2014-03-18,,2026-01-10,,United States,https://liquipedia.net/smash/Ken_(Melee_player)


In [22]:
smash_dedup_df2.to_csv(
    "/content/smash_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

smash_teams_checked_df.to_csv(
    "/content/smash_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

smash_teams_final_df.to_csv(
    "/content/smash_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

smash_people_raw_df.to_csv(
    "/content/smash_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

smash_people_unique_df.to_csv(
    "/content/smash_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

smash_errors_df.to_csv(
    "/content/smash_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/smash_teams_final_v4.csv")
print("/content/smash_people_unique_v4.csv")

저장 완료
/content/smash_teams_final_v4.csv
/content/smash_people_unique_v4.csv


#크로스파이어

In [23]:
# =========================================
# CrossFire Liquipedia 설정
# =========================================

GAME_SLUG = "crossfire"
GAME_LABEL = "CrossFire"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanCrossFireEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/crossfire
https://liquipedia.net/crossfire/api.php


In [24]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    CrossFire Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [25]:
CROSSFIRE_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "CrossFire Korea",
    "Cross Fire Korea",

    # 예시/한국 팀 후보
    "Hidden",
    "Hidden Korea",
    "Team Korea",
    "Korea CrossFire",
    "Korean CrossFire",
    "kEs",
    "mAestro",
    "Doolly",
    "FeArless",
    "Clara",
]

In [26]:
search_rows = []

for keyword in CROSSFIRE_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

crossfire_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(crossfire_search_df))
display(crossfire_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: Korean / offset=0
검색: Korean / offset=50
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: Korean team / offset=0
검색: Korean team / offset=50
검색: Korea esports / offset=0
검색: Korea esports / offset=50
검색: CrossFire Korea / offset=0
검색: CrossFire Korea / offset=50
검색: Cross Fire Korea / offset=0
검색: Cross Fire Korea / offset=50
검색: Hidden / offset=0
검색: Hidden / offset=50
검색: Hidden Korea / offset=0
검색: Hidden Korea / offset=50
검색: Team Korea / offset=0
검색: Team Korea / offset=50
검색: Korea CrossFire / offset=0
검색: Korea CrossFire / offset=50
검색: Korean CrossFire / offset=0
검색: Korean CrossFire / offset=50
검색: kEs / offset=0
검색: kEs / offset=50
검색: mAestro / offset=0
검색: mAestro / offset=50
검색: Doolly / offset=0
검색: Doolly / offset=50
검색: FeArless / offset=0
검색: FeArless / offset=50
검색: Clara / offset=0
검색: Clara / offset=50
검색 후보 수: 83


,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,crossfire,CrossFire,S-Tier Tournaments,Gamers CrossFire Stars Invitational 2024 May 7...,3974,https://liquipedia.net/crossfire/S-Tier_Tourna...,korea
1,crossfire,CrossFire,Hidden,Hidden Team Information Location: South Korea ...,6105,https://liquipedia.net/crossfire/Hidden,korea
2,crossfire,CrossFire,CrossFire Stars/Invitational/2024,Organizer: Smilegate Game version: CrossFire T...,6248,https://liquipedia.net/crossfire/CrossFire_Sta...,korea
3,crossfire,CrossFire,DRX,DRX Team Information Location: Vietnam South K...,7390,https://liquipedia.net/crossfire/DRX,korea
4,crossfire,CrossFire,Qualifier Tournaments,Japan 5 Vault ast* e-Stars Seoul 2010 Asia Cha...,2371,https://liquipedia.net/crossfire/Qualifier_Tou...,korea
...,...,...,...,...,...,...,...
78,crossfire,CrossFire,Qing Jiu Club/Played Matches,"June 2, 2024 - 19:00 CST D-Tier Glory Cup 2024...",4673,https://liquipedia.net/crossfire/Qing_Jiu_Club...,kEs
79,crossfire,CrossFire,MAestro/Results,,6024,https://liquipedia.net/crossfire/MAestro/Results,mAestro
80,crossfire,CrossFire,Doolly/Results,,989,https://liquipedia.net/crossfire/Doolly/Results,Doolly
81,crossfire,CrossFire,FeArless/Results,,1069,https://liquipedia.net/crossfire/FeArless/Results,FeArless


In [27]:
def build_crossfire_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "hidden",
        "korea",
        "korean",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "matches",
        "rankings",
        "awards",
        "finals",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("CrossFire 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [28]:
crossfire_team_candidate_loose_df = build_crossfire_team_candidate_loose_df(crossfire_search_df)

display(crossfire_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

CrossFire 완화 후보 수: 5
예상 parse 소요 시간: 2.6 분
예상 parse 소요 시간: 0.04 시간


,title,snippet,search_keyword,url
0,Hidden,Hidden Team Information Location: South Korea ...,korea,https://liquipedia.net/crossfire/Hidden
1,DRX,DRX Team Information Location: Vietnam South K...,korea,https://liquipedia.net/crossfire/DRX
2,MAze,"South Korea Born: March 5, 1990 (age 35) Statu...",korea,https://liquipedia.net/crossfire/MAze
3,All Gamers,&quot;秀B1租借加盟AG战队 全新阵容出征韩国CFS &quot; [Xiu B1 j...,korea,https://liquipedia.net/crossfire/All_Gamers
4,BaiSha Gaming,&quot;秀B1租借加盟AG战队 全新阵容出征韩国CFS &quot; [Xiu B1 j...,korea,https://liquipedia.net/crossfire/BaiSha_Gaming


In [29]:
# =========================================
# CrossFire 완화 후보에서 팀 후보만 재정제
# output: crossfire_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = crossfire_team_candidate_loose_df.copy()

crossfire_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "hidden",
]

crossfire_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
]

crossfire_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

crossfire_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_crossfire_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in crossfire_bad_page_markers):
        return False

    for pattern in crossfire_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in crossfire_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in crossfire_team_name_keywords):
        return True

    return False


crossfire_dedup_df2 = source_df[
    source_df.apply(is_refined_crossfire_team_candidate, axis=1)
].copy().reset_index(drop=True)

crossfire_dedup_df2["title_clean"] = (
    crossfire_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

crossfire_dedup_df2 = crossfire_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("CrossFire 완화 후보 수:", len(crossfire_team_candidate_loose_df))
print("재정제 후 crossfire_dedup_df2 후보 수:", len(crossfire_dedup_df2))
print("예상 parse 소요 시간:", round(len(crossfire_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(crossfire_dedup_df2) * 31 / 3600, 2), "시간")

display(crossfire_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

CrossFire 완화 후보 수: 5
재정제 후 crossfire_dedup_df2 후보 수: 3
예상 parse 소요 시간: 1.6 분
예상 parse 소요 시간: 0.03 시간


,title,snippet,search_keyword,url
0,Hidden,Hidden Team Information Location: South Korea ...,korea,https://liquipedia.net/crossfire/Hidden
1,DRX,DRX Team Information Location: Vietnam South K...,korea,https://liquipedia.net/crossfire/DRX
2,BaiSha Gaming,&quot;秀B1租借加盟AG战队 全新阵容出征韩国CFS &quot; [Xiu B1 j...,korea,https://liquipedia.net/crossfire/BaiSha_Gaming


In [30]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [31]:
def extract_crossfire_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "crossfire", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Gallery",
        "Data",
        "References",
        "Awards",
        "Statistics",
        "Placement Summary",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|cfs|cfpl|cfel)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former players" in sub_low or "players" in sub_low:
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        if table_type == "organization":
            pos = row_data.get("position", "")
            if pos:
                return pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if table_type == "organization":
            if "position" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "CrossFire",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [32]:
test_team = "Hidden"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_crossfire_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: Hidden
페이지 존재 여부: True
에러: None
추출 row 수: 14


,team_title,game_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Hidden,CrossFire,Former,Dominic,Cha Dong-min,Former Player,2013-06-01 [ 2 ] [ 3 ],,2013-11-21 [ 4 ],Hidden,South Korea,https://liquipedia.net/crossfire/Dominic
1,Hidden,CrossFire,Former,Dolly,,Former Player,2012-11-26 [ 1 ],,2013-06-01 [ 2 ] [ 3 ],,South Korea,https://liquipedia.net/crossfire/index.php?tit...
2,Hidden,CrossFire,Former,Kyo,Yang Mu-gil,Former Player,2012-11-26 [ 1 ],,2013-06-01 [ 2 ] [ 3 ],,South Korea,https://liquipedia.net/crossfire/index.php?tit...
3,Hidden,CrossFire,Former,MazeOrrsia,Yang Mu-gil,Former Player,2013-06-01 [ 2 ] [ 3 ],,2014-11-25 [ 5 ],,South Korea,https://liquipedia.net/crossfire/MazeOrrsia
4,Hidden,CrossFire,Former,troller,,Former Player,2013-11-21 [ 4 ],,2014-11-25 [ 6 ],,South Korea,https://liquipedia.net/crossfire/index.php?tit...
5,Hidden,CrossFire,Former,mAze,Kimg Jae-ho,Former Player,2014-12-05 [ 7 ] [ 8 ],,2015-12-04 [ 11 ] [ 12 ],,South Korea,https://liquipedia.net/crossfire/MAze
6,Hidden,CrossFire,Former,Clara,Jang Hyun-jun,Former Player,2012-11-26 [ 1 ],,2015-11-20 [ 9 ] [ 10 ],Hidden,South Korea,https://liquipedia.net/crossfire/Clara
7,Hidden,CrossFire,Former,Dominic,Cha Dong-min,Former Player,2014-11-25 [ 5 ],,2015-11-20 [ 9 ] [ 10 ],,South Korea,https://liquipedia.net/crossfire/Dominic
8,Hidden,CrossFire,Former,Below,Heo Jae-yoon,Former Player,2012-11-26 [ 1 ],,2016-12-02 [ 13 ] [ 14 ],,South Korea,https://liquipedia.net/crossfire/Below
9,Hidden,CrossFire,Former,Clara,Jang Hyun-jun,Former Player,2016-12-02 [ 13 ] [ 14 ],,2020-03-03 [ 15 ],,South Korea,https://liquipedia.net/crossfire/Clara


In [33]:
def crawl_crossfire_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_crossfire_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [34]:
crossfire_teams_checked_df, crossfire_teams_final_df, crossfire_people_raw_df, crossfire_errors_df = crawl_crossfire_teams_and_people_v4(
    candidate_df=crossfire_dedup_df2,
    output_prefix="crossfire_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(crossfire_teams_final_df.head(100))
display(crossfire_people_raw_df.head(100))

크롤링 대상 후보 수: 3
예상 소요 시간: 1.6 분
예상 소요 시간: 0.03 시간


crossfire_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/3 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 2
추출 인원 row 수: 21
에러 수: 0
저장 완료
/content/crossfire_korean_mainteam_refined_v4_teams_checked.csv
/content/crossfire_korean_mainteam_refined_v4_teams_final.csv
/content/crossfire_korean_mainteam_refined_v4_people_raw.csv
/content/crossfire_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,crossfire,CrossFire,Hidden,Hidden,https://liquipedia.net/crossfire/Hidden,True,True,[ e ][ h ] Hidden Team Information Location: S...,None,main_team_refined
1,crossfire,CrossFire,DRX,DRX,https://liquipedia.net/crossfire/DRX,True,True,[ e ][ h ] DRX Team Information Location: Viet...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,1,...,2013-06-01 [ 2 ] [ 3 ],,2013-11-21 [ 4 ],Hidden,South Korea,Dominic,https://liquipedia.net/crossfire/Dominic,Dominic Cha Dong-min 2013-06-01 [ 2 ] [ 3 ] 20...,Hidden,main_team_refined
1,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,1,...,2012-11-26 [ 1 ],,2013-06-01 [ 2 ] [ 3 ],,South Korea,index.php?title=Dolly&action=edit&redlink=1,https://liquipedia.net/crossfire/index.php?tit...,Dolly 2012-11-26 [ 1 ] 2013-06-01 [ 2 ] [ 3 ],Hidden,main_team_refined
2,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,1,...,2012-11-26 [ 1 ],,2013-06-01 [ 2 ] [ 3 ],,South Korea,index.php?title=Kyo&action=edit&redlink=1,https://liquipedia.net/crossfire/index.php?tit...,Kyo Yang Mu-gil 2012-11-26 [ 1 ] 2013-06-01 [ ...,Hidden,main_team_refined
3,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,2,...,2013-06-01 [ 2 ] [ 3 ],,2014-11-25 [ 5 ],,South Korea,MazeOrrsia,https://liquipedia.net/crossfire/MazeOrrsia,MazeOrrsia Yang Mu-gil 2013-06-01 [ 2 ] [ 3 ] ...,Hidden,main_team_refined
4,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,2,...,2013-11-21 [ 4 ],,2014-11-25 [ 6 ],,South Korea,index.php?title=Troller&action=edit&redlink=1,https://liquipedia.net/crossfire/index.php?tit...,troller 2013-11-21 [ 4 ] 2014-11-25 [ 6 ],Hidden,main_team_refined
5,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,3,...,2014-12-05 [ 7 ] [ 8 ],,2015-12-04 [ 11 ] [ 12 ],,South Korea,MAze,https://liquipedia.net/crossfire/MAze,mAze Kimg Jae-ho 2014-12-05 [ 7 ] [ 8 ] 2015-1...,Hidden,main_team_refined
6,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,3,...,2012-11-26 [ 1 ],,2015-11-20 [ 9 ] [ 10 ],Hidden,South Korea,Clara,https://liquipedia.net/crossfire/Clara,Clara Jang Hyun-jun 2012-11-26 [ 1 ] 2015-11-2...,Hidden,main_team_refined
7,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,3,...,2014-11-25 [ 5 ],,2015-11-20 [ 9 ] [ 10 ],,South Korea,Dominic,https://liquipedia.net/crossfire/Dominic,Dominic Cha Dong-min 2014-11-25 [ 5 ] 2015-11-...,Hidden,main_team_refined
8,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,4,...,2012-11-26 [ 1 ],,2016-12-02 [ 13 ] [ 14 ],,South Korea,Below,https://liquipedia.net/crossfire/Below,Below Heo Jae-yoon 2012-11-26 [ 1 ] 2016-12-02...,Hidden,main_team_refined
9,crossfire,CrossFire,CrossFire,Hidden,https://liquipedia.net/crossfire/Hidden,Player Roster,Former,player_roster,Former,5,...,2016-12-02 [ 13 ] [ 14 ],,2020-03-03 [ 15 ],,South Korea,Clara,https://liquipedia.net/crossfire/Clara,Clara Jang Hyun-jun 2016-12-02 [ 13 ] [ 14 ] 2...,Hidden,main_team_refined


In [35]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [36]:
crossfire_people_unique_df = make_unique_people_latest_from_raw(crossfire_people_raw_df)

print("raw row 수:", len(crossfire_people_raw_df))
print("최신 기준 unique 인원 수:", len(crossfire_people_unique_df))

display(crossfire_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 21
최신 기준 unique 인원 수: 19


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,CrossFire,Hidden,Former,Below,Heo Jae-yoon,Former Player,2012-11-26 [ 1 ],,2016-12-02 [ 13 ] [ 14 ],,South Korea,https://liquipedia.net/crossfire/Below
1,CrossFire,Hidden,Former,Clara,Jang Hyun-jun,Former Player,2016-12-02 [ 13 ] [ 14 ],,2020-03-03 [ 15 ],,South Korea,https://liquipedia.net/crossfire/Clara
2,CrossFire,Hidden,Former,Dominic,Cha Dong-min,Former Player,2014-11-25 [ 5 ],,2015-11-20 [ 9 ] [ 10 ],,South Korea,https://liquipedia.net/crossfire/Dominic
3,CrossFire,Hidden,Former,Doolly,Jang Hyun-woo,Former Player,2015-11-20 [ 9 ] [ 10 ],,2020-03-03 [ 15 ],,South Korea,https://liquipedia.net/crossfire/Doolly
4,CrossFire,Hidden,Former,FeArless,Shin Dong-wook,Former Player,2015-12-04 [ 11 ] [ 12 ],,2020-03-03 [ 15 ],,South Korea,https://liquipedia.net/crossfire/FeArless
5,CrossFire,Hidden,Former,kEs,Park Jae-young,Former Player,2012-11-26 [ 1 ],,2020-03-03 [ 15 ],,South Korea,https://liquipedia.net/crossfire/KEs
6,CrossFire,DRX,Active,KZ,Phạm Quốc Khánh,Player,2025-06-10 [ 2 ],,,,Vietnam,https://liquipedia.net/crossfire/KZ
7,CrossFire,Hidden,Former,mAestro,Lee Jin-sung,Former Player,2014-11-25 [ 6 ],,2020-03-03 [ 15 ],,South Korea,https://liquipedia.net/crossfire/MAestro
8,CrossFire,Hidden,Former,mAze,Kimg Jae-ho,Former Player,2014-12-05 [ 7 ] [ 8 ],,2015-12-04 [ 11 ] [ 12 ],,South Korea,https://liquipedia.net/crossfire/MAze
9,CrossFire,DRX,Active,MEOU,Nguyễn Trần Xuân Huy,Player,2025-06-10 [ 2 ],,,,Vietnam,https://liquipedia.net/crossfire/MEOU


In [37]:
crossfire_dedup_df2.to_csv(
    "/content/crossfire_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

crossfire_teams_checked_df.to_csv(
    "/content/crossfire_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

crossfire_teams_final_df.to_csv(
    "/content/crossfire_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

crossfire_people_raw_df.to_csv(
    "/content/crossfire_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

crossfire_people_unique_df.to_csv(
    "/content/crossfire_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

crossfire_errors_df.to_csv(
    "/content/crossfire_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/crossfire_teams_final_v4.csv")
print("/content/crossfire_people_unique_v4.csv")

저장 완료
/content/crossfire_teams_final_v4.csv
/content/crossfire_people_unique_v4.csv


#하스스톤

In [38]:
# =========================================
# Hearthstone Liquipedia 설정
# =========================================

GAME_SLUG = "hearthstone"
GAME_LABEL = "Hearthstone"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanHearthstoneEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/hearthstone
https://liquipedia.net/hearthstone/api.php


In [39]:
def clean_text(text):
    if text is None:
        return ""
    text = re.sub(r"\s+", " ", str(text))
    return text.strip()


def make_page_url(title):
    title = title or ""
    return f"{BASE_URL}/{quote(title.replace(' ', '_'))}"


def api_get(params, sleep=REQUEST_SLEEP):
    params = {
        **params,
        "format": "json"
    }

    r = requests.get(
        API_URL,
        params=params,
        headers=HEADERS,
        timeout=40
    )

    time.sleep(sleep)

    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")

    data = r.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data

In [40]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Hearthstone Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [41]:
HEARTHSTONE_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "Hearthstone Korea",
    "Hearthstone Korean",

    # 한국/관련 팀 후보
    "T1",
    "SK Telecom T1",
    "Gen.G",
    "Gen.G Esports",
    "Team BlossoM",
    "ROX",
    "Kranich",
    "Surrender",
    "JinBae",
    "DawN",
    "Flurry",
]

In [42]:
search_rows = []

for keyword in HEARTHSTONE_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

hearthstone_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(hearthstone_search_df))
display(hearthstone_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: Korean / offset=150
검색: Korean / offset=200
검색: Korean / offset=250
검색: Korean / offset=300
검색: Korean / offset=350
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: South Korean / offset=100
검색: South Korean / offset=150
검색: South Korean / offset=200
검색: Korean team / offset=0
검색: Korean team / offset=50
검색: Korean team / offset=100
검색: Korean team / offset=150
검색: Korean team / offset=200
검색: Korea esports / offset=0
검색: Korea esports / offset=50
검색: Korea esports / offset=100
검색: Hearthstone Korea / offset=0
검색: Hearthstone Korea / offset=50
검색: Hearthstone Korea / offset=100
검색: Hea

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,hearthstone,Hearthstone,South Korea,Team South Korea Team Information Location: So...,18657,https://liquipedia.net/hearthstone/South_Korea,korea
1,hearthstone,Hearthstone,T1,Location: South Korea Region: Korea Approx. To...,23664,https://liquipedia.net/hearthstone/T1,korea
2,hearthstone,Hearthstone,Surrender,Player Information Name: 김정수 Romanized Name: K...,3939,https://liquipedia.net/hearthstone/Surrender,korea
3,hearthstone,Hearthstone,Kranich,Player Information Name: 백학준 Romanized Name: B...,4341,https://liquipedia.net/hearthstone/Kranich,korea
4,hearthstone,Hearthstone,Sooni,Player Information Name: 남상수 Romanized Name: N...,22804,https://liquipedia.net/hearthstone/Sooni,korea
...,...,...,...,...,...,...,...
95,hearthstone,Hearthstone,Mintcandy,Se-min Nationality: South Korea Approx. Total ...,35306,https://liquipedia.net/hearthstone/Mintcandy,korea
96,hearthstone,Hearthstone,Zanyang,Player Information Nationality: South Korea Ap...,31305,https://liquipedia.net/hearthstone/Zanyang,korea
97,hearthstone,Hearthstone,Hearthstone BattleRoyal - Korea vs Japan/Japan...,Astan 16 players. Single Elimination Bracket f...,12934,https://liquipedia.net/hearthstone/Hearthstone...,korea
98,hearthstone,Hearthstone,Hearthstone Premier League,Premier League League Information Server: kr T...,10781,https://liquipedia.net/hearthstone/Hearthstone...,korea


In [43]:
def build_hearthstone_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "t1",
        "sk telecom",
        "gen.g",
        "geng",
        "blossom",
        "rox",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "player results",
        "team results",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "matches",
        "rankings",
        "awards",
        "finals",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Hearthstone 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [44]:
hearthstone_team_candidate_loose_df = build_hearthstone_team_candidate_loose_df(hearthstone_search_df)

display(hearthstone_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Hearthstone 완화 후보 수: 58
예상 parse 소요 시간: 30.0 분
예상 parse 소요 시간: 0.5 시간


,title,snippet,search_keyword,url
0,South Korea,Team South Korea Team Information Location: So...,korea,https://liquipedia.net/hearthstone/South_Korea
1,T1,Location: South Korea Region: Korea Approx. To...,korea,https://liquipedia.net/hearthstone/T1
2,Seulsiho,Information Name: 정한슬 Romanized Name: Jung Han...,korea,https://liquipedia.net/hearthstone/Seulsiho
3,ShangHigh,Information Name: 김한결 Romanized Name: Kim Han-...,korea,https://liquipedia.net/hearthstone/ShangHigh
4,DawN,Information Name: 장현재 Romanized Name: Jang Hyu...,korea,https://liquipedia.net/hearthstone/DawN
5,Ryvius,Dasol Nationality: South Korea Approx. Total W...,korea,https://liquipedia.net/hearthstone/Ryvius
6,Hi3,Information Name: 조원경 Romanized Name: Cho Won-...,korea,https://liquipedia.net/hearthstone/Hi3
7,Grr,Information Name: 박기정 Romanized Name: Park Ki-...,korea,https://liquipedia.net/hearthstone/Grr
8,Steelo,Information Name: 조강현 Romanized Name: Cho Gang...,korea,https://liquipedia.net/hearthstone/Steelo
9,RenieHouR,Information Name: 이정환 Romanized Name: Lee Jung...,korea,https://liquipedia.net/hearthstone/RenieHouR


In [45]:
# =========================================
# Hearthstone 완화 후보에서 팀 후보만 재정제
# output: hearthstone_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = hearthstone_team_candidate_loose_df.copy()

hearthstone_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "t1",
    "sk telecom",
    "gen.g",
    "geng",
    "blossom",
    "rox",
]

hearthstone_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "player results",
    "team results",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "matches",
    "rankings",
    "awards",
    "finals",
]

hearthstone_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

hearthstone_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_hearthstone_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in hearthstone_bad_page_markers):
        return False

    for pattern in hearthstone_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in hearthstone_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in hearthstone_team_name_keywords):
        return True

    return False


hearthstone_dedup_df2 = source_df[
    source_df.apply(is_refined_hearthstone_team_candidate, axis=1)
].copy().reset_index(drop=True)

hearthstone_dedup_df2["title_clean"] = (
    hearthstone_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

hearthstone_dedup_df2 = hearthstone_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Hearthstone 완화 후보 수:", len(hearthstone_team_candidate_loose_df))
print("재정제 후 hearthstone_dedup_df2 후보 수:", len(hearthstone_dedup_df2))
print("예상 parse 소요 시간:", round(len(hearthstone_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(hearthstone_dedup_df2) * 31 / 3600, 2), "시간")

display(hearthstone_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Hearthstone 완화 후보 수: 58
재정제 후 hearthstone_dedup_df2 후보 수: 58
예상 parse 소요 시간: 30.0 분
예상 parse 소요 시간: 0.5 시간


,title,snippet,search_keyword,url
0,South Korea,Team South Korea Team Information Location: So...,korea,https://liquipedia.net/hearthstone/South_Korea
1,T1,Location: South Korea Region: Korea Approx. To...,korea,https://liquipedia.net/hearthstone/T1
2,Seulsiho,Information Name: 정한슬 Romanized Name: Jung Han...,korea,https://liquipedia.net/hearthstone/Seulsiho
3,ShangHigh,Information Name: 김한결 Romanized Name: Kim Han-...,korea,https://liquipedia.net/hearthstone/ShangHigh
4,DawN,Information Name: 장현재 Romanized Name: Jang Hyu...,korea,https://liquipedia.net/hearthstone/DawN
5,Ryvius,Dasol Nationality: South Korea Approx. Total W...,korea,https://liquipedia.net/hearthstone/Ryvius
6,Hi3,Information Name: 조원경 Romanized Name: Cho Won-...,korea,https://liquipedia.net/hearthstone/Hi3
7,Grr,Information Name: 박기정 Romanized Name: Park Ki-...,korea,https://liquipedia.net/hearthstone/Grr
8,Steelo,Information Name: 조강현 Romanized Name: Cho Gang...,korea,https://liquipedia.net/hearthstone/Steelo
9,RenieHouR,Information Name: 이정환 Romanized Name: Lee Jung...,korea,https://liquipedia.net/hearthstone/RenieHouR


In [46]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [47]:
def extract_hearthstone_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "hearthstone", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "References",
        "Awards",
        "Statistics",
        "Player Achievements",
        "Team Achievements",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|playoffs|championship|cup|tournament|league|series|qualifier|finals|masters|grandmasters)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former players" in sub_low or "players" in sub_low:
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        if table_type == "organization":
            pos = row_data.get("position", "")
            if pos:
                return pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if table_type == "organization":
            if "position" not in normalized_headers and len(normalized_headers) >= 4:
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            for idx, cell in enumerate(cells):
                if idx >= len(normalized_headers):
                    continue

                col = normalized_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "Hearthstone",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [48]:
test_team = "T1"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_hearthstone_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: T1
페이지 존재 여부: True
에러: None
추출 row 수: 8


,team_title,game_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,T1,Hearthstone,Former,Hoej,Frederik Høj Nielsen,Former Player,2018-07-30 [ 1 ],,2019-05-09 [ 4 ],,Denmark,https://liquipedia.net/hearthstone/Hoej
1,T1,Hearthstone,Former,Xixo,Sebastian Bentert,Former Player,2018-07-30 [ 1 ],,2019-04-17 [ 2 ],,Germany,https://liquipedia.net/hearthstone/Xixo
2,T1,Hearthstone,Former,BoarControl,George Webb,Former Player,2019-04-19 [ 3 ],,2020-10-12 [ 5 ],Retired,United Kingdom,https://liquipedia.net/hearthstone/BoarControl
3,T1,Hearthstone,Former,Orange,Jon Westberg,Former Player,2019-04-19 [ 3 ],,2021-10-13 [ 8 ],,Sweden,https://liquipedia.net/hearthstone/Orange
4,T1,Hearthstone,Former,Fenomeno,Christos Tsakopoulos,Former Player,2019-04-19 [ 3 ],,2022-04-23 [ 10 ],,Greece,https://liquipedia.net/hearthstone/Fenomeno
5,T1,Hearthstone,Former,JinBae,Kim Jong-soo,Former Player,2021-09-01 [ 7 ],,2022-03-31 [ 9 ],,South Korea,https://liquipedia.net/hearthstone/JinBae
6,T1,Hearthstone,Former,Kranich,Baek Hak-jun,Former Player,2021-04-01 [ 6 ],,2022-03-31 [ 9 ],,South Korea,https://liquipedia.net/hearthstone/Kranich
7,T1,Hearthstone,Former,Surrender,Kim Jung-soo,Former Player,2018-07-30 [ 1 ],,2022-03-31 [ 9 ],,South Korea,https://liquipedia.net/hearthstone/Surrender


In [49]:
def crawl_hearthstone_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_hearthstone_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [50]:
hearthstone_teams_checked_df, hearthstone_teams_final_df, hearthstone_people_raw_df, hearthstone_errors_df = crawl_hearthstone_teams_and_people_v4(
    candidate_df=hearthstone_dedup_df2,
    output_prefix="hearthstone_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(hearthstone_teams_final_df.head(100))
display(hearthstone_people_raw_df.head(100))

크롤링 대상 후보 수: 58
예상 소요 시간: 30.0 분
예상 소요 시간: 0.5 시간


hearthstone_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/58 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 5
추출 인원 row 수: 36
에러 수: 0
저장 완료
/content/hearthstone_korean_mainteam_refined_v4_teams_checked.csv
/content/hearthstone_korean_mainteam_refined_v4_teams_final.csv
/content/hearthstone_korean_mainteam_refined_v4_people_raw.csv
/content/hearthstone_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,hearthstone,Hearthstone,South Korea,South Korea,https://liquipedia.net/hearthstone/South_Korea,True,True,[ e ][ h ] Team South Korea Team Information L...,None,main_team_refined
1,hearthstone,Hearthstone,T1,T1,https://liquipedia.net/hearthstone/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
2,hearthstone,Hearthstone,All-Killers,All-Killers,https://liquipedia.net/hearthstone/All-Killers,True,True,[ e ][ h ] All-Killers Team Information Locati...,None,main_team_refined
3,hearthstone,Hearthstone,Golden Coin,Golden Coin,https://liquipedia.net/hearthstone/Golden_Coin,True,True,[ e ][ h ] Golden Coin Team Information Locati...,None,main_team_refined
4,hearthstone,Hearthstone,Team Magicamy,Team Magicamy,https://liquipedia.net/hearthstone/Team_Magicamy,True,True,[ e ][ h ] Team Magicamy Team Information Loca...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,0,...,2018-07-30 [ 1 ],,2019-05-09 [ 4 ],,Denmark,Hoej,https://liquipedia.net/hearthstone/Hoej,Hoej Frederik Høj Nielsen 2018-07-30 [ 1 ] 201...,T1,main_team_refined
1,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,0,...,2018-07-30 [ 1 ],,2019-04-17 [ 2 ],,Germany,Xixo,https://liquipedia.net/hearthstone/Xixo,Xixo Sebastian Bentert 2018-07-30 [ 1 ] 2019-0...,T1,main_team_refined
2,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,1,...,2019-04-19 [ 3 ],,2020-10-12 [ 5 ],Retired,United Kingdom,BoarControl,https://liquipedia.net/hearthstone/BoarControl,BoarControl George Webb 2019-04-19 [ 3 ] 2020-...,T1,main_team_refined
3,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,2,...,2019-04-19 [ 3 ],,2021-10-13 [ 8 ],,Sweden,Orange,https://liquipedia.net/hearthstone/Orange,Orange Jon Westberg 2019-04-19 [ 3 ] 2021-10-1...,T1,main_team_refined
4,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,3,...,2019-04-19 [ 3 ],,2022-04-23 [ 10 ],,Greece,Fenomeno,https://liquipedia.net/hearthstone/Fenomeno,Fenomeno Christos Tsakopoulos 2019-04-19 [ 3 ]...,T1,main_team_refined
5,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,3,...,2021-09-01 [ 7 ],,2022-03-31 [ 9 ],,South Korea,JinBae,https://liquipedia.net/hearthstone/JinBae,JinBae Kim Jong-soo 2021-09-01 [ 7 ] 2022-03-3...,T1,main_team_refined
6,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,3,...,2021-04-01 [ 6 ],,2022-03-31 [ 9 ],,South Korea,Kranich,https://liquipedia.net/hearthstone/Kranich,Kranich Baek Hak-jun 2021-04-01 [ 6 ] 2022-03-...,T1,main_team_refined
7,hearthstone,Hearthstone,Hearthstone,T1,https://liquipedia.net/hearthstone/T1,Player Roster,Former,player_roster,Former,3,...,2018-07-30 [ 1 ],,2022-03-31 [ 9 ],,South Korea,Surrender,https://liquipedia.net/hearthstone/Surrender,Surrender Kim Jung-soo 2018-07-30 [ 1 ] 2022-0...,T1,main_team_refined
8,hearthstone,Hearthstone,Hearthstone,All-Killers,https://liquipedia.net/hearthstone/All-Killers,Player Roster,Former,player_roster,Former,1,...,2015-05-02,,,,South Korea,Palmblad,https://liquipedia.net/hearthstone/Palmblad,Palmblad Kwak Woong-sub (곽웅섭) 2015-05-02,All-Killers,main_team_refined
9,hearthstone,Hearthstone,Hearthstone,All-Killers,https://liquipedia.net/hearthstone/All-Killers,Player Roster,Former,player_roster,Former,1,...,2015-05-02,,,,South Korea,LookSam,https://liquipedia.net/hearthstone/LookSam,LookSam Kim Jin-hyo 2015-05-02,All-Killers,main_team_refined


In [51]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [52]:
hearthstone_people_unique_df = make_unique_people_latest_from_raw(hearthstone_people_raw_df)

print("raw row 수:", len(hearthstone_people_raw_df))
print("최신 기준 unique 인원 수:", len(hearthstone_people_unique_df))

display(hearthstone_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 36
최신 기준 unique 인원 수: 32


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Hearthstone,T1,Former,BoarControl,George Webb,Former Player,2019-04-19 [ 3 ],,2020-10-12 [ 5 ],Retired,United Kingdom,https://liquipedia.net/hearthstone/BoarControl
1,Hearthstone,T1,Former,Fenomeno,Christos Tsakopoulos,Former Player,2019-04-19 [ 3 ],,2022-04-23 [ 10 ],,Greece,https://liquipedia.net/hearthstone/Fenomeno
2,Hearthstone,All-Killers,Former,flurry,Cho Hyun-soo,Former Player,2015-05-02,,,,South Korea,https://liquipedia.net/hearthstone/Flurry
3,Hearthstone,Golden Coin,Former,Ghost,Sugwang Park,Former Player,2014-08-15,,2016-04-10,,South Korea,https://liquipedia.net/hearthstone/Ghost
4,Hearthstone,T1,Former,Hoej,Frederik Høj Nielsen,Former Player,2018-07-30 [ 1 ],,2019-05-09 [ 4 ],,Denmark,https://liquipedia.net/hearthstone/Hoej
5,Hearthstone,T1,Former,JinBae,Kim Jong-soo,Former Player,2021-09-01 [ 7 ],,2022-03-31 [ 9 ],,South Korea,https://liquipedia.net/hearthstone/JinBae
6,Hearthstone,Golden Coin,Active,jjangnara,Donghyuk Kim,Player,,,,,South Korea,https://liquipedia.net/hearthstone/Jjangnara
7,Hearthstone,T1,Former,Kranich,Baek Hak-jun,Former Player,2021-04-01 [ 6 ],,2022-03-31 [ 9 ],,South Korea,https://liquipedia.net/hearthstone/Kranich
8,Hearthstone,All-Killers,Former,LookSam,Kim Jin-hyo,Former Player,2015-05-02,,,,South Korea,https://liquipedia.net/hearthstone/LookSam
9,Hearthstone,Team Magicamy,Former,MagicAmy,Hyerim Lee,Former Player,,,,Tempo Storm,South Korea,https://liquipedia.net/hearthstone/Magicamy


In [53]:
hearthstone_dedup_df2.to_csv(
    "/content/hearthstone_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

hearthstone_teams_checked_df.to_csv(
    "/content/hearthstone_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

hearthstone_teams_final_df.to_csv(
    "/content/hearthstone_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

hearthstone_people_raw_df.to_csv(
    "/content/hearthstone_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

hearthstone_people_unique_df.to_csv(
    "/content/hearthstone_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

hearthstone_errors_df.to_csv(
    "/content/hearthstone_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/hearthstone_teams_final_v4.csv")
print("/content/hearthstone_people_unique_v4.csv")

저장 완료
/content/hearthstone_teams_final_v4.csv
/content/hearthstone_people_unique_v4.csv


#콜오브듀티

In [54]:
# =========================================
# Call of Duty Liquipedia 설정
# =========================================

GAME_SLUG = "callofduty"
GAME_LABEL = "Call of Duty"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanCallOfDutyEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/callofduty
https://liquipedia.net/callofduty/api.php


In [55]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Call of Duty Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [56]:
CALLOFDUTY_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "Call of Duty Korea",
    "Call of Duty Korean",
    "CoD Korea",
    "CoD Korean",

    # 한국/아시아 관련 가능 후보
    "Team Korea",
    "Korea Call of Duty",
    "Korean Call of Duty",
    "AF Academy",
    "Afreeca",
    "T1",
    "Gen.G",
    "DRX",
]

In [57]:
search_rows = []

for keyword in CALLOFDUTY_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

cod_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(cod_search_df))
display(cod_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: Korean team / offset=0
검색: Korean team / offset=50
검색: Korea esports / offset=0
검색: Korea esports / offset=50
검색: Call of Duty Korea / offset=0
검색: Call of Duty Korea / offset=50
검색: Call of Duty Korean / offset=0
검색: Call of Duty Korean / offset=50
검색: CoD Korea / offset=0
검색: CoD Korea / offset=50
검색: CoD Korean / offset=0
검색: CoD Korean / offset=50
검색: Team Korea / offset=0
검색: Team Korea / offset=50
검색: Korea Call of Duty / offset=0
검색: Korea Call of Duty / offset=50
검색: Korean Call of Duty / offset=0
검색: Korean Call of Duty / offset=50
검색: AF Academy / offset=0
검색: AF Academy / offset=50
검색: Afreeca / offset=0
검색: T1 / offset=0
검색: T1 / offset=50
검색: T1 / offset=100
검색: T1 / offset=150
검색: T1 / offset=200
검색: T1 / offset=250
검색: T1 / 

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,callofduty,Call of Duty,Prodah,Prodahh Player Information Name: Brady Silowka...,18619,https://liquipedia.net/callofduty/Prodah,korea
1,callofduty,Call of Duty,F10W3R,Player Information Name: 유민우 Romanized Name: Y...,12868,https://liquipedia.net/callofduty/F10W3R,korea
2,callofduty,Call of Duty,Call of Duty Championship/2014/Korea,of Duty Championship 2014 - Korea Qualifier Le...,19100,https://liquipedia.net/callofduty/Call_of_Duty...,korea
3,callofduty,Call of Duty,Call of Duty Mobile World Championship/2020/So...,Call of Duty Mobile World Championship 2020 - ...,12404,https://liquipedia.net/callofduty/Call_of_Duty...,korea
4,callofduty,Call of Duty,Hippo,Hippo 河马 Player Information Name: Thomas Ik-Hy...,18190,https://liquipedia.net/callofduty/Hippo,korea
...,...,...,...,...,...,...,...
95,callofduty,Call of Duty,T1,T1 Player Information Name: Sander Heubacher N...,21269,https://liquipedia.net/callofduty/T1,T1
96,callofduty,Call of Duty,Call of Duty Championship/2015,"3:0(Bo5) G2 SSOF 1 3 x5.T1 March 27, 2015 - 15...",11250,https://liquipedia.net/callofduty/Call_of_Duty...,T1
97,callofduty,Call of Duty,Trident T1 Dotters,Trident T1 Dotters Team Information Location: ...,20580,https://liquipedia.net/callofduty/Trident_T1_D...,T1
98,callofduty,Call of Duty,Call of Duty Asia Pacific Championship/2015,eX Exile5.T1x5.T1 3 Integral NationiN 0 Februa...,20197,https://liquipedia.net/callofduty/Call_of_Duty...,T1


In [58]:
def build_cod_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "academy",
        "call of duty",
        "cod",
        "korea",
        "korean",
        "af academy",
        "afreeca",
        "t1",
        "gen.g",
        "drx",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "stage",
        "major",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "rankings",
        "awards",
        "finals",
        "playoffs",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Call of Duty 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [59]:
cod_team_candidate_loose_df = build_cod_team_candidate_loose_df(cod_search_df)

display(cod_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Call of Duty 완화 후보 수: 7
예상 parse 소요 시간: 3.6 분
예상 parse 소요 시간: 0.06 시간


,title,snippet,search_keyword,url
0,1K Gaming,Location: South Korea Region: Korea Approx. To...,korea,https://liquipedia.net/callofduty/1K_Gaming
1,Gee,Information Name: 고정현 Romanized Name: Go Joung...,korea,https://liquipedia.net/callofduty/Gee
2,Call of Duty: Mobile,"Garena for Southeast Asia on September 29, 201...",korea,https://liquipedia.net/callofduty/Call_of_Duty...
3,CanYou,Information Name: 김태성 Romanized Name: Kim Tae-...,korea,https://liquipedia.net/callofduty/CanYou
4,Ghosts,"Teams each from Brazil, Canada and Oceania qua...",korea,https://liquipedia.net/callofduty/Ghosts
5,Advanced Warfare,"Mitchell, who is deployed on their assignment ...",korea,https://liquipedia.net/callofduty/Advanced_War...
6,Holic (Korean player),South Korea Region: Asia Status: Active Approx...,korea,https://liquipedia.net/callofduty/Holic_%28Kor...


In [60]:
# =========================================
# Call of Duty 완화 후보에서 팀 후보만 재정제
# output: cod_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = cod_team_candidate_loose_df.copy()

cod_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "academy",
    "call of duty",
    "cod",
    "af academy",
    "afreeca",
    "t1",
    "gen.g",
    "drx",
]

cod_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "stage",
    "major",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "rankings",
    "awards",
    "finals",
    "playoffs",
]

cod_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

cod_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_cod_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in cod_bad_page_markers):
        return False

    for pattern in cod_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in cod_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in cod_team_name_keywords):
        return True

    return False


cod_dedup_df2 = source_df[
    source_df.apply(is_refined_cod_team_candidate, axis=1)
].copy().reset_index(drop=True)

cod_dedup_df2["title_clean"] = (
    cod_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

cod_dedup_df2 = cod_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Call of Duty 완화 후보 수:", len(cod_team_candidate_loose_df))
print("재정제 후 cod_dedup_df2 후보 수:", len(cod_dedup_df2))
print("예상 parse 소요 시간:", round(len(cod_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(cod_dedup_df2) * 31 / 3600, 2), "시간")

display(cod_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Call of Duty 완화 후보 수: 7
재정제 후 cod_dedup_df2 후보 수: 4
예상 parse 소요 시간: 2.1 분
예상 parse 소요 시간: 0.03 시간


,title,snippet,search_keyword,url
0,1K Gaming,Location: South Korea Region: Korea Approx. To...,korea,https://liquipedia.net/callofduty/1K_Gaming
1,Call of Duty: Mobile,"Garena for Southeast Asia on September 29, 201...",korea,https://liquipedia.net/callofduty/Call_of_Duty...
2,Ghosts,"Teams each from Brazil, Canada and Oceania qua...",korea,https://liquipedia.net/callofduty/Ghosts
3,Holic (Korean player),South Korea Region: Asia Status: Active Approx...,korea,https://liquipedia.net/callofduty/Holic_%28Kor...


In [61]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [62]:
def extract_cod_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "call of duty", "cod", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Statistics",
        "Gallery",
        "Past Logos",
        "References",
        "Achievements",
        "Recent Matches",
        "Upcoming Matches",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|stage|major|playoffs|championship|cup|tournament|league|series|qualifier|finals|cdl|cwl)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former players" in sub_low or "players" in sub_low:
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        row_pos = row_data.get("position", "")

        if table_type == "organization":
            if row_pos:
                return row_pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            # Call of Duty 표는 Name과 Join Date 사이에 Substitute/RFA 같은 역할이 들어가는 경우가 있음
            if row_pos:
                return row_pos

            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        # 표 구조 보정:
        # Active Player: ID / Name / Join Date 인데, 일부 row는 ID / Name / Position / Join Date처럼 들어가기도 함
        # Organization: ID / Name / Role / Join Date
        if len(normalized_headers) >= 4 and "position" not in normalized_headers:
            # ID, Name, 빈 역할, Join Date 구조 보정
            if normalized_headers[0] == "id" and normalized_headers[1] == "name":
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            # row의 cell 개수가 header보다 하나 많은 경우:
            # 예: ID / Name / Substitute / Join Date
            local_headers = normalized_headers.copy()
            if len(cells) == len(local_headers) + 1:
                if "position" not in local_headers and "join_date" in local_headers:
                    join_idx = local_headers.index("join_date")
                    local_headers.insert(join_idx, "position")

            for idx, cell in enumerate(cells):
                if idx >= len(local_headers):
                    continue

                col = local_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "Call of Duty",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [63]:
test_team = "FaZe Vegas"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_cod_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: FaZe Vegas
페이지 존재 여부: True
에러: None
추출 row 수: 30


,team_title,game_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,FaZe Vegas,Call of Duty,player_roster,Active,04,Jovan Rodriguez,Player,2025-09-19 [ 36 ],,,,United States,https://liquipedia.net/callofduty/04
1,FaZe Vegas,Call of Duty,player_roster,Active,Abuzah,Jordan François,Player,2025-09-19 [ 36 ],,,,Belgium,https://liquipedia.net/callofduty/Abuzah
2,FaZe Vegas,Call of Duty,player_roster,Active,Drazah,Zachary Jordan,Player,2025-09-19 [ 36 ],,,,United States,https://liquipedia.net/callofduty/Drazah
3,FaZe Vegas,Call of Duty,player_roster,Active,Simp,Chris Lehr,Player,2025-09-19 [ 36 ],,,,United States,https://liquipedia.net/callofduty/Simp
4,FaZe Vegas,Call of Duty,player_roster,Active,Alluka,Mohammed Bindal,Substitute,2025-12-29 [ 37 ],,,,Germany,https://liquipedia.net/callofduty/Alluka
5,FaZe Vegas,Call of Duty,player_roster,Former,JurNii,Juan Antonio González Muñoz,Substitute,2019-12-05 [ 6 ],,2020-10-22 [ 11 ],RAMS,Spain,https://liquipedia.net/callofduty/JurNii
6,FaZe Vegas,Call of Duty,player_roster,Former,GRVTY,Thomas Malin,Substitute,2019-12-05 [ 6 ],,2020-10-07 [ 10 ],EastR,United States,https://liquipedia.net/callofduty/GRVTY
7,FaZe Vegas,Call of Duty,player_roster,Former,MajorManiak,Michael Szymaniak,Former Player,2019-10-26 [ 5 ],,2020-09-15 [ 8 ],Minnesota RØKKR,United States,https://liquipedia.net/callofduty/Minnesota_R%...
8,FaZe Vegas,Call of Duty,player_roster,Former,Priestahh,Preston Greiner,Former Player,2019-10-26 [ 5 ],,2020-09-15 [ 8 ],Minnesota RØKKR,United States,https://liquipedia.net/callofduty/Priestahh
9,FaZe Vegas,Call of Duty,player_roster,Former,Sib,Daunte Gray,Substitute,2020-11-07 [ 12 ],,2021-09-20 [ 14 ],Seattle Surge,United States,https://liquipedia.net/callofduty/Sib


In [64]:
def crawl_cod_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_cod_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [65]:
cod_teams_checked_df, cod_teams_final_df, cod_people_raw_df, cod_errors_df = crawl_cod_teams_and_people_v4(
    candidate_df=cod_dedup_df2,
    output_prefix="callofduty_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(cod_teams_final_df.head(100))
display(cod_people_raw_df.head(100))

크롤링 대상 후보 수: 4
예상 소요 시간: 2.1 분
예상 소요 시간: 0.03 시간


callofduty_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/4 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 1
추출 인원 row 수: 12
에러 수: 0
저장 완료
/content/callofduty_korean_mainteam_refined_v4_teams_checked.csv
/content/callofduty_korean_mainteam_refined_v4_teams_final.csv
/content/callofduty_korean_mainteam_refined_v4_people_raw.csv
/content/callofduty_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,callofduty,Call of Duty,1K Gaming,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,True,True,[ e ][ h ] 1K Gaming Team Information Location...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22 [ 3 ],T1,South Korea,CanYou,https://liquipedia.net/callofduty/CanYou,CanYou Kim Tae-seong 2020-05-17 [ 1 ] 2020-10-...,1K Gaming,main_team_refined
1,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22 [ 3 ],T1,South Korea,F10W3R,https://liquipedia.net/callofduty/F10W3R,F10W3R Yoo Min-woo 2020-05-17 [ 1 ] 2020-10-22...,1K Gaming,main_team_refined
2,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22 [ 3 ],T1,South Korea,JongBok,https://liquipedia.net/callofduty/JongBok,JongBok Kim Jong-bok 2020-05-?? 2020-10-22 [ 3...,1K Gaming,main_team_refined
3,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22 [ 3 ],T1,South Korea,Gee,https://liquipedia.net/callofduty/Gee,Gee Go Joung-hyun 2020-08-12 [ 2 ] 2020-10-22 ...,1K Gaming,main_team_refined
4,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22 [ 3 ],T1,South Korea,index.php?title=Holic&action=edit&redlink=1,https://liquipedia.net/callofduty/index.php?ti...,Holic Lee Min-ho 2020-08-12 [ 2 ] 2020-10-22 [...,1K Gaming,main_team_refined
5,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22,,South Korea,index.php?title=Regret&action=edit&redlink=1,https://liquipedia.net/callofduty/index.php?ti...,Regret 2020-05-17 [ 1 ] 2020-10-22,1K Gaming,main_team_refined
6,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22,,South Korea,Chris,https://liquipedia.net/callofduty/Chris,Chris 2020-05-17 [ 1 ] 2020-10-22,1K Gaming,main_team_refined
7,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22,,South Korea,index.php?title=Goguma&action=edit&redlink=1,https://liquipedia.net/callofduty/index.php?ti...,Goguma 2020-05-17 [ 1 ] 2020-10-22,1K Gaming,main_team_refined
8,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-10-22,,South Korea,index.php?title=Exia&action=edit&redlink=1,https://liquipedia.net/callofduty/index.php?ti...,Exia 2020-08-12 [ 2 ] 2020-10-22,1K Gaming,main_team_refined
9,callofduty,Call of Duty,Call of Duty,1K Gaming,https://liquipedia.net/callofduty/1K_Gaming,Player Roster,Former,player_roster,Former,0,...,,,2020-??-??,,South Korea,index.php?title=Demon&action=edit&redlink=1,https://liquipedia.net/callofduty/index.php?ti...,Demon 2020-08-12 [ 2 ] 2020-??-??,1K Gaming,main_team_refined


In [66]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [67]:
cod_people_unique_df = make_unique_people_latest_from_raw(cod_people_raw_df)

print("raw row 수:", len(cod_people_raw_df))
print("최신 기준 unique 인원 수:", len(cod_people_unique_df))

display(cod_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 12
최신 기준 unique 인원 수: 12


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Call of Duty,1K Gaming,Former,CanYou,Kim Tae-seong,2020-05-17 [ 1 ],,,2020-10-22 [ 3 ],T1,South Korea,https://liquipedia.net/callofduty/CanYou
1,Call of Duty,1K Gaming,Former,Chris,,2020-05-17 [ 1 ],,,2020-10-22,,South Korea,https://liquipedia.net/callofduty/Chris
2,Call of Duty,1K Gaming,Former,F10W3R,Yoo Min-woo,2020-05-17 [ 1 ],,,2020-10-22 [ 3 ],T1,South Korea,https://liquipedia.net/callofduty/F10W3R
3,Call of Duty,1K Gaming,Former,Gee,Go Joung-hyun,2020-08-12 [ 2 ],,,2020-10-22 [ 3 ],T1,South Korea,https://liquipedia.net/callofduty/Gee
4,Call of Duty,1K Gaming,Former,JongBok,Kim Jong-bok,2020-05-??,,,2020-10-22 [ 3 ],T1,South Korea,https://liquipedia.net/callofduty/JongBok
5,Call of Duty,1K Gaming,Former,Demon,,2020-08-12 [ 2 ],,,2020-??-??,,South Korea,https://liquipedia.net/callofduty/index.php?ti...
6,Call of Duty,1K Gaming,Former,Exia,,2020-08-12 [ 2 ],,,2020-10-22,,South Korea,https://liquipedia.net/callofduty/index.php?ti...
7,Call of Duty,1K Gaming,Former,M.eg,,Manager,2020-05-17 [ 1 ],,2020-10-22,,South Korea,https://liquipedia.net/callofduty/index.php?ti...
8,Call of Duty,1K Gaming,Former,Goguma,,2020-05-17 [ 1 ],,,2020-10-22,,South Korea,https://liquipedia.net/callofduty/index.php?ti...
9,Call of Duty,1K Gaming,Former,Holic,Lee Min-ho,2020-08-12 [ 2 ],,,2020-10-22 [ 3 ],T1,South Korea,https://liquipedia.net/callofduty/index.php?ti...


In [68]:
cod_dedup_df2.to_csv(
    "/content/callofduty_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

cod_teams_checked_df.to_csv(
    "/content/callofduty_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

cod_teams_final_df.to_csv(
    "/content/callofduty_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

cod_people_raw_df.to_csv(
    "/content/callofduty_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

cod_people_unique_df.to_csv(
    "/content/callofduty_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

cod_errors_df.to_csv(
    "/content/callofduty_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/callofduty_teams_final_v4.csv")
print("/content/callofduty_people_unique_v4.csv")

저장 완료
/content/callofduty_teams_final_v4.csv
/content/callofduty_people_unique_v4.csv


#레인보우식스

In [69]:
# =========================================
# Rainbow Six Siege Liquipedia 설정
# =========================================

GAME_SLUG = "rainbowsix"
GAME_LABEL = "Rainbow Six Siege"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanRainbowSixEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/rainbowsix
https://liquipedia.net/rainbowsix/api.php


In [70]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Rainbow Six Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [71]:
RAINBOWSIX_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "Rainbow Six Korea",
    "Rainbow Six Korean",
    "Rainbow Six Siege Korea",
    "Rainbow Six Siege Korean",
    "R6 Korea",
    "R6 Korean",

    # 한국/관련 팀 후보
    "FEARX",
    "FearX",
    "BNK FEARX",
    "SANDBOX Gaming",
    "Liiv SANDBOX",
    "Dplus KIA",
    "DWG KIA",
    "KIA",
    "Talon Esports",
    "WEBL",
    "Cloud9 Korea",
    "Team BlossoM",
    "Mantis FPS",
    "SCARZ Korea",
    "T1",
    "Gen.G",
    "DRX",
]

In [72]:
search_rows = []

for keyword in RAINBOWSIX_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

r6_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(r6_search_df))
display(r6_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: korea / offset=650
검색: korea / offset=700
검색: korea / offset=750
검색: korea / offset=800
검색: korea / offset=850
검색: korea / offset=900
검색: korea / offset=950
max_total 도달: 1000
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: South Korea / offset=350
검색: South Korea / offset=400
검색: South Korea / offset=450
검색: South Korea / offset=500
검색: South Korea / offset=550
검색: South Korea / offset=600
검색: South Korea / offset=650
검색: South Korea / offset=700
검색: South Korea / offset=750
검색: South Korea / offset=800
검색: South Korea / offset=850
검색: S

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,rainbowsix,Rainbow Six Siege,A-Tier Tournaments,North America 9 Soniqs DarkZero South Korea Le...,44034,https://liquipedia.net/rainbowsix/A-Tier_Tourn...,korea
1,rainbowsix,Rainbow Six Siege,B-Tier Tournaments,"04–18, 2020 $21,749 France 8 PENTA IziDream Ko...",44043,https://liquipedia.net/rainbowsix/B-Tier_Tourn...,korea
2,rainbowsix,Rainbow Six Siege,Weekly Tournaments,Winner Runner-up Week. (D-Tier) BLAST Communit...,30787,https://liquipedia.net/rainbowsix/Weekly_Tourn...,korea
3,rainbowsix,Rainbow Six Siege,Qualifier Tournaments/2026,Challenger Series - North - Open Qualifier #4 ...,75097,https://liquipedia.net/rainbowsix/Qualifier_To...,korea
4,rainbowsix,Rainbow Six Siege,D-Tier Tournaments,OneUp Esports STRIKER in the hot summer ~R6S J...,44045,https://liquipedia.net/rainbowsix/D-Tier_Tourn...,korea
...,...,...,...,...,...,...,...
95,rainbowsix,Rainbow Six Siege,SummerRain,Player Information Name: 김인영 Romanized Name: K...,27207,https://liquipedia.net/rainbowsix/SummerRain,korea
96,rainbowsix,Rainbow Six Siege,Korea Cup/2018/Monthly/August,Korea Cup - Monthly August 2018 League Informa...,21716,https://liquipedia.net/rainbowsix/Korea_Cup/20...,korea
97,rainbowsix,Rainbow Six Siege,Korea Cup/2019/Weekly/May/3,Korea Cup - Weekly #3 May 2019 League Informat...,34079,https://liquipedia.net/rainbowsix/Korea_Cup/20...,korea
98,rainbowsix,Rainbow Six Siege,Korea Cup/2019/Weekly/July/2,Korea Cup - Weekly #2 July 2019 League Informa...,36996,https://liquipedia.net/rainbowsix/Korea_Cup/20...,korea


In [74]:
def build_r6_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "rainbow six",
        "r6",
        "siege",
        "academy",

        "fearx",
        "sandbox",
        "liiv",
        "bnk",
        "dplus",
        "dwg",
        "kia",
        "talon",
        "webl",
        "cloud9",
        "blossom",
        "mantis",
        "scarz",
        "t1",
        "gen.g",
        "drx",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "stage",
        "major",
        "minor",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "rankings",
        "awards",
        "finals",
        "playoffs",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Rainbow Six 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [75]:
r6_team_candidate_loose_df = build_r6_team_candidate_loose_df(r6_search_df)

display(r6_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Rainbow Six 완화 후보 수: 56
예상 parse 소요 시간: 28.9 분
예상 parse 소요 시간: 0.48 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea B...,korea,https://liquipedia.net/rainbowsix/Dplus
1,Talon Esports,Talon Esports Team Information Location: South...,korea,https://liquipedia.net/rainbowsix/Talon_Esports
2,FEARX,FEARX Team Information Location: South Korea R...,korea,https://liquipedia.net/rainbowsix/FEARX
3,Beyond Stratos Gaming,Beyond Stratos Gaming Team Information Locatio...,korea,https://liquipedia.net/rainbowsix/Beyond_Strat...
4,Mir Gaming,South Korea Region: Asia Approx. Total Winning...,korea,https://liquipedia.net/rainbowsix/Mir_Gaming
5,Ram,General Information Real Name: Bo-Ram Choi Bor...,korea,https://liquipedia.net/rainbowsix/Ram
6,Mantis FPS,mantis FPS Team Information Location: South Ko...,korea,https://liquipedia.net/rainbowsix/Mantis_FPS
7,PSG Talon,PSG Talon Team Information Location: Hong Kong...,korea,https://liquipedia.net/rainbowsix/PSG_Talon
8,BlossoM,South Korea Region: Asia Approx. Total Winning...,korea,https://liquipedia.net/rainbowsix/BlossoM
9,WEBL,South Korea Region: Asia Approx. Total Winning...,korea,https://liquipedia.net/rainbowsix/WEBL


In [76]:
# =========================================
# Rainbow Six 완화 후보에서 팀 후보만 재정제
# output: r6_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = r6_team_candidate_loose_df.copy()

r6_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "rainbow six",
    "r6",
    "siege",
    "academy",

    "fearx",
    "sandbox",
    "liiv",
    "bnk",
    "dplus",
    "dwg",
    "kia",
    "talon",
    "webl",
    "cloud9",
    "blossom",
    "mantis",
    "scarz",
    "t1",
    "gen.g",
    "drx",
]

r6_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "stage",
    "major",
    "minor",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "rankings",
    "awards",
    "finals",
    "playoffs",
]

r6_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

r6_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_r6_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in r6_bad_page_markers):
        return False

    for pattern in r6_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in r6_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in r6_team_name_keywords):
        return True

    return False


r6_dedup_df2 = source_df[
    source_df.apply(is_refined_r6_team_candidate, axis=1)
].copy().reset_index(drop=True)

r6_dedup_df2["title_clean"] = (
    r6_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

r6_dedup_df2 = r6_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Rainbow Six 완화 후보 수:", len(r6_team_candidate_loose_df))
print("재정제 후 r6_dedup_df2 후보 수:", len(r6_dedup_df2))
print("예상 parse 소요 시간:", round(len(r6_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(r6_dedup_df2) * 31 / 3600, 2), "시간")

display(r6_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Rainbow Six 완화 후보 수: 56
재정제 후 r6_dedup_df2 후보 수: 56
예상 parse 소요 시간: 28.9 분
예상 parse 소요 시간: 0.48 시간


,title,snippet,search_keyword,url
0,Dplus,Dplus Team Information Location: South Korea B...,korea,https://liquipedia.net/rainbowsix/Dplus
1,Talon Esports,Talon Esports Team Information Location: South...,korea,https://liquipedia.net/rainbowsix/Talon_Esports
2,FEARX,FEARX Team Information Location: South Korea R...,korea,https://liquipedia.net/rainbowsix/FEARX
3,Beyond Stratos Gaming,Beyond Stratos Gaming Team Information Locatio...,korea,https://liquipedia.net/rainbowsix/Beyond_Strat...
4,Mir Gaming,South Korea Region: Asia Approx. Total Winning...,korea,https://liquipedia.net/rainbowsix/Mir_Gaming
5,Ram,General Information Real Name: Bo-Ram Choi Bor...,korea,https://liquipedia.net/rainbowsix/Ram
6,Mantis FPS,mantis FPS Team Information Location: South Ko...,korea,https://liquipedia.net/rainbowsix/Mantis_FPS
7,PSG Talon,PSG Talon Team Information Location: Hong Kong...,korea,https://liquipedia.net/rainbowsix/PSG_Talon
8,BlossoM,South Korea Region: Asia Approx. Total Winning...,korea,https://liquipedia.net/rainbowsix/BlossoM
9,WEBL,South Korea Region: Asia Approx. Total Winning...,korea,https://liquipedia.net/rainbowsix/WEBL


In [77]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [78]:
def extract_r6_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "rainbow six", "rainbowsix", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Statistics",
        "Awards",
        "Logos",
        "Media",
        "References",
        "Gallery",
        "Upcoming Matches",
        "Upcoming Tournaments",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|stage|major|minor|playoffs|championship|cup|tournament|league|series|qualifier|finals|six invitational|blast)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former players" in sub_low or "players" in sub_low:
            if "stand" in sub_low:
                return "player_roster", "Stand-in", main_section
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        row_pos = row_data.get("position", "")

        if table_type == "organization":
            if row_pos:
                return row_pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if row_pos:
                return row_pos
            if status == "Stand-in":
                return "Stand-in"
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if len(normalized_headers) >= 4 and "position" not in normalized_headers:
            if normalized_headers[0] == "id" and normalized_headers[1] == "name":
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}

            local_headers = normalized_headers.copy()

            if len(cells) == len(local_headers) + 1:
                if "position" not in local_headers and "join_date" in local_headers:
                    join_idx = local_headers.index("join_date")
                    local_headers.insert(join_idx, "position")

            for idx, cell in enumerate(cells):
                if idx >= len(local_headers):
                    continue

                col = local_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "Rainbow Six Siege",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [79]:
test_team = "FEARX"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_r6_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: FEARX
페이지 존재 여부: True
에러: None
추출 row 수: 29


,team_title,game_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,FEARX,Rainbow Six Siege,player_roster,Active,Woogiman,Park Jin-wook,Player,2025-02-17 [ 23 ],,,,South Korea,https://liquipedia.net/rainbowsix/Woogiman
1,FEARX,Rainbow Six Siege,player_roster,Active,Misa,Hong Sang-yeong,Player,2026-03-11 [ 27 ],,,,South Korea,https://liquipedia.net/rainbowsix/Misa
2,FEARX,Rainbow Six Siege,player_roster,Active,NL,Hwang Gyo-min,Player,2026-03-11 [ 27 ],,,,South Korea,https://liquipedia.net/rainbowsix/NL
3,FEARX,Rainbow Six Siege,player_roster,Active,RuMaTick,Kim Jang-wook,Player,2026-03-11 [ 27 ],,,,South Korea,https://liquipedia.net/rainbowsix/RuMaTick
4,FEARX,Rainbow Six Siege,player_roster,Active,Stettzll,,Player,2026-03-11 [ 27 ],,,,South Korea,https://liquipedia.net/rainbowsix/Stettzll
5,FEARX,Rainbow Six Siege,player_roster,Former,SyAIL,Song Dong-seon,2021-10-22 [ 2 ],,,2022-12-12 [ 8 ],Retired,South Korea,https://liquipedia.net/rainbowsix/SyAIL
6,FEARX,Rainbow Six Siege,player_roster,Former,Harp3rXD,Lee Hyo-jun,2021-10-22 [ 2 ],,,2022-05-27 [ 6 ],Burning Core,South Korea,https://liquipedia.net/rainbowsix/Harp3rXD
7,FEARX,Rainbow Six Siege,player_roster,Former,Nova,Lee Si-hun,2021-10-22 [ 2 ],,,2022-03-15 [ 5 ],SANDBOX Gaming (Coach),South Korea,https://liquipedia.net/rainbowsix/Nova
8,FEARX,Rainbow Six Siege,player_roster,Former,EnvyTaylor,Kim Seong-soo,2021-10-22 [ 2 ],,,2023-08-08 [ 12 ],SANDBOX Gaming (Coach),South Korea,https://liquipedia.net/rainbowsix/EnvyTaylor
9,FEARX,Rainbow Six Siege,player_roster,Former,Static,Han Chan-yong,2021-10-22 [ 2 ],,,2023-02-06 [ 10 ],Retired,South Korea,https://liquipedia.net/rainbowsix/Static


In [80]:
def crawl_r6_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_r6_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [81]:
r6_teams_checked_df, r6_teams_final_df, r6_people_raw_df, r6_errors_df = crawl_r6_teams_and_people_v4(
    candidate_df=r6_dedup_df2,
    output_prefix="rainbowsix_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(r6_teams_final_df.head(100))
display(r6_people_raw_df.head(100))

크롤링 대상 후보 수: 56
예상 소요 시간: 28.9 분
예상 소요 시간: 0.48 시간


rainbowsix_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/56 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 22
추출 인원 row 수: 393
에러 수: 0
저장 완료
/content/rainbowsix_korean_mainteam_refined_v4_teams_checked.csv
/content/rainbowsix_korean_mainteam_refined_v4_teams_final.csv
/content/rainbowsix_korean_mainteam_refined_v4_people_raw.csv
/content/rainbowsix_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,rainbowsix,Rainbow Six Siege,Dplus,Dplus,https://liquipedia.net/rainbowsix/Dplus,True,True,[ e ][ h ] Dplus Team Information Location: So...,None,main_team_refined
1,rainbowsix,Rainbow Six Siege,Talon Esports,Talon Esports,https://liquipedia.net/rainbowsix/Talon_Esports,True,True,[ e ][ h ] Talon Esports Team Information Loca...,None,main_team_refined
2,rainbowsix,Rainbow Six Siege,FEARX,FEARX,https://liquipedia.net/rainbowsix/FEARX,True,True,[ e ][ h ] FEARX Team Information Location: So...,None,main_team_refined
3,rainbowsix,Rainbow Six Siege,Beyond Stratos Gaming,Beyond Stratos Gaming,https://liquipedia.net/rainbowsix/Beyond_Strat...,True,True,[ e ][ h ] Beyond Stratos Gaming Team Informat...,None,main_team_refined
4,rainbowsix,Rainbow Six Siege,Mir Gaming,Mir Gaming,https://liquipedia.net/rainbowsix/Mir_Gaming,True,True,[ e ][ h ] Mir Gaming Team Information Locatio...,None,main_team_refined
5,rainbowsix,Rainbow Six Siege,Mantis FPS,Mantis FPS,https://liquipedia.net/rainbowsix/Mantis_FPS,True,True,[ e ][ h ] mantis FPS Team Information Locatio...,None,main_team_refined
6,rainbowsix,Rainbow Six Siege,PSG Talon,PSG Talon,https://liquipedia.net/rainbowsix/PSG_Talon,True,True,[ e ][ h ] PSG Talon Team Information Location...,None,main_team_refined
7,rainbowsix,Rainbow Six Siege,BlossoM,BlossoM,https://liquipedia.net/rainbowsix/BlossoM,True,True,[ e ][ h ] BlossoM Team Information Location: ...,None,main_team_refined
8,rainbowsix,Rainbow Six Siege,WEBL,WEBL,https://liquipedia.net/rainbowsix/WEBL,True,True,[ e ][ h ] WEBL Team Information Location: Sou...,None,main_team_refined
9,rainbowsix,Rainbow Six Siege,LAVEGA Esports,LAVEGA Esports,https://liquipedia.net/rainbowsix/LAVEGA_Esports,True,True,[ e ][ h ] LAVEGA Esports Team Information Loc...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Dplus,https://liquipedia.net/rainbowsix/Dplus,Player Roster,Active,player_roster,Active,0,...,2025-02-18 [ 31 ],,,,Brazil,Faallz,https://liquipedia.net/rainbowsix/Faallz,Faallz Kaique Stephano Moreira 2025-02-18 [ 31 ],Dplus,main_team_refined
1,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Dplus,https://liquipedia.net/rainbowsix/Dplus,Player Roster,Active,player_roster,Active,0,...,2025-02-18 [ 31 ],,,,Brazil,Levy,https://liquipedia.net/rainbowsix/Levy,Levy Juliano Andrade dos Santos Benos 2025-02-...,Dplus,main_team_refined
2,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Dplus,https://liquipedia.net/rainbowsix/Dplus,Player Roster,Active,player_roster,Active,0,...,2025-02-18 [ 33 ],,,,Brazil,Mity,https://liquipedia.net/rainbowsix/Mity,Mity Dyjair Soares 2025-02-18 [ 33 ],Dplus,main_team_refined
3,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Dplus,https://liquipedia.net/rainbowsix/Dplus,Player Roster,Active,player_roster,Active,0,...,2025-02-18 [ 31 ],,,,Brazil,NearZ,https://liquipedia.net/rainbowsix/NearZ,NearZ Nicolas Fresnel 2025-02-18 [ 31 ],Dplus,main_team_refined
4,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Dplus,https://liquipedia.net/rainbowsix/Dplus,Player Roster,Active,player_roster,Active,0,...,2026-03-17 [ 36 ],,,,Brazil,Muzi,https://liquipedia.net/rainbowsix/Muzi,Muzi Murilo Ripoli Moscatelli 2026-03-17 [ 36 ],Dplus,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Beyond Stratos Gaming,https://liquipedia.net/rainbowsix/Beyond_Strat...,Player Roster,Former,player_roster,Former,0,...,,,2023-05-08 [ 7 ],BlossoM,South Korea,Kira-Miki,https://liquipedia.net/rainbowsix/Kira-Miki,Kira-Miki Park Do-hyeon 2022-08-29 [ 1 ] 2023-...,Beyond Stratos Gaming,main_team_refined
96,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Beyond Stratos Gaming,https://liquipedia.net/rainbowsix/Beyond_Strat...,Player Roster,Former,player_roster,Former,0,...,,,2023-05-08 [ 7 ],Retired,South Korea,Retaddress,https://liquipedia.net/rainbowsix/Retaddress,retaddress Lee Sang-min 2022-08-29 [ 1 ] 2023-...,Beyond Stratos Gaming,main_team_refined
97,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Beyond Stratos Gaming,https://liquipedia.net/rainbowsix/Beyond_Strat...,Player Roster,Former,player_roster,Former,1,...,,,2024-06-05 [ 17 ],Ex-Beyond Stratos Gaming,South Korea,CrazyBoy,https://liquipedia.net/rainbowsix/CrazyBoy,CrazyBoy Choi Min-ho 2023-05-16 [ 9 ] 2024-06-...,Beyond Stratos Gaming,main_team_refined
98,rainbowsix,Rainbow Six Siege,Rainbow Six Siege,Beyond Stratos Gaming,https://liquipedia.net/rainbowsix/Beyond_Strat...,Player Roster,Former,player_roster,Former,1,...,,,2024-06-05 [ 17 ],Retired,South Korea,Dochi,https://liquipedia.net/rainbowsix/Dochi,Dochi Hong Ji-ho 2024-01-02 [ 16 ] 2024-06-05 ...,Beyond Stratos Gaming,main_team_refined


In [82]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [83]:
r6_people_unique_df = make_unique_people_latest_from_raw(r6_people_raw_df)

print("raw row 수:", len(r6_people_raw_df))
print("최신 기준 unique 인원 수:", len(r6_people_unique_df))

display(r6_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 393
최신 기준 unique 인원 수: 182


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Rainbow Six Siege,BlossoM,Former,aDonis,Kim Hee-man,2024-02-19 [ 8 ],,,2024-09-01 [ 11 ],Retired,South Korea,https://liquipedia.net/rainbowsix/ADonis
1,Rainbow Six Siege,Mir Gaming,Former,aEnde,Jeong Tae-hoon,2025-05-30 [ 8 ],,,2025-08-12 [ 12 ],Pannuhuone (Assistant Coach),South Korea,https://liquipedia.net/rainbowsix/AEnde
2,Rainbow Six Siege,BlossoM,Former,Accident,Yoo Yeon-woo,2023-03-09 [ 1 ],,,2024-02-19 [ 7 ],Retired,South Korea,https://liquipedia.net/rainbowsix/Accident
3,Rainbow Six Siege,Gen.G Esports,Former,Akhdar,Erdal Coti,Head Coach,2025-04-23 [ 1 ],,2026-01-23 [ 8 ],Retired,France,https://liquipedia.net/rainbowsix/Akhdar
4,Rainbow Six Siege,SGA eSPORTS,Former,Arms_Tina,Yang Dong-young,Coach,2021-02-03 [ 8 ],,2021-07-29 [ 12 ],Esports at WMU,South Korea,https://liquipedia.net/rainbowsix/Arms_Tina
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Rainbow Six Siege,Beyond Stratos Gaming,Former,MinGoran,Kim Min-geun,Head Coach,2022-08-29 [ 1 ],,2023-05-14 [ 8 ],Team Bliss (Coach/Analyst),South Korea,https://liquipedia.net/rainbowsix/MinGoran
96,Rainbow Six Siege,FEARX,Active,Misa,Hong Sang-yeong,Player,2026-03-11 [ 27 ],,,,South Korea,https://liquipedia.net/rainbowsix/Misa
97,Rainbow Six Siege,Dplus,Active,Mity,Dyjair Soares,Player,2025-02-18 [ 33 ],,,,Brazil,https://liquipedia.net/rainbowsix/Mity
98,Rainbow Six Siege,WEBL,Former,moon,Mun Seung-chan,2024-09-01 [ 7 ],,,2024-12-07 [ 11 ],Retired,South Korea,https://liquipedia.net/rainbowsix/Moon_(Korean...


In [95]:
r6_dedup_df2.to_csv(
    "/content/rainbowsix_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

r6_teams_checked_df.to_csv(
    "/content/rainbowsix_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

r6_teams_final_df.to_csv(
    "/content/rainbowsix_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

r6_people_raw_df.to_csv(
    "/content/rainbowsix_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

r6_people_unique_df.to_csv(
    "/content/rainbowsix_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

r6_errors_df.to_csv(
    "/content/rainbowsix_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/rainbowsix_teams_final_v4.csv")
print("/content/rainbowsix_people_unique_v4.csv")

저장 완료
/content/rainbowsix_teams_final_v4.csv
/content/rainbowsix_people_unique_v4.csv


#에이펙스


In [84]:
# =========================================
# Apex Legends Liquipedia 설정
# =========================================

GAME_SLUG = "apexlegends"
GAME_LABEL = "Apex Legends"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanApexLegendsEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/apexlegends
https://liquipedia.net/apexlegends/api.php


In [85]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Apex Legends Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [86]:
APEX_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "Apex Korea",
    "Apex Legends Korea",
    "Apex Legends Korean",
    "ALGS Korea",
    "ALGS Korean",

    # 한국/아시아권 관련 후보
    "ENTER FORCE.36",
    "ENTER FORCE 36",
    "ENTER FORCE",
    "Crazy Raccoon Korea",
    "T1",
    "Gen.G",
    "DRX",
    "Dplus KIA",
    "Dplus",
    "DWG KIA",
    "Kwangdong Freecs",
    "Afreeca Freecs",
    "Nongshim RedForce",
    "Team BlossoM",
    "REJECT",
    "Riddle",
    "FNATIC Korea",
]

In [87]:
search_rows = []

for keyword in APEX_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

apex_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(apex_search_df))
display(apex_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: korea / offset=650
검색: korea / offset=700
검색: korea / offset=750
검색: korea / offset=800
검색: korea / offset=850
검색: korea / offset=900
검색: korea / offset=950
max_total 도달: 1000
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: South Korea / offset=350
검색: South Korea / offset=400
검색: South Korea / offset=450
검색: South Korea / offset=500
검색: South Korea / offset=550
검색: South Korea / offset=600
검색: South Korea / offset=650
검색: South Korea / offset=700
검색: South Korea / offset=750
검색: South Korea / offset=800
검색: South Korea / offset=850
검색: S

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,apexlegends,Apex Legends,C-Tier Tournaments,Split 2 Challenger Circuit #4 - APAC North Oct...,29106,https://liquipedia.net/apexlegends/C-Tier_Tour...,korea
1,apexlegends,Apex Legends,Qualifier Tournaments,Split 1 Challenger Circuit #4 - APAC North Jun...,729,https://liquipedia.net/apexlegends/Qualifier_T...,korea
2,apexlegends,Apex Legends,B-Tier Tournaments,TBD ALGS: 2026 Online Open #4 - APAC North Mar...,342,https://liquipedia.net/apexlegends/B-Tier_Tour...,korea
3,apexlegends,Apex Legends,Apex Legends Global Series/2026-27,"Turkey, Ukraine, United Arab Emirates, United ...",53398,https://liquipedia.net/apexlegends/Apex_Legend...,korea
4,apexlegends,Apex Legends,Apex Legends Global Series/Online Tournament 6...,ALGS Online #6 - Korea League Information Seri...,26515,https://liquipedia.net/apexlegends/Apex_Legend...,korea
...,...,...,...,...,...,...,...
95,apexlegends,Apex Legends,GLL/Community Cups 14/APAC North,Community Cups Organizer: GLL Platform: PC Typ...,29633,https://liquipedia.net/apexlegends/GLL/Communi...,korea
96,apexlegends,Apex Legends,GLL/Community Cups 1/APAC North,Community Cups Organizer: GLL Platform: PC Typ...,27973,https://liquipedia.net/apexlegends/GLL/Communi...,korea
97,apexlegends,Apex Legends,GLL/Community Cups 7/APAC North,Community Cups Organizer: GLL Platform: PC Typ...,28709,https://liquipedia.net/apexlegends/GLL/Communi...,korea
98,apexlegends,Apex Legends,GLL/Community Cups 4/APAC North,Community Cups Organizer: GLL Platform: PC Typ...,28496,https://liquipedia.net/apexlegends/GLL/Communi...,korea


In [88]:
def build_apex_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "apex",
        "apex legends",
        "algs",

        "enter force",
        "enter force.36",
        "t1",
        "gen.g",
        "drx",
        "dplus",
        "dwg",
        "kia",
        "kwangdong",
        "freecs",
        "afreeca",
        "nongshim",
        "redforce",
        "blossom",
        "reject",
        "riddle",
        "fnatic",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "split",
        "stage",
        "major",
        "minor",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "rankings",
        "awards",
        "finals",
        "playoffs",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "south korea",
            "korea",
            "korean",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Apex Legends 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [89]:

apex_team_candidate_loose_df = build_apex_team_candidate_loose_df(apex_search_df)

display(apex_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Apex Legends 완화 후보 수: 115
예상 parse 소요 시간: 59.4 분
예상 parse 소요 시간: 0.99 시간


,title,snippet,search_keyword,url
0,Apex Legends Global Series,Championship 2021 - APAC North May 22 – Jun 06...,korea,https://liquipedia.net/apexlegends/Apex_Legend...
1,Parkha,&quot;Parkha&quot; Jeong-teak (born February 4...,korea,https://liquipedia.net/apexlegends/Parkha
2,Electronic Arts,Black MXF 2023 APEX LEGENDS KOREA CHAMPIONSHIP...,korea,https://liquipedia.net/apexlegends/Electronic_...
3,ENTER FORCE.36,ENTER FORCE.36 Team Information Location: Japa...,korea,https://liquipedia.net/apexlegends/ENTER_FORCE.36
4,T1,T1 Team Information Location: South Korea Regi...,korea,https://liquipedia.net/apexlegends/T1
...,...,...,...,...
110,Mirage (Taiwanese player),- 2019-12-12 18th D-Tier (Mon.) T1 Apex Korean...,korea,https://liquipedia.net/apexlegends/Mirage_%28T...
111,Far East Society,$581 2019-12-12 10th D-Tier (Mon.) T1 Apex Kor...,korea,https://liquipedia.net/apexlegends/Far_East_So...
112,PinOcChiOs,UP League - No Space for The Weak - 2019-10-31...,korea,https://liquipedia.net/apexlegends/PinOcChiOs
113,Ki5tE,UP League - No Space for The Weak - 2019-10-31...,korea,https://liquipedia.net/apexlegends/Ki5tE


In [90]:
# =========================================
# Apex Legends 완화 후보에서 팀 후보만 재정제
# output: apex_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = apex_team_candidate_loose_df.copy()

apex_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "apex",
    "apex legends",
    "algs",

    "enter force",
    "enter force.36",
    "t1",
    "gen.g",
    "drx",
    "dplus",
    "dwg",
    "kia",
    "kwangdong",
    "freecs",
    "afreeca",
    "nongshim",
    "redforce",
    "blossom",
    "reject",
    "riddle",
    "fnatic",
]

apex_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "split",
    "stage",
    "major",
    "minor",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "rankings",
    "awards",
    "finals",
    "playoffs",
]

apex_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

apex_korea_terms = [
    "location: south korea",
    "region: korea",
    "south korea",
    "korea",
    "korean",
]


def is_refined_apex_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in apex_bad_page_markers):
        return False

    for pattern in apex_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in apex_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in apex_team_name_keywords):
        return True

    return False


apex_dedup_df2 = source_df[
    source_df.apply(is_refined_apex_team_candidate, axis=1)
].copy().reset_index(drop=True)

apex_dedup_df2["title_clean"] = (
    apex_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

apex_dedup_df2 = apex_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Apex Legends 완화 후보 수:", len(apex_team_candidate_loose_df))
print("재정제 후 apex_dedup_df2 후보 수:", len(apex_dedup_df2))
print("예상 parse 소요 시간:", round(len(apex_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(apex_dedup_df2) * 31 / 3600, 2), "시간")

display(apex_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Apex Legends 완화 후보 수: 115
재정제 후 apex_dedup_df2 후보 수: 115
예상 parse 소요 시간: 59.4 분
예상 parse 소요 시간: 0.99 시간


,title,snippet,search_keyword,url
0,Apex Legends Global Series,Championship 2021 - APAC North May 22 – Jun 06...,korea,https://liquipedia.net/apexlegends/Apex_Legend...
1,Parkha,&quot;Parkha&quot; Jeong-teak (born February 4...,korea,https://liquipedia.net/apexlegends/Parkha
2,Electronic Arts,Black MXF 2023 APEX LEGENDS KOREA CHAMPIONSHIP...,korea,https://liquipedia.net/apexlegends/Electronic_...
3,ENTER FORCE.36,ENTER FORCE.36 Team Information Location: Japa...,korea,https://liquipedia.net/apexlegends/ENTER_FORCE.36
4,T1,T1 Team Information Location: South Korea Regi...,korea,https://liquipedia.net/apexlegends/T1
...,...,...,...,...
110,Mirage (Taiwanese player),- 2019-12-12 18th D-Tier (Mon.) T1 Apex Korean...,korea,https://liquipedia.net/apexlegends/Mirage_%28T...
111,Far East Society,$581 2019-12-12 10th D-Tier (Mon.) T1 Apex Kor...,korea,https://liquipedia.net/apexlegends/Far_East_So...
112,PinOcChiOs,UP League - No Space for The Weak - 2019-10-31...,korea,https://liquipedia.net/apexlegends/PinOcChiOs
113,Ki5tE,UP League - No Space for The Weak - 2019-10-31...,korea,https://liquipedia.net/apexlegends/Ki5tE


In [91]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [92]:
def extract_apex_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "apex legends", "apex", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Statistics",
        "Awards",
        "Logos",
        "Media",
        "References",
        "Gallery",
        "Upcoming Matches",
        "Upcoming Tournaments",
        "Timeline",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h in ["role", "position"]:
            return "position"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|split|stage|major|minor|playoffs|championship|cup|tournament|league|series|qualifier|finals|algs)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former players" in sub_low or "players" in sub_low:
            if "stand" in sub_low:
                return "player_roster", "Stand-in", main_section
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        row_pos = row_data.get("position", "")

        if table_type == "organization":
            if row_pos:
                return row_pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if row_pos:
                return row_pos
            if status == "Stand-in":
                return "Stand-in"
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if len(normalized_headers) >= 4 and "position" not in normalized_headers:
            if normalized_headers[0] == "id" and normalized_headers[1] == "name":
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}
            local_headers = normalized_headers.copy()

            if len(cells) == len(local_headers) + 1:
                if "position" not in local_headers and "join_date" in local_headers:
                    join_idx = local_headers.index("join_date")
                    local_headers.insert(join_idx, "position")

            for idx, cell in enumerate(cells):
                if idx >= len(local_headers):
                    continue

                col = local_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []
            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "Apex Legends",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [93]:
test_team = "ENTER FORCE.36"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_apex_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: ENTER FORCE.36
페이지 존재 여부: True
에러: None
추출 row 수: 14


,team_title,game_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,ENTER FORCE.36,Apex Legends,player_roster,Active,Cinap,Kim Sang-rok (김상록),Player,2024-01-28 [ 6 ] [ 8 ],,,,South Korea,https://liquipedia.net/apexlegends/Cinap
1,ENTER FORCE.36,Apex Legends,player_roster,Active,Jusna,Shin Yong-ju (신용주),Player,2025-03-01 [ 10 ],,,,South Korea,https://liquipedia.net/apexlegends/Jusna
2,ENTER FORCE.36,Apex Legends,player_roster,Active,Obly,Lim Jung-hyun (임정현),Player,2026-03-20 [ 14 ],,,,South Korea,https://liquipedia.net/apexlegends/Obly
3,ENTER FORCE.36,Apex Legends,player_roster,Active,Aimbot,Park Ji-hoon (박지훈),Substitute,2026-03-20 [ 14 ],,,,South Korea,https://liquipedia.net/apexlegends/Aimbot
4,ENTER FORCE.36,Apex Legends,player_roster,Former,ILY,Choi Jun-hyeok (최준혁),2023-03-22 [ 3 ],,2026-02-05 [ 11 ],2026-02-26 [ 12 ] [ 13 ],Fnatic,South Korea,https://liquipedia.net/apexlegends/ILY
5,ENTER FORCE.36,Apex Legends,player_roster,Former,Aimbot,Park Ji-hoon (박지훈),2022-06-17 [ 1 ],,,2025-02-12 [ 9 ],ENTER FORCE.36 (Substitute),South Korea,https://liquipedia.net/apexlegends/Aimbot
6,ENTER FORCE.36,Apex Legends,player_roster,Former,Vor3z,,2023-07-01 [ 4 ],,,2024-02-06 [ 7 ],Diaz,South Korea,https://liquipedia.net/apexlegends/Vor3z
7,ENTER FORCE.36,Apex Legends,player_roster,Former,YunD,Lee Yoon-ho (이윤호),2022-06-17 [ 1 ],,,2023-09-30 [ 5 ],Crazy Raccoon (Coach),South Korea,https://liquipedia.net/apexlegends/YunD
8,ENTER FORCE.36,Apex Legends,player_roster,Former,ahn2e,Kim Gun-ho (김건호),2022-06-17 [ 1 ],,,2023-03-06 [ 2 ],REJECT (Coach),South Korea,https://liquipedia.net/apexlegends/Ahn2e
9,ENTER FORCE.36,Apex Legends,player_roster,Stand-in,Vor3z,,Stand-in,,,,,South Korea,https://liquipedia.net/apexlegends/Cinap


In [94]:
def crawl_apex_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "region:korea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_apex_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [96]:
apex_teams_checked_df, apex_teams_final_df, apex_people_raw_df, apex_errors_df = crawl_apex_teams_and_people_v4(
    candidate_df=apex_dedup_df2,
    output_prefix="apexlegends_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(apex_teams_final_df.head(100))
display(apex_people_raw_df.head(100))

크롤링 대상 후보 수: 115
예상 소요 시간: 59.4 분
예상 소요 시간: 0.99 시간


apexlegends_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/115 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 31
추출 인원 row 수: 243
에러 수: 0
저장 완료
/content/apexlegends_korean_mainteam_refined_v4_teams_checked.csv
/content/apexlegends_korean_mainteam_refined_v4_teams_final.csv
/content/apexlegends_korean_mainteam_refined_v4_people_raw.csv
/content/apexlegends_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,apexlegends,Apex Legends,ENTER FORCE.36,ENTER FORCE.36,https://liquipedia.net/apexlegends/ENTER_FORCE.36,True,True,[ e ][ h ] ENTER FORCE.36 Team Information Loc...,None,main_team_refined
1,apexlegends,Apex Legends,T1,T1,https://liquipedia.net/apexlegends/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
2,apexlegends,Apex Legends,FENNEL Korea,FENNEL Korea,https://liquipedia.net/apexlegends/FENNEL_Korea,True,True,[ e ][ h ] FENNEL Korea Team Information Locat...,None,main_team_refined
3,apexlegends,Apex Legends,OP.GG Sports,OP.GG Sports,https://liquipedia.net/apexlegends/OP.GG_Sports,True,True,[ e ][ h ] OP.GG Sports Team Information Locat...,None,main_team_refined
4,apexlegends,Apex Legends,High Quality,High Quality,https://liquipedia.net/apexlegends/High_Quality,True,True,[ e ][ h ] High Quality Team Information Locat...,None,main_team_refined
5,apexlegends,Apex Legends,Hybrid Eclipse Arise,Hybrid Eclipse Arise,https://liquipedia.net/apexlegends/Hybrid_Ecli...,True,True,[ e ][ h ] Hybrid Eclipse Arise Team Informati...,None,main_team_refined
6,apexlegends,Apex Legends,Ganbare otousan,ganbare otousan,https://liquipedia.net/apexlegends/Ganbare_oto...,True,True,[ e ][ h ] ganbare otousan Team Information Lo...,None,main_team_refined
7,apexlegends,Apex Legends,Predator,Predator,https://liquipedia.net/apexlegends/Predator,True,True,[ e ][ h ] Predator Team Information Location:...,None,main_team_refined
8,apexlegends,Apex Legends,Fun123,fun123,https://liquipedia.net/apexlegends/Fun123,True,True,[ e ][ h ] fun123 Team Information Location: S...,None,main_team_refined
9,apexlegends,Apex Legends,The Start,The Start,https://liquipedia.net/apexlegends/The_Start,True,True,[ e ][ h ] The Start Team Information Location...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,apexlegends,Apex Legends,Apex Legends,ENTER FORCE.36,https://liquipedia.net/apexlegends/ENTER_FORCE.36,Player Roster,Active,player_roster,Active,0,...,2024-01-28 [ 6 ] [ 8 ],,,,South Korea,Cinap,https://liquipedia.net/apexlegends/Cinap,Cinap Kim Sang-rok (김상록) 2024-01-28 [ 6 ] [ 8 ],ENTER FORCE.36,main_team_refined
1,apexlegends,Apex Legends,Apex Legends,ENTER FORCE.36,https://liquipedia.net/apexlegends/ENTER_FORCE.36,Player Roster,Active,player_roster,Active,0,...,2025-03-01 [ 10 ],,,,South Korea,Jusna,https://liquipedia.net/apexlegends/Jusna,Jusna Shin Yong-ju (신용주) 2025-03-01 [ 10 ],ENTER FORCE.36,main_team_refined
2,apexlegends,Apex Legends,Apex Legends,ENTER FORCE.36,https://liquipedia.net/apexlegends/ENTER_FORCE.36,Player Roster,Active,player_roster,Active,0,...,2026-03-20 [ 14 ],,,,South Korea,Obly,https://liquipedia.net/apexlegends/Obly,Obly Lim Jung-hyun (임정현) 2026-03-20 [ 14 ],ENTER FORCE.36,main_team_refined
3,apexlegends,Apex Legends,Apex Legends,ENTER FORCE.36,https://liquipedia.net/apexlegends/ENTER_FORCE.36,Player Roster,Active,player_roster,Active,0,...,2026-03-20 [ 14 ],,,,South Korea,Aimbot,https://liquipedia.net/apexlegends/Aimbot,Aimbot Park Ji-hoon (박지훈) Substitute 2026-03-2...,ENTER FORCE.36,main_team_refined
4,apexlegends,Apex Legends,Apex Legends,ENTER FORCE.36,https://liquipedia.net/apexlegends/ENTER_FORCE.36,Player Roster,Former,player_roster,Former,1,...,,2026-02-05 [ 11 ],2026-02-26 [ 12 ] [ 13 ],Fnatic,South Korea,ILY,https://liquipedia.net/apexlegends/ILY,ILY Choi Jun-hyeok (최준혁) 2023-03-22 [ 3 ] 2026...,ENTER FORCE.36,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,apexlegends,Apex Legends,Apex Legends,Cyma,https://liquipedia.net/apexlegends/Cyma,Player Roster,Former,player_roster,Former,0,...,,,2020-04-27 [ 3 ],OP.GG Sports,South Korea,Minseong,https://liquipedia.net/apexlegends/Minseong,Minseong Jung Min-sung 2019-11-01 [ 1 ] 2020-0...,Cyma,main_team_refined
96,apexlegends,Apex Legends,Apex Legends,Cyma,https://liquipedia.net/apexlegends/Cyma,Player Roster,Former,player_roster,Former,0,...,,,2020-04-27 [ 3 ],OP.GG Sports,South Korea,Dogma,https://liquipedia.net/apexlegends/Dogma,Dogma Kim Jeong-jin 2019-11-01 [ 1 ] 2020-04-2...,Cyma,main_team_refined
97,apexlegends,Apex Legends,Apex Legends,Cyma,https://liquipedia.net/apexlegends/Cyma,Player Roster,Former,player_roster,Former,0,...,,,2020-04-04 [ 2 ],High Quality,South Korea,Ashes,https://liquipedia.net/apexlegends/Ashes,Ashes Kim Hyun-Soo 2019-11-01 [ 1 ] 2020-04-04...,Cyma,main_team_refined
98,apexlegends,Apex Legends,Apex Legends,Cyma,https://liquipedia.net/apexlegends/Cyma,Organization,Former,organization,Former,1,...,,,,Retired,South Korea,index.php?title=Head&action=edit&redlink=1,https://liquipedia.net/apexlegends/index.php?t...,head Woo Gil Park Head Coach Retired,Cyma,main_team_refined


In [97]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [98]:
apex_people_unique_df = make_unique_people_latest_from_raw(apex_people_raw_df)

print("raw row 수:", len(apex_people_raw_df))
print("최신 기준 unique 인원 수:", len(apex_people_unique_df))

display(apex_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 243
최신 기준 unique 인원 수: 145


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Apex Legends,ENTER FORCE.36,Active,aWesomeguy,Kim Sung-hoon (김성훈),Coach,2022-06-17 [ 1 ],,,,South Korea,https://liquipedia.net/apexlegends/AWesomeguy
1,Apex Legends,ENTER FORCE.36,Former,ahn2e,Kim Gun-ho (김건호),2022-06-17 [ 1 ],,,2023-03-06 [ 2 ],REJECT (Coach),South Korea,https://liquipedia.net/apexlegends/Ahn2e
2,Apex Legends,ENTER FORCE.36,Active,Aimbot,Park Ji-hoon (박지훈),Substitute,2026-03-20 [ 14 ],,,,South Korea,https://liquipedia.net/apexlegends/Aimbot
3,Apex Legends,MVP,Former,Aka,Kim Suhwan (김수환),2019-04-08 [ 1 ],,,2019-10-17 [ 3 ],Retired,South Korea,https://liquipedia.net/apexlegends/Aka
4,Apex Legends,Chouette Gaming Korea,Former,Ashes,Kim Hyun-Soo (김현수),2021-05-15 [ 1 ],,,2021-06-30 [ 2 ],Sengoku Gaming,South Korea,https://liquipedia.net/apexlegends/Ashes
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Apex Legends,FOR7,Active,Hiroyuki Nakamura,Hiroyuki Nakamura,CEO,2020-??-??,,,,Japan,https://liquipedia.net/apexlegends/index.php?t...
96,Apex Legends,T1 Korea,Former,HyoO,Kim Hyo Young,Coach,2020-03-09 [ 3 ],,2020-03-24 [ 4 ],,South Korea,https://liquipedia.net/apexlegends/index.php?t...
97,Apex Legends,High Quality,Stand-in,IKEMAN,,T1 North American League: Season 3,,,,,South Korea,https://liquipedia.net/apexlegends/index.php?t...
98,Apex Legends,Meta Gaming,Former,iloy,Nam Hoon Kim,Manager,2017-??-??,,,,South Korea,https://liquipedia.net/apexlegends/index.php?t...


In [99]:
apex_dedup_df2.to_csv(
    "/content/apexlegends_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

apex_teams_checked_df.to_csv(
    "/content/apexlegends_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

apex_teams_final_df.to_csv(
    "/content/apexlegends_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

apex_people_raw_df.to_csv(
    "/content/apexlegends_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

apex_people_unique_df.to_csv(
    "/content/apexlegends_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

apex_errors_df.to_csv(
    "/content/apexlegends_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/apexlegends_teams_final_v4.csv")
print("/content/apexlegends_people_unique_v4.csv")

저장 완료
/content/apexlegends_teams_final_v4.csv
/content/apexlegends_people_unique_v4.csv


#fc

In [100]:
# =========================================
# EA SPORTS FC / FC Online Liquipedia 설정
# =========================================

GAME_SLUG = "easportsfc"
GAME_LABEL = "EA SPORTS FC Online"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"

# 반드시 본인 이메일/프로젝트명으로 변경
USER_AGENT = "KoreanEASportsFCOnlineEsportsResearch/0.1 (your-email@example.com)"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept-Encoding": "gzip"
}

REQUEST_SLEEP = 2.2
PARSE_SLEEP = 31

print(BASE_URL)
print(API_URL)

https://liquipedia.net/easportsfc
https://liquipedia.net/easportsfc/api.php


In [101]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    EA SPORTS FC Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [102]:
EASPORTSFC_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",

    "FC Online Korea",
    "FC Online Korean",
    "EA SPORTS FC Online Korea",
    "EA SPORTS FC Online Korean",
    "FCO Korea",
    "FCO Korean",
    "FC Pro Korea",
    "FSL Korea",

    # 한국 팀 후보
    "Dplus",
    "Dplus KIA",
    "Dplus Gaming",
    "DWG KIA",
    "Damwon Gaming",
    "Gen.G Esports",
    "GEN CITY",
    "DRX",
    "Kwangdong Freecs",
    "Afreeca Freecs",
    "KT Rolster",
    "T1",
    "Nongshim RedForce",
    "FearX",
    "BNK FearX",
    "AJ Esports",
]

In [103]:
search_rows = []

for keyword in EASPORTSFC_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

eafc_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(eafc_search_df))
display(eafc_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: Korean / offset=0
검색: Korean / offset=50
검색: Korean / offset=100
검색: Korean / offset=150
검색: Korean / offset=200
검색: South Korean / offset=0
검색: South Korean / offset=50
검색: South Korean / offset=100
검색: South Korean / offset=150
검색: Korean team / offset=0
검색: Korean team / offset=50
검색: Korean team / offset=100
검색: Korean team / offset=150
검색: Korea esports / offset=0
검색: Korea esports / offset=50
검색: Korea esports / offset=100
검색: FC Online Korea / offset=0
검색: FC Online Korea / offset=50
검색: FC Online Korea / offset=100
검색: FC Online Korean / offset=0
검색: FC Online Korean / offset=50
검색: FC Online Korean / offset=100
검색: EA SPORTS FC Online Korea / offset=0
검색: EA SPORTS FC Online Korea / offset=50
검색: EA SPORTS FC Online Korean / offset=0
검색: EA SPORTS FC O

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,easportsfc,EA SPORTS FC Online,B-Tier Tournaments,"$36,866 South Korea 9 Daejeon Hana Citizen FC ...",2031,https://liquipedia.net/easportsfc/B-Tier_Tourn...,korea
1,easportsfc,EA SPORTS FC Online,Freecs,Team Information Location: South Korea Approx....,13038,https://liquipedia.net/easportsfc/Freecs,korea
2,easportsfc,EA SPORTS FC Online,KT Rolster,Information Location: South Korea Approx. Tota...,11247,https://liquipedia.net/easportsfc/KT_Rolster,korea
3,easportsfc,EA SPORTS FC Online,Serry7,Player Information Name: 나호철 Romanized Name: N...,7774,https://liquipedia.net/easportsfc/Serry7,korea
4,easportsfc,EA SPORTS FC Online,KWAK,Information Name: 곽준혁 Romanized Name: Kwak Jun...,11150,https://liquipedia.net/easportsfc/KWAK,korea
...,...,...,...,...,...,...,...
95,easportsfc,EA SPORTS FC Online,FC Pro Masters/2025/Online,Phở (Huy Phước) Chinese Commentators: Kaka Kin...,14206,https://liquipedia.net/easportsfc/FC_Pro_Maste...,korea
96,easportsfc,EA SPORTS FC Online,EK League/2023/Spring,eK League Championship 2023 Spring Road to EAC...,8905,https://liquipedia.net/easportsfc/EK_League/20...,korea
97,easportsfc,EA SPORTS FC Online,FCO Super Champions League/Team Battle/2025/Su...,FCO Super Champions League Team Battle 2025 Su...,14626,https://liquipedia.net/easportsfc/FCO_Super_Ch...,korea
98,easportsfc,EA SPORTS FC Online,Spearhead Invitational/2014,South Korea Thailand Vietnam Seeding Matches T...,6390,https://liquipedia.net/easportsfc/Spearhead_In...,korea


In [104]:
def build_eafc_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "fc online",
        "fc pro",
        "fco",
        "fsl",

        "dplus",
        "dplus kia",
        "dwg",
        "damwon",
        "gen.g",
        "gen city",
        "drx",
        "kwangdong",
        "freecs",
        "afreeca",
        "kt rolster",
        "t1",
        "nongshim",
        "redforce",
        "fearx",
        "aj esports",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "stage",
        "major",
        "minor",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "rankings",
        "awards",
        "finals",
        "playoffs",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "region: korea",
            "region: asia",
            "south korea",
            "korea",
            "korean",
            "korean teams",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("EA SPORTS FC Online 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [105]:
eafc_team_candidate_loose_df = build_eafc_team_candidate_loose_df(eafc_search_df)

display(eafc_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

EA SPORTS FC Online 완화 후보 수: 22
예상 parse 소요 시간: 11.4 분
예상 parse 소요 시간: 0.19 시간


,title,snippet,search_keyword,url
0,Freecs,Team Information Location: South Korea Approx....,korea,https://liquipedia.net/easportsfc/Freecs
1,KT Rolster,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/KT_Rolster
2,KWAK,Information Name: 곽준혁 Romanized Name: Kwak Jun...,korea,https://liquipedia.net/easportsfc/KWAK
3,Crazy Win,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/Crazy_Win
4,Gen.G Esports,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/Gen.G_Esports
5,CHAN,Information Name: 박찬화 Romanized Name: Park Cha...,korea,https://liquipedia.net/easportsfc/CHAN
6,FEARX,Rift Teams for Efficient Operations] (in Korea...,korea,https://liquipedia.net/easportsfc/FEARX
7,WHGaming,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/WHGaming
8,GWAN,Information Name: 김관형 Romanized Name: Kim Kwan...,korea,https://liquipedia.net/easportsfc/GWAN
9,Tofu,Information Name: 박기영 Romanized Name: Park Gi-...,korea,https://liquipedia.net/easportsfc/Tofu


In [106]:
# =========================================
# EA SPORTS FC Online 완화 후보에서 팀 후보만 재정제
# output: eafc_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = eafc_team_candidate_loose_df.copy()

eafc_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "fc online",
    "fc pro",
    "fco",
    "fsl",

    "dplus",
    "dplus kia",
    "dwg",
    "damwon",
    "gen.g",
    "gen city",
    "drx",
    "kwangdong",
    "freecs",
    "afreeca",
    "kt rolster",
    "t1",
    "nongshim",
    "redforce",
    "fearx",
    "aj esports",
]

eafc_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "stage",
    "major",
    "minor",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "rankings",
    "awards",
    "finals",
    "playoffs",
]

eafc_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

eafc_korea_terms = [
    "location: south korea",
    "region: korea",
    "region: asia",
    "south korea",
    "korea",
    "korean",
    "korean teams",
]


def is_refined_eafc_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in eafc_bad_page_markers):
        return False

    for pattern in eafc_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in eafc_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in eafc_team_name_keywords):
        return True

    return False


eafc_dedup_df2 = source_df[
    source_df.apply(is_refined_eafc_team_candidate, axis=1)
].copy().reset_index(drop=True)

eafc_dedup_df2["title_clean"] = (
    eafc_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

eafc_dedup_df2 = eafc_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("EA SPORTS FC Online 완화 후보 수:", len(eafc_team_candidate_loose_df))
print("재정제 후 eafc_dedup_df2 후보 수:", len(eafc_dedup_df2))
print("예상 parse 소요 시간:", round(len(eafc_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(eafc_dedup_df2) * 31 / 3600, 2), "시간")

display(eafc_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

EA SPORTS FC Online 완화 후보 수: 22
재정제 후 eafc_dedup_df2 후보 수: 22
예상 parse 소요 시간: 11.4 분
예상 parse 소요 시간: 0.19 시간


,title,snippet,search_keyword,url
0,Freecs,Team Information Location: South Korea Approx....,korea,https://liquipedia.net/easportsfc/Freecs
1,KT Rolster,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/KT_Rolster
2,KWAK,Information Name: 곽준혁 Romanized Name: Kwak Jun...,korea,https://liquipedia.net/easportsfc/KWAK
3,Crazy Win,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/Crazy_Win
4,Gen.G Esports,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/Gen.G_Esports
5,CHAN,Information Name: 박찬화 Romanized Name: Park Cha...,korea,https://liquipedia.net/easportsfc/CHAN
6,FEARX,Rift Teams for Efficient Operations] (in Korea...,korea,https://liquipedia.net/easportsfc/FEARX
7,WHGaming,Information Location: South Korea Approx. Tota...,korea,https://liquipedia.net/easportsfc/WHGaming
8,GWAN,Information Name: 김관형 Romanized Name: Kim Kwan...,korea,https://liquipedia.net/easportsfc/GWAN
9,Tofu,Information Name: 박기영 Romanized Name: Park Gi-...,korea,https://liquipedia.net/easportsfc/Tofu


In [107]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [108]:
def extract_eafc_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "ea sports fc", "fc online", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Statistics",
        "Awards",
        "Logos",
        "Media",
        "References",
        "Gallery",
        "Upcoming Matches",
        "Upcoming Tournaments",
        "Timeline",
        "Achievements",
        "Recent Matches",
        "Individual Achievements",
        "Team Achievements",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_first_country_from_row(row_el):
        for img in row_el.select("img"):
            for attr in ["alt", "title"]:
                value = clean_text(img.get(attr))
                if not value:
                    continue
                if value.lower().endswith((".png", ".svg", ".jpg", ".webp")):
                    continue
                if value.lower() in ["edit", "view", "history"]:
                    continue
                return value
        return ""

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h == "name":
            return "name"
        if h in ["role", "position"]:
            return "position"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|stage|major|minor|playoffs|championship|cup|tournament|league|series|qualifier|finals|fc pro|fsl|fco)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        if "player roster" in main_low or "former players" in sub_low or "players" in sub_low:
            if "stand" in sub_low:
                return "player_roster", "Stand-in", main_section
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        row_pos = row_data.get("position", "")

        if table_type == "organization":
            if row_pos:
                return row_pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if row_pos:
                return row_pos
            if status == "Stand-in":
                return "Stand-in"
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "position" in lowered
                or "role" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if len(normalized_headers) >= 4 and "position" not in normalized_headers:
            if normalized_headers[0] == "id" and normalized_headers[1] == "name":
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}
            local_headers = normalized_headers.copy()

            if len(cells) == len(local_headers) + 1:
                if "position" not in local_headers and "join_date" in local_headers:
                    join_idx = local_headers.index("join_date")
                    local_headers.insert(join_idx, "position")

            for idx, cell in enumerate(cells):
                if idx >= len(local_headers):
                    continue

                col = local_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []

            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality = (
                row_data.get("nationality", "")
                or extract_first_country_from_row(tr)
            )

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "EA SPORTS FC Online",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [109]:
test_team = "Dplus"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_eafc_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "table_type",
            "status",
            "id",
            "name",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: Dplus
페이지 존재 여부: True
에러: None
추출 row 수: 5


,team_title,game_title,table_type,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Dplus,EA SPORTS FC Online,player_roster,Active,Clutch,,Player,2025-02-21 [ 4 ],,,,South Korea,https://liquipedia.net/easportsfc/index.php?ti...
1,Dplus,EA SPORTS FC Online,player_roster,Active,Exito,,Player,2025-02-21 [ 4 ],,,,South Korea,https://liquipedia.net/easportsfc/index.php?ti...
2,Dplus,EA SPORTS FC Online,player_roster,Active,JubJub,Phatanasak Varanan,Player,2025-02-21 [ 4 ],,,,Thailand,https://liquipedia.net/easportsfc/JubJub
3,Dplus,EA SPORTS FC Online,player_roster,Active,KWAK,Kwak Jun-hyouk,Player,2025-02-21 [ 4 ],,,,South Korea,https://liquipedia.net/easportsfc/KWAK
4,Dplus,EA SPORTS FC Online,player_roster,Active,MiBOB,,Player,2025-02-21 [ 4 ],,,,South Korea,https://liquipedia.net/easportsfc/index.php?ti...


In [110]:
def crawl_eafc_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        # easportsfc는 Region이 Korea가 아니라 Asia로 나오는 경우가 있어서
        # Location: South Korea 또는 Category: Korean Teams까지 함께 봄
        full_text_lower = clean_text(BeautifulSoup(page["html"], "lxml").get_text(" ", strip=True)).lower()

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "region:korea" in infobox_compact
                or "region: korea" in infobox_lower
                or "south korea" in infobox_lower
                or "korean teams" in full_text_lower
                or "category:korean teams" in full_text_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_eafc_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 raw row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("자동 저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [111]:
eafc_teams_checked_df, eafc_teams_final_df, eafc_people_raw_df, eafc_errors_df = crawl_eafc_teams_and_people_v4(
    candidate_df=eafc_dedup_df2,
    output_prefix="easportsfc_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(eafc_teams_final_df.head(100))
display(eafc_people_raw_df.head(100))

크롤링 대상 후보 수: 22
예상 소요 시간: 11.4 분
예상 소요 시간: 0.19 시간


easportsfc_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/22 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 13
추출 인원 raw row 수: 43
에러 수: 0
자동 저장 완료
/content/easportsfc_korean_mainteam_refined_v4_teams_checked.csv
/content/easportsfc_korean_mainteam_refined_v4_teams_final.csv
/content/easportsfc_korean_mainteam_refined_v4_people_raw.csv
/content/easportsfc_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,easportsfc,EA SPORTS FC Online,Freecs,Freecs,https://liquipedia.net/easportsfc/Freecs,True,True,[ e ][ h ] Freecs Team Information Location: S...,None,main_team_refined
1,easportsfc,EA SPORTS FC Online,KT Rolster,KT Rolster,https://liquipedia.net/easportsfc/KT_Rolster,True,True,[ e ][ h ] KT Rolster Team Information Locatio...,None,main_team_refined
2,easportsfc,EA SPORTS FC Online,Crazy Win,Crazy Win,https://liquipedia.net/easportsfc/Crazy_Win,True,True,[ e ][ h ] Crazy Win Team Information Location...,None,main_team_refined
3,easportsfc,EA SPORTS FC Online,Gen.G Esports,GEN CITY,https://liquipedia.net/easportsfc/Gen.G_Esports,True,True,[ e ][ h ] GEN CITY Team Information Location:...,None,main_team_refined
4,easportsfc,EA SPORTS FC Online,FEARX,FEARX,https://liquipedia.net/easportsfc/FEARX,True,True,[ e ][ h ] FearX Team Information Location: So...,None,main_team_refined
5,easportsfc,EA SPORTS FC Online,WHGaming,WHGaming,https://liquipedia.net/easportsfc/WHGaming,True,True,[ e ][ h ] WHGaming Team Information Location:...,None,main_team_refined
6,easportsfc,EA SPORTS FC Online,FN Esports,FN Esports,https://liquipedia.net/easportsfc/FN_Esports,True,True,[ e ][ h ] FN Esports Team Information Locatio...,None,main_team_refined
7,easportsfc,EA SPORTS FC Online,T1,T1,https://liquipedia.net/easportsfc/T1,True,True,[ e ][ h ] T1 Team Information Location: South...,None,main_team_refined
8,easportsfc,EA SPORTS FC Online,Nongshim RedForce,Nongshim RedForce,https://liquipedia.net/easportsfc/Nongshim_Red...,True,True,[ e ][ h ] Nongshim RedForce Team Information ...,None,main_team_refined
9,easportsfc,EA SPORTS FC Online,Dplus,Dplus,https://liquipedia.net/easportsfc/Dplus,True,True,[ e ][ h ] Dplus Team Information Location: So...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,Freecs,https://liquipedia.net/easportsfc/Freecs,Player Roster,Active,player_roster,Active,5,...,2024-01-22 [ 6 ],,,,South Korea,Sikyung,https://liquipedia.net/easportsfc/Sikyung,9KKI Kim Si-kyung 2024-01-22 [ 6 ],Freecs,main_team_refined
1,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,Freecs,https://liquipedia.net/easportsfc/Freecs,Player Roster,Former,player_roster,Former,6,...,2021-12-06 [ 3 ],,2023-06-18 [ 5 ],,South Korea,index.php?title=Jung Sung-min&action=edit&redl...,https://liquipedia.net/easportsfc/index.php?ti...,Jung Sung-min 2021-12-06 [ 3 ] 2023-06-18 [ 5 ],Freecs,main_team_refined
2,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,Freecs,https://liquipedia.net/easportsfc/Freecs,Player Roster,Former,player_roster,Former,7,...,,,2025-02-27 [ 7 ],T1,South Korea,Byul,https://liquipedia.net/easportsfc/Byul,Byul Park Ki-hong 2022-10-17 [ 4 ] 2025-02-27 ...,Freecs,main_team_refined
3,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,KT Rolster,https://liquipedia.net/easportsfc/KT_Rolster,Player Roster,Active,player_roster,Active,5,...,2023-03-01 [ 2 ],,,,South Korea,CHAN,https://liquipedia.net/easportsfc/CHAN,CHAN Park Chan-hwa 2023-03-01 [ 2 ],KT Rolster,main_team_refined
4,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,KT Rolster,https://liquipedia.net/easportsfc/KT_Rolster,Player Roster,Active,player_roster,Active,5,...,2023-03-01 [ 2 ],,,,South Korea,GWAN,https://liquipedia.net/easportsfc/GWAN,GWAN Kim Kwan-hyung 2023-03-01 [ 2 ],KT Rolster,main_team_refined
5,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,KT Rolster,https://liquipedia.net/easportsfc/KT_Rolster,Player Roster,Active,player_roster,Active,5,...,2023-03-01 [ 2 ],,,,South Korea,JungMin,https://liquipedia.net/easportsfc/JungMin,JungMin Kim Jung-min 2023-03-01 [ 2 ],KT Rolster,main_team_refined
6,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,KT Rolster,https://liquipedia.net/easportsfc/KT_Rolster,Player Roster,Former,player_roster,Former,6,...,,,2025-02-21 [ 3 ],Dplus,South Korea,KWAK,https://liquipedia.net/easportsfc/KWAK,KWAK Kwak Jun-hyouk 2023-03-01 [ 2 ] 2025-02-2...,KT Rolster,main_team_refined
7,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,Crazy Win,https://liquipedia.net/easportsfc/Crazy_Win,Player Roster,Former,player_roster,Former,5,...,2020-08-01 [ 1 ],,2021-02-15 [ 2 ],Saddler,South Korea,index.php?title=Jung Sung-min&action=edit&redl...,https://liquipedia.net/easportsfc/index.php?ti...,Jung Sung-min 2020-08-01 [ 1 ] 2021-02-15 [ 2 ...,Crazy Win,main_team_refined
8,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,Crazy Win,https://liquipedia.net/easportsfc/Crazy_Win,Player Roster,Former,player_roster,Former,6,...,,,2022-08-01 [ 6 ],PLAYGROUND,South Korea,CHAN,https://liquipedia.net/easportsfc/CHAN,CHAN Park Chan-hwa 2022-05-01 [ 4 ] 2022-08-01...,Crazy Win,main_team_refined
9,easportsfc,EA SPORTS FC Online,EA SPORTS FC Online,Crazy Win,https://liquipedia.net/easportsfc/Crazy_Win,Player Roster,Former,player_roster,Former,6,...,,,2022-08-01 [ 6 ],,South Korea,index.php?title=Choi Joon-ho&action=edit&redli...,https://liquipedia.net/easportsfc/index.php?ti...,Choi Joon-ho 2020-08-01 [ 1 ] 2022-08-01 [ 6 ],Crazy Win,main_team_refined


In [112]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [113]:
eafc_people_unique_df = make_unique_people_latest_from_raw(eafc_people_raw_df)

print("raw row 수:", len(eafc_people_raw_df))
print("최신 기준 unique 인원 수:", len(eafc_people_unique_df))

display(eafc_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

raw row 수: 43
최신 기준 unique 인원 수: 33


,game_title,team_title,status,id,name,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,EA SPORTS FC Online,T1,Active,Byul,Park Ki-hong,Player,2025-02-27 [ 4 ],,,,South Korea,https://liquipedia.net/easportsfc/Byul
1,EA SPORTS FC Online,KT Rolster,Active,CHAN,Park Chan-hwa,Player,2023-03-01 [ 2 ],,,,South Korea,https://liquipedia.net/easportsfc/CHAN
2,EA SPORTS FC Online,Gen.G Esports,Active,Crong,Hwang Se-jong,Player,2024-05-07 [ 6 ] [ 7 ],,,,South Korea,https://liquipedia.net/easportsfc/Crong
3,EA SPORTS FC Online,PREP eSPORTS,Former,DwMartin,Daewon Kwon,2022-08-03 [ 2 ],,,2022-10-25 [ 3 ],,South Korea,https://liquipedia.net/easportsfc/DwMartin
4,EA SPORTS FC Online,FN Esports,Active,Eggsy,Ega Rahmaditya,Player,2024-11-15 [ 2 ],,,,Indonesia,https://liquipedia.net/easportsfc/Eggsy
5,EA SPORTS FC Online,KT Rolster,Active,GWAN,Kim Kwan-hyung,Player,2023-03-01 [ 2 ],,,,South Korea,https://liquipedia.net/easportsfc/GWAN
6,EA SPORTS FC Online,PREP eSPORTS,Active,Hojin,Hojin Chun,Player,2022-08-03 [ 2 ],,,,South Korea,https://liquipedia.net/easportsfc/Hojin
7,EA SPORTS FC Online,T1,Active,Hoseok,Choi Ho-suk,Player,2025-02-27 [ 4 ],,,,South Korea,https://liquipedia.net/easportsfc/Hoseok
8,EA SPORTS FC Online,Dplus,Active,JubJub,Phatanasak Varanan,Player,2025-02-21 [ 4 ],,,,Thailand,https://liquipedia.net/easportsfc/JubJub
9,EA SPORTS FC Online,KT Rolster,Active,JungMin,Kim Jung-min,Player,2023-03-01 [ 2 ],,,,South Korea,https://liquipedia.net/easportsfc/JungMin


In [114]:
eafc_dedup_df2.to_csv(
    "/content/easportsfc_team_candidates_refined.csv",
    index=False,
    encoding="utf-8-sig"
)

eafc_teams_checked_df.to_csv(
    "/content/easportsfc_teams_checked_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

eafc_teams_final_df.to_csv(
    "/content/easportsfc_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

eafc_people_raw_df.to_csv(
    "/content/easportsfc_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

eafc_people_unique_df.to_csv(
    "/content/easportsfc_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

eafc_errors_df.to_csv(
    "/content/easportsfc_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/easportsfc_teams_final_v4.csv")
print("/content/easportsfc_people_raw_v4.csv")
print("/content/easportsfc_people_unique_v4.csv")

저장 완료
/content/easportsfc_teams_final_v4.csv
/content/easportsfc_people_raw_v4.csv
/content/easportsfc_people_unique_v4.csv


#warcraft

In [115]:
GAME_SLUG = "warcraft"
GAME_LABEL = "Warcraft III"

BASE_URL = f"https://liquipedia.net/{GAME_SLUG}"
API_URL = f"{BASE_URL}/api.php"


In [116]:
def search_pages_all(keyword, limit=50, max_total=2000):
    """
    Warcraft Liquipedia Main namespace 검색 결과 전체 수집.
    페이지 본문 크롤링 X, 검색 API만 사용.
    """
    all_rows = []
    offset = 0

    while True:
        print(f"검색: {keyword} / offset={offset}")

        data = api_get({
            "action": "query",
            "list": "search",
            "srsearch": keyword,
            "srnamespace": 0,
            "srlimit": limit,
            "sroffset": offset,
        })

        results = data.get("query", {}).get("search", [])

        if not results:
            break

        for item in results:
            title = item.get("title", "")
            all_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "snippet": clean_text(re.sub("<.*?>", "", item.get("snippet", ""))),
                "pageid": item.get("pageid"),
                "url": make_page_url(title),
                "search_keyword": keyword
            })

        offset += limit

        if len(all_rows) >= max_total:
            print("max_total 도달:", max_total)
            break

    return all_rows

In [117]:
WARCRAFT_SEARCH_KEYWORDS = [
    "korea",
    "South Korea",
    "Korean",
    "South Korean",
    "Korean team",
    "Korea esports",
    "Warcraft Korea",
    "Warcraft III Korea",
    "Warcraft III Korean",
    "WC3 Korea",
    "WC3 Korean",

    # 한국 팀 후보
    "DN SOOPers",
    "DN Freecs",
    "Afreeca Freecs",
    "DRX",
    "Phantom Aces",
    "Motaesolo",
    "Kings of Azeroth order",
    "Wandering Dragons",
    "JUCON Entertainment",
    "Rocket Beans",
]

In [118]:
search_rows = []

for keyword in WARCRAFT_SEARCH_KEYWORDS:
    rows = search_pages_all(keyword, limit=50, max_total=1000)
    search_rows.extend(rows)

warcraft_search_df = (
    pd.DataFrame(search_rows)
    .drop_duplicates(subset=["title"])
    .reset_index(drop=True)
)

print("검색 후보 수:", len(warcraft_search_df))
display(warcraft_search_df.head(100))

검색: korea / offset=0
검색: korea / offset=50
검색: korea / offset=100
검색: korea / offset=150
검색: korea / offset=200
검색: korea / offset=250
검색: korea / offset=300
검색: korea / offset=350
검색: korea / offset=400
검색: korea / offset=450
검색: korea / offset=500
검색: korea / offset=550
검색: korea / offset=600
검색: korea / offset=650
검색: korea / offset=700
검색: korea / offset=750
검색: korea / offset=800
검색: korea / offset=850
검색: korea / offset=900
검색: korea / offset=950
max_total 도달: 1000
검색: South Korea / offset=0
검색: South Korea / offset=50
검색: South Korea / offset=100
검색: South Korea / offset=150
검색: South Korea / offset=200
검색: South Korea / offset=250
검색: South Korea / offset=300
검색: South Korea / offset=350
검색: South Korea / offset=400
검색: South Korea / offset=450
검색: South Korea / offset=500
검색: South Korea / offset=550
검색: South Korea / offset=600
검색: South Korea / offset=650
검색: South Korea / offset=700
검색: South Korea / offset=750
검색: South Korea / offset=800
검색: South Korea / offset=850
검색: S

,source_wiki,game_label,title,snippet,pageid,url,search_keyword
0,warcraft,Warcraft III,Korea,Korea Map Information Creator: Alexey Baryjiko...,24571,https://liquipedia.net/warcraft/Korea,korea
1,warcraft,Warcraft III,Tier 3 Tournaments,$840.93 South Korea 4 participants FoCuS LuCha...,5636,https://liquipedia.net/warcraft/Tier_3_Tournam...,korea
2,warcraft,Warcraft III,Tier 2 Tournaments,4 participants Fortitude Lyn Warcraft Survival...,5635,https://liquipedia.net/warcraft/Tier_2_Tournam...,korea
3,warcraft,Warcraft III,Weekly Tournaments,"Cup 10 Nov 07, 2019 $86.23 South Korea 19 part...",8836,https://liquipedia.net/warcraft/Weekly_Tournam...,korea
4,warcraft,Warcraft III,Qualifier Tournaments,2011 South Korea Qoo kimkihyun TBD World Cyber...,10666,https://liquipedia.net/warcraft/Qualifier_Tour...,korea
...,...,...,...,...,...,...,...
95,warcraft,Warcraft III,ReiGn,Player Information Name: 강서우 Romanized Name: K...,3912,https://liquipedia.net/warcraft/ReiGn,korea
96,warcraft,Warcraft III,SoJu,Player Information Name: 이성덕 Romanized Name: L...,6185,https://liquipedia.net/warcraft/SoJu,korea
97,warcraft,Warcraft III,Battle.net,"East gateway, Northrend : the Europe gateway, ...",14226,https://liquipedia.net/warcraft/Battle.net,korea
98,warcraft,Warcraft III,China,1st Tier 1 Stars War X - China vs Korea 3 : 2 ...,7544,https://liquipedia.net/warcraft/China,korea


In [119]:
def build_warcraft_team_candidate_loose_df(search_df):
    team_like_keywords = [
        "team",
        "esports",
        "e-sports",
        "gaming",
        "organization",
        "clan",
        "squad",
        "warcraft",
        "warcraft iii",
        "wc3",

        "dn soopers",
        "dn freecs",
        "afreeca",
        "freecs",
        "drx",
        "phantom aces",
        "motaesolo",
        "kings of azeroth",
        "wandering dragons",
        "jucon",
        "rocket beans",
    ]

    bad_title_patterns = [
        "/",
        r"\b20\d{2}\b",
        "season",
        "stage",
        "major",
        "minor",
        "tournament",
        "championship",
        "qualifier",
        "results",
        "played matches",
        "matches",
        "match history",
        "statistics",
        "schedule",
        "broadcast",
        "vod",
        "interview",
        "standings",
        "bracket",
        "group stage",
        "participants",
        "rankings",
        "awards",
        "finals",
        "playoffs",
        "players_",
    ]

    bad_markers = [
        "player information",
        "tournament information",
        "league information",
        "match statistics",
        "broadcast talent",
    ]

    def is_candidate(row):
        title = str(row.get("title", ""))
        snippet = str(row.get("snippet", ""))
        text = f"{title} {snippet}".lower()

        if any(marker in text for marker in bad_markers):
            return False

        for pattern in bad_title_patterns:
            if re.search(pattern, title, flags=re.IGNORECASE):
                return False

        korea_terms = [
            "location: south korea",
            "south korea",
            "korea",
            "korean",
            "korean teams",
        ]

        has_korea = any(term in text for term in korea_terms)
        has_team_like = any(keyword in text for keyword in team_like_keywords)

        if "team information" in text and has_korea:
            return True

        if has_korea and has_team_like:
            return True

        return False

    df = search_df[
        search_df.apply(is_candidate, axis=1)
    ].copy().reset_index(drop=True)

    df["title_clean"] = (
        df["title"]
        .astype(str)
        .str.replace("_", " ", regex=False)
        .str.strip()
    )

    df = df.drop_duplicates(subset=["title_clean"]).reset_index(drop=True)

    print("Warcraft III 완화 후보 수:", len(df))
    print("예상 parse 소요 시간:", round(len(df) * 31 / 60, 1), "분")
    print("예상 parse 소요 시간:", round(len(df) * 31 / 3600, 2), "시간")

    return df

In [120]:
warcraft_team_candidate_loose_df = build_warcraft_team_candidate_loose_df(warcraft_search_df)

display(warcraft_team_candidate_loose_df[[
    "title",
    "snippet",
    "search_keyword",
    "url"
]].head(200))

Warcraft III 완화 후보 수: 73
예상 parse 소요 시간: 37.7 분
예상 parse 소요 시간: 0.63 시간


,title,snippet,search_keyword,url
0,All for One,Clan tag: A1 Location: South Korea Approx. Tot...,korea,https://liquipedia.net/warcraft/All_for_One
1,South Korea,South Korea Team Information Clan tag: KOR Loc...,korea,https://liquipedia.net/warcraft/South_Korea
2,DRX,Clan tag: DRX Location: South Korea Approx. To...,korea,https://liquipedia.net/warcraft/DRX
3,DN SOOPers,Location: South Korea Approx. Total Winnings: ...,korea,https://liquipedia.net/warcraft/DN_SOOPers
4,ANGRY KOREA MAN,nickname after meeting several raging opponent...,korea,https://liquipedia.net/warcraft/ANGRY_KOREA_MAN
...,...,...,...,...
68,SiL Cup,Meadows The SiL Cup is an open cup organized b...,Korean team,https://liquipedia.net/warcraft/SiL_Cup
69,Rest of World Players,Players ID Real Name Team Links ANGRY_KOREA_MA...,Korean team,https://liquipedia.net/warcraft/Rest_of_World_...
70,Warcraft III Invitational,player FFA match - Won by HawK 4v4 Europe vs. ...,Korean team,https://liquipedia.net/warcraft/Warcraft_III_I...
71,Seer Cup,esportsearnings. Complete results. Playlist wi...,Warcraft Korea,https://liquipedia.net/warcraft/Seer_Cup


In [121]:
# =========================================
# Warcraft III 완화 후보에서 팀 후보만 재정제
# output: warcraft_dedup_df2
# 페이지 불러오기 X
# =========================================

source_df = warcraft_team_candidate_loose_df.copy()

warcraft_team_name_keywords = [
    "team",
    "esports",
    "e-sports",
    "gaming",
    "organization",
    "clan",
    "squad",
    "warcraft",
    "warcraft iii",
    "wc3",

    "dn soopers",
    "dn freecs",
    "afreeca",
    "freecs",
    "drx",
    "phantom aces",
    "motaesolo",
    "kings of azeroth",
    "wandering dragons",
    "jucon",
    "rocket beans",
]

warcraft_bad_title_patterns = [
    "/",
    r"\b20\d{2}\b",
    "season",
    "stage",
    "major",
    "minor",
    "tournament",
    "championship",
    "qualifier",
    "results",
    "played matches",
    "matches",
    "match history",
    "statistics",
    "schedule",
    "broadcast",
    "vod",
    "interview",
    "standings",
    "bracket",
    "group stage",
    "participants",
    "rankings",
    "awards",
    "finals",
    "playoffs",
    "players_",
]

warcraft_bad_page_markers = [
    "player information",
    "tournament information",
    "league information",
    "match statistics",
    "broadcast talent",
]

warcraft_korea_terms = [
    "location: south korea",
    "south korea",
    "korea",
    "korean",
    "korean teams",
]


def is_refined_warcraft_team_candidate(row):
    title = str(row.get("title", ""))
    snippet = str(row.get("snippet", ""))
    text = f"{title} {snippet}".lower()

    if any(marker in text for marker in warcraft_bad_page_markers):
        return False

    for pattern in warcraft_bad_title_patterns:
        if re.search(pattern, title, flags=re.IGNORECASE):
            return False

    if not any(term in text for term in warcraft_korea_terms):
        return False

    if "team information" in text:
        return True

    if any(keyword in text for keyword in warcraft_team_name_keywords):
        return True

    return False


warcraft_dedup_df2 = source_df[
    source_df.apply(is_refined_warcraft_team_candidate, axis=1)
].copy().reset_index(drop=True)

warcraft_dedup_df2["title_clean"] = (
    warcraft_dedup_df2["title"]
    .astype(str)
    .str.replace("_", " ", regex=False)
    .str.strip()
)

warcraft_dedup_df2 = warcraft_dedup_df2.drop_duplicates(
    subset=["title_clean"]
).reset_index(drop=True)

print("Warcraft III 완화 후보 수:", len(warcraft_team_candidate_loose_df))
print("재정제 후 warcraft_dedup_df2 후보 수:", len(warcraft_dedup_df2))
print("예상 parse 소요 시간:", round(len(warcraft_dedup_df2) * 31 / 60, 1), "분")
print("예상 parse 소요 시간:", round(len(warcraft_dedup_df2) * 31 / 3600, 2), "시간")

display(warcraft_dedup_df2[["title", "snippet", "search_keyword", "url"]].head(200))

Warcraft III 완화 후보 수: 73
재정제 후 warcraft_dedup_df2 후보 수: 73
예상 parse 소요 시간: 37.7 분
예상 parse 소요 시간: 0.63 시간


,title,snippet,search_keyword,url
0,All for One,Clan tag: A1 Location: South Korea Approx. Tot...,korea,https://liquipedia.net/warcraft/All_for_One
1,South Korea,South Korea Team Information Clan tag: KOR Loc...,korea,https://liquipedia.net/warcraft/South_Korea
2,DRX,Clan tag: DRX Location: South Korea Approx. To...,korea,https://liquipedia.net/warcraft/DRX
3,DN SOOPers,Location: South Korea Approx. Total Winnings: ...,korea,https://liquipedia.net/warcraft/DN_SOOPers
4,ANGRY KOREA MAN,nickname after meeting several raging opponent...,korea,https://liquipedia.net/warcraft/ANGRY_KOREA_MAN
...,...,...,...,...
68,SiL Cup,Meadows The SiL Cup is an open cup organized b...,Korean team,https://liquipedia.net/warcraft/SiL_Cup
69,Rest of World Players,Players ID Real Name Team Links ANGRY_KOREA_MA...,Korean team,https://liquipedia.net/warcraft/Rest_of_World_...
70,Warcraft III Invitational,player FFA match - Won by HawK 4v4 Europe vs. ...,Korean team,https://liquipedia.net/warcraft/Warcraft_III_I...
71,Seer Cup,esportsearnings. Complete results. Playlist wi...,Warcraft Korea,https://liquipedia.net/warcraft/Seer_Cup


In [122]:
def get_page_html(title):
    try:
        data = api_get({
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|categories|links"
        }, sleep=PARSE_SLEEP)

        parsed = data.get("parse", {})
        html = parsed.get("text", {}).get("*", "")
        display_title = parsed.get("displaytitle", title)

        categories = [
            c.get("*", "")
            for c in parsed.get("categories", [])
        ]

        return {
            "title": title,
            "display_title": clean_text(BeautifulSoup(display_title, "html.parser").get_text(" ", strip=True)),
            "html": html,
            "categories": categories,
            "url": make_page_url(title),
            "exists": True,
            "error": None
        }

    except Exception as e:
        return {
            "title": title,
            "display_title": title,
            "html": "",
            "categories": [],
            "url": make_page_url(title),
            "exists": False,
            "error": str(e)
        }


def extract_infobox_text(html):
    soup = BeautifulSoup(html, "lxml")

    selectors = [
        ".fo-nttax-infobox",
        ".fo-nttax-infobox-wrapper",
        ".infobox",
        "table.infobox",
    ]

    for selector in selectors:
        box = soup.select_one(selector)
        if box:
            return clean_text(box.get_text(" ", strip=True))

    return ""

In [123]:
def extract_warcraft_people_from_team_html_v4(team_title, html):
    soup = BeautifulSoup(html, "lxml")
    rows = []

    race_terms = ["Human", "Orc", "Night Elf", "Undead", "Random"]

    bad_link_keywords = [
        "edit", "view", "history", "team", "teams", "player", "players",
        "tournament", "tournaments", "overview", "results", "matches",
        "statistics", "schedule", "vods", "references", "news",
        "main page", "warcraft", "liquipedia",
    ]

    stop_main_sections = [
        "Results",
        "Statistics",
        "Awards",
        "Logos",
        "Media",
        "References",
        "Gallery",
        "Upcoming Matches",
        "Upcoming Tournaments",
        "Timeline",
        "Achievements",
        "Earnings Breakdown",
    ]

    def get_prev_heading(el, levels):
        prev = el.find_previous(levels)
        if prev:
            return clean_text(prev.get_text(" ", strip=True)).replace("[edit]", "")
        return ""

    def get_main_section(el):
        return get_prev_heading(el, ["h2"])

    def get_sub_section(el):
        return get_prev_heading(el, ["h3", "h4"])

    def clean_ref_text(text):
        text = clean_text(text)
        text = re.sub(r"\[\d+\]", "", text)
        return clean_text(text)

    def extract_country_and_race_from_row(row_el):
        nationality = ""
        race = ""

        for img in row_el.select("img"):
            text = " ".join([
                clean_text(img.get("alt")),
                clean_text(img.get("title")),
                clean_text(img.get("src")),
            ])

            text_low = text.lower()

            if not nationality:
                if "south korea" in text_low:
                    nationality = "South Korea"
                elif "korea" in text_low:
                    nationality = "South Korea"

            if not race:
                for r in race_terms:
                    if r.lower().replace(" ", "") in text_low.replace(" ", ""):
                        race = r
                        break

        return nationality, race

    def normalize_header(h):
        h = clean_text(h).lower()

        if h in ["id", "player"]:
            return "id"
        if h in ["real name", "name"]:
            return "name"
        if h in ["race"]:
            return "race"
        if h in ["role", "position"]:
            return "position"
        if h in ["join date", "joined"]:
            return "join_date"
        if h in ["leave date", "left"]:
            return "leave_date"
        if h == "inactive date":
            return "inactive_date"
        if h in ["new team", "new_team"]:
            return "new_team"
        if h in ["country", "nationality"]:
            return "nationality"
        if h == "":
            return "position"

        return h.replace(" ", "_")

    def is_bad_person_link(name, href):
        name = clean_text(name)
        name_lower = name.lower()
        href_lower = href.lower()

        if not name:
            return True

        if name_lower in bad_link_keywords:
            return True

        if any(x.lower() in href_lower for x in [
            "/file:", "/category:", "/special:", "/help:", "/template:",
            "/portal:", "/liquipedia:", "/user:"
        ]):
            return True

        if len(name) > 80:
            return True

        if re.search(
            r"(season|stage|major|minor|playoffs|championship|cup|tournament|league|series|qualifier|finals|w3champions|dreamhack)",
            name_lower,
            flags=re.I
        ):
            return True

        return False

    def table_context(table):
        main_section = get_main_section(table)
        sub_section = get_sub_section(table)

        main_low = main_section.lower()
        sub_low = sub_section.lower()

        if any(s.lower() in main_low for s in stop_main_sections):
            return None, None, None

        # Warcraft는 Player Roster가 아니라 Squad인 경우가 많음
        if (
            "squad" in main_low
            or "player roster" in main_low
            or "former players" in sub_low
            or "players" in sub_low
        ):
            if "stand" in sub_low:
                return "player_roster", "Stand-in", main_section
            if "inactive" in sub_low:
                return "player_roster", "Inactive", main_section
            if "former" in sub_low:
                return "player_roster", "Former", main_section
            return "player_roster", "Active", main_section

        if "organization" in main_low or "staff" in main_low:
            if "former" in sub_low:
                return "organization", "Former", main_section
            return "organization", "Active", main_section

        return None, None, None

    def default_position(table_type, status, row_data):
        row_pos = row_data.get("position", "")

        if table_type == "organization":
            if row_pos:
                return row_pos
            return "Former Staff" if status == "Former" else "Staff"

        if table_type == "player_roster":
            if row_pos:
                return row_pos
            if status == "Stand-in":
                return "Stand-in"
            if status == "Inactive":
                return "Inactive Player"
            if status == "Former":
                return "Former Player"
            return "Player"

        return ""

    for table_idx, table in enumerate(soup.find_all("table")):
        table_type, status, main_section = table_context(table)

        if table_type is None:
            continue

        subheading = get_sub_section(table)
        trs = table.find_all("tr")

        if not trs:
            continue

        headers = []
        header_row_idx = None

        for i, tr in enumerate(trs[:10]):
            cells = tr.find_all(["th", "td"])
            cell_texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
            lowered = [c.lower() for c in cell_texts]

            if (
                "id" in lowered
                or "player" in lowered
                or "name" in lowered
                or "real name" in lowered
                or "race" in lowered
                or "join date" in lowered
                or "leave date" in lowered
                or "inactive date" in lowered
                or "organization" in lowered
            ):
                headers = cell_texts
                header_row_idx = i
                break

        if not headers:
            continue

        normalized_headers = [normalize_header(h) for h in headers]

        if len(normalized_headers) >= 4 and "position" not in normalized_headers:
            if normalized_headers[0] == "id" and normalized_headers[1] == "name":
                normalized_headers[2] = "position"

        for tr in trs[(header_row_idx + 1):]:
            cells = tr.find_all(["td", "th"])

            if not cells:
                continue

            row_text = clean_text(tr.get_text(" ", strip=True))

            if not row_text:
                continue

            if len(cells) == 1:
                continue

            row_data = {}
            local_headers = normalized_headers.copy()

            if len(cells) == len(local_headers) + 1:
                if "position" not in local_headers and "join_date" in local_headers:
                    join_idx = local_headers.index("join_date")
                    local_headers.insert(join_idx, "position")

            for idx, cell in enumerate(cells):
                if idx >= len(local_headers):
                    continue

                col = local_headers[idx]
                value = clean_text(cell.get_text(" ", strip=True))

                if col in ["join_date", "leave_date", "inactive_date"]:
                    value = clean_ref_text(value)

                row_data[col] = value

            links = []

            for a in tr.select(f'a[href^="/{GAME_SLUG}/"]'):
                link_text = clean_text(a.get_text(" ", strip=True))
                href = a.get("href", "")

                if is_bad_person_link(link_text, href):
                    continue

                links.append({
                    "name_from_link": link_text,
                    "page": href.split(f"/{GAME_SLUG}/", 1)[-1].replace("_", " "),
                    "url": "https://liquipedia.net" + href,
                })

            if links:
                main_link = links[0]
                person_id = row_data.get("id", "") or main_link["name_from_link"]
                person_page = main_link["page"]
                person_url = main_link["url"]
            else:
                person_id = row_data.get("id", "")
                person_page = ""
                person_url = ""

            if not person_id:
                continue

            if person_page.lower().replace(" ", "_") == team_title.lower().replace(" ", "_"):
                continue

            nationality_from_icon, race_from_icon = extract_country_and_race_from_row(tr)

            nationality = row_data.get("nationality", "") or nationality_from_icon
            race = row_data.get("race", "") or race_from_icon

            position = default_position(table_type, status, row_data)

            rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "game_title": "Warcraft III",
                "team_title": team_title,
                "team_url": make_page_url(team_title),
                "section_heading": main_section,
                "subheading": subheading,
                "table_type": table_type,
                "status": status,
                "table_index": table_idx,
                "id": person_id,
                "name": row_data.get("name", ""),
                "race": race,
                "position": position,
                "join_date": row_data.get("join_date", ""),
                "inactive_date": row_data.get("inactive_date", ""),
                "leave_date": row_data.get("leave_date", ""),
                "new_team": row_data.get("new_team", ""),
                "nationality": nationality,
                "person_page": person_page,
                "person_url": person_url,
                "row_text": row_text,
            })

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows)

    df = df.drop_duplicates(
        subset=[
            "team_title",
            "table_type",
            "status",
            "id",
            "name",
            "race",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "person_url",
        ]
    ).reset_index(drop=True)

    return df

In [124]:
test_team = "DN SOOPers"
page = get_page_html(test_team)

print("테스트 팀:", test_team)
print("페이지 존재 여부:", page["exists"])
print("에러:", page.get("error"))

if page["exists"]:
    test_people_df = extract_warcraft_people_from_team_html_v4(
        test_team,
        page["html"]
    )

    print("추출 row 수:", len(test_people_df))

    if len(test_people_df) > 0:
        display(test_people_df[[
            "team_title",
            "game_title",
            "table_type",
            "status",
            "id",
            "name",
            "race",
            "position",
            "join_date",
            "inactive_date",
            "leave_date",
            "new_team",
            "nationality",
            "person_url"
        ]].head(100))
    else:
        print("페이지는 열렸지만 row가 추출되지 않았습니다.")

테스트 팀: DN SOOPers
페이지 존재 여부: True
에러: None
추출 row 수: 1


,team_title,game_title,table_type,status,id,name,race,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,DN SOOPers,Warcraft III,player_roster,Active,Sok,Jung Ho-wook,Human,Player,2020-01-29,,,,South Korea,https://liquipedia.net/warcraft/Sok


In [125]:
def crawl_warcraft_teams_and_people_v4(candidate_df, output_prefix, team_type):
    team_rows = []
    people_dfs = []
    errors = []

    print("크롤링 대상 후보 수:", len(candidate_df))
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 60, 1), "분")
    print("예상 소요 시간:", round(len(candidate_df) * 31 / 3600, 2), "시간")

    for title in tqdm(candidate_df["title"], desc=f"{output_prefix} 팀 검증 + 인원 추출 v4"):
        page = get_page_html(title)

        if not page["exists"]:
            team_rows.append({
                "source_wiki": GAME_SLUG,
                "game_label": GAME_LABEL,
                "title": title,
                "display_title": page["display_title"],
                "url": page["url"],
                "exists": False,
                "is_korean_team_candidate": False,
                "infobox_text": "",
                "error": page["error"],
                "team_type": team_type,
            })
            continue

        infobox = extract_infobox_text(page["html"])
        infobox_lower = infobox.lower()
        infobox_compact = infobox_lower.replace(" ", "")

        full_text_lower = clean_text(
            BeautifulSoup(page["html"], "lxml").get_text(" ", strip=True)
        ).lower()

        is_team_page = "team information" in infobox_lower
        is_player_page = "player information" in infobox_lower

        is_korean_team = (
            is_team_page
            and not is_player_page
            and (
                "location:southkorea" in infobox_compact
                or "location: south korea" in infobox_lower
                or "south korea" in infobox_lower
                or "korean teams" in full_text_lower
                or "category:korean teams" in full_text_lower
            )
        )

        team_rows.append({
            "source_wiki": GAME_SLUG,
            "game_label": GAME_LABEL,
            "title": page["title"],
            "display_title": page["display_title"],
            "url": page["url"],
            "exists": True,
            "is_korean_team_candidate": is_korean_team,
            "infobox_text": infobox,
            "error": None,
            "team_type": team_type,
        })

        if is_korean_team:
            try:
                people_df = extract_warcraft_people_from_team_html_v4(
                    page["title"],
                    page["html"]
                )

                if len(people_df) > 0:
                    people_df["team_display_title"] = page["display_title"]
                    people_df["team_url"] = page["url"]
                    people_df["team_type"] = team_type
                    people_dfs.append(people_df)

            except Exception as e:
                errors.append({
                    "team_title": page["title"],
                    "error": str(e),
                    "team_type": team_type,
                })

    teams_checked_df = pd.DataFrame(team_rows)

    teams_final_df = teams_checked_df[
        (teams_checked_df["exists"] == True)
        & (teams_checked_df["is_korean_team_candidate"] == True)
    ].copy().reset_index(drop=True)

    if people_dfs:
        people_raw_df = pd.concat(people_dfs, ignore_index=True)
    else:
        people_raw_df = pd.DataFrame()

    errors_df = pd.DataFrame(errors)

    print("검증 완료")
    print("최종 한국 팀 수:", len(teams_final_df))
    print("추출 인원 raw row 수:", len(people_raw_df))
    print("에러 수:", len(errors_df))

    teams_checked_path = f"/content/{output_prefix}_teams_checked.csv"
    teams_final_path = f"/content/{output_prefix}_teams_final.csv"
    people_raw_path = f"/content/{output_prefix}_people_raw.csv"
    errors_path = f"/content/{output_prefix}_errors.csv"

    teams_checked_df.to_csv(teams_checked_path, index=False, encoding="utf-8-sig")
    teams_final_df.to_csv(teams_final_path, index=False, encoding="utf-8-sig")
    people_raw_df.to_csv(people_raw_path, index=False, encoding="utf-8-sig")
    errors_df.to_csv(errors_path, index=False, encoding="utf-8-sig")

    print("자동 저장 완료")
    print(teams_checked_path)
    print(teams_final_path)
    print(people_raw_path)
    print(errors_path)

    return teams_checked_df, teams_final_df, people_raw_df, errors_df

In [126]:
warcraft_teams_checked_df, warcraft_teams_final_df, warcraft_people_raw_df, warcraft_errors_df = crawl_warcraft_teams_and_people_v4(
    candidate_df=warcraft_dedup_df2,
    output_prefix="warcraft_korean_mainteam_refined_v4",
    team_type="main_team_refined"
)

display(warcraft_teams_final_df.head(100))
display(warcraft_people_raw_df.head(100))

크롤링 대상 후보 수: 73
예상 소요 시간: 37.7 분
예상 소요 시간: 0.63 시간


warcraft_korean_mainteam_refined_v4 팀 검증 + 인원 추출 v4:   0%|          | 0/73 [00:00<?, ?it/s]

검증 완료
최종 한국 팀 수: 19
추출 인원 raw row 수: 70
에러 수: 0
자동 저장 완료
/content/warcraft_korean_mainteam_refined_v4_teams_checked.csv
/content/warcraft_korean_mainteam_refined_v4_teams_final.csv
/content/warcraft_korean_mainteam_refined_v4_people_raw.csv
/content/warcraft_korean_mainteam_refined_v4_errors.csv


,source_wiki,game_label,title,display_title,url,exists,is_korean_team_candidate,infobox_text,error,team_type
0,warcraft,Warcraft III,All for One,All for One,https://liquipedia.net/warcraft/All_for_One,True,True,[ e ][ h ] All for One Team Information Clan t...,None,main_team_refined
1,warcraft,Warcraft III,South Korea,South Korea,https://liquipedia.net/warcraft/South_Korea,True,True,[ e ][ h ] South Korea Team Information Clan t...,None,main_team_refined
2,warcraft,Warcraft III,DRX,DRX,https://liquipedia.net/warcraft/DRX,True,True,[ e ][ h ] DRX Team Information Clan tag: DRX ...,None,main_team_refined
3,warcraft,Warcraft III,DN SOOPers,DN SOOPers,https://liquipedia.net/warcraft/DN_SOOPers,True,True,[ e ][ h ] DN SOOPers Team Information Clan ta...,None,main_team_refined
4,warcraft,Warcraft III,WeRRa,WeRRa,https://liquipedia.net/warcraft/WeRRa,True,True,[ e ][ h ] WeRRa Team Information Clan tag: We...,None,main_team_refined
5,warcraft,Warcraft III,Legend of,Legend of,https://liquipedia.net/warcraft/Legend_of,True,True,[ e ][ h ] Legend of Team Information Clan tag...,None,main_team_refined
6,warcraft,Warcraft III,FrienZ,FrienZ,https://liquipedia.net/warcraft/FrienZ,True,True,[ e ][ h ] Sonokong FrienZ Team Information Cl...,None,main_team_refined
7,warcraft,Warcraft III,WeMade FOX,WeMade FOX,https://liquipedia.net/warcraft/WeMade_FOX,True,True,[ e ][ h ] WeMade FOX Team Information Clan ta...,None,main_team_refined
8,warcraft,Warcraft III,SAINT,SAINT,https://liquipedia.net/warcraft/SAINT,True,True,[ e ][ h ] SAINT Team Information Clan tag: sa...,None,main_team_refined
9,warcraft,Warcraft III,Vision Strikers,Vision Strikers,https://liquipedia.net/warcraft/Vision_Strikers,True,True,[ e ][ h ] Vision Strikers Team Information Cl...,None,main_team_refined


,source_wiki,game_label,game_title,team_title,team_url,section_heading,subheading,table_type,status,table_index,...,join_date,inactive_date,leave_date,new_team,nationality,person_page,person_url,row_text,team_display_title,team_type
0,warcraft,Warcraft III,Warcraft III,DRX,https://liquipedia.net/warcraft/DRX,Player Roster,,player_roster,Active,0,...,2022-01-07,,,,South Korea,Moon,https://liquipedia.net/warcraft/Moon,Moon Jang Jae-ho 2022-01-07,DRX,main_team_refined
1,warcraft,Warcraft III,Warcraft III,DRX,https://liquipedia.net/warcraft/DRX,Player Roster,Former,player_roster,Former,1,...,,,2022-07-29,,South Korea,Lucifer,https://liquipedia.net/warcraft/Lucifer,LuChaeL Noh Jae-wook 2022-01-07 2022-07-29,DRX,main_team_refined
2,warcraft,Warcraft III,Warcraft III,DN SOOPers,https://liquipedia.net/warcraft/DN_SOOPers,Squad,Active,player_roster,Active,0,...,2020-01-29,,,,South Korea,Sok,https://liquipedia.net/warcraft/Sok,Sok Jung Ho-wook 2020-01-29,DN SOOPers,main_team_refined
3,warcraft,Warcraft III,Warcraft III,WeRRa,https://liquipedia.net/warcraft/WeRRa,Player Roster,Former,player_roster,Former,0,...,,,,Retired,South Korea,index.php?title=Star&action=edit&redlink=1,https://liquipedia.net/warcraft/index.php?titl...,Star Choi Won-il Retired,WeRRa,main_team_refined
4,warcraft,Warcraft III,Warcraft III,WeRRa,https://liquipedia.net/warcraft/WeRRa,Player Roster,Former,player_roster,Former,0,...,,,,,South Korea,WhO,https://liquipedia.net/warcraft/WhO,WhO Jang Du-sub,WeRRa,main_team_refined
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,warcraft,Warcraft III,Warcraft III,Motaesolo,https://liquipedia.net/warcraft/Motaesolo,Organization,Former,organization,Former,2,...,,,,,South Korea,index.php?title=CooLWiND&action=edit&redlink=1,https://liquipedia.net/warcraft/index.php?titl...,CooLWiND Streamer,Motaesolo,main_team_refined
66,warcraft,Warcraft III,Warcraft III,ARirang,https://liquipedia.net/warcraft/ARirang,Organization,Active,organization,Active,1,...,,,,,South Korea,index.php?title=HaN&action=edit&redlink=1,https://liquipedia.net/warcraft/index.php?titl...,HaN Manager,aRirang,main_team_refined
67,warcraft,Warcraft III,Warcraft III,ARirang,https://liquipedia.net/warcraft/ARirang,Organization,Active,organization,Active,1,...,,,,,South Korea,index.php?title=EnNaNo&action=edit&redlink=1,https://liquipedia.net/warcraft/index.php?titl...,EnNaNo Jun Young-sik Organizer,aRirang,main_team_refined
68,warcraft,Warcraft III,Warcraft III,ARirang,https://liquipedia.net/warcraft/ARirang,Organization,Active,organization,Active,1,...,,,,,,LovelySnOw,https://liquipedia.net/warcraft/LovelySnOw,trickSy Ross Jones Organizer,aRirang,main_team_refined


In [128]:
def make_unique_people_latest_from_raw(people_raw_df):
    """
    raw 데이터에서 사람 단위 unique 매핑.

    기준:
    1. person_url이 있으면 person_url 기준으로 같은 사람 묶음
    2. person_url이 없으면 id + name 기준으로 묶음
    3. join_date가 가장 최신인 row 우선
    4. join_date가 없으면 inactive_date, 그다음 leave_date를 대체 날짜로 사용
    5. 같은 날짜면 Stand-in > Inactive > Active > Former 우선
    6. 사람당 1행만 남김
    """

    if people_raw_df is None or len(people_raw_df) == 0:
        return pd.DataFrame()

    df = people_raw_df.copy()

    required_cols = [
        "person_url",
        "id",
        "name",
        "position",
        "status",
        "join_date",
        "inactive_date",
        "leave_date",
        "team_title",
        "team_url",
        "nationality",
        "new_team",
    ]

    for col in required_cols:
        if col not in df.columns:
            df[col] = ""

    df["person_unique_key"] = (
        df["person_url"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    empty_url_mask = df["person_unique_key"] == ""

    df.loc[empty_url_mask, "person_unique_key"] = (
        df.loc[empty_url_mask, "id"].fillna("").astype(str).str.lower().str.strip()
        + "|"
        + df.loc[empty_url_mask, "name"].fillna("").astype(str).str.lower().str.strip()
    )

    df = df[
        df["person_unique_key"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("|")
    ].copy()

    def extract_date(value):
        value = "" if pd.isna(value) else str(value)
        matched = re.search(r"(\d{4}-\d{2}-\d{2})", value)
        if matched:
            return matched.group(1)
        return ""

    df["join_date_clean"] = df["join_date"].apply(extract_date)
    df["inactive_date_clean"] = df["inactive_date"].apply(extract_date)
    df["leave_date_clean"] = df["leave_date"].apply(extract_date)

    df["join_date_parsed"] = pd.to_datetime(df["join_date_clean"], errors="coerce")
    df["inactive_date_parsed"] = pd.to_datetime(df["inactive_date_clean"], errors="coerce")
    df["leave_date_parsed"] = pd.to_datetime(df["leave_date_clean"], errors="coerce")

    df["effective_date_parsed"] = (
        df["join_date_parsed"]
        .fillna(df["inactive_date_parsed"])
        .fillna(df["leave_date_parsed"])
    )

    status_priority_map = {
        "Stand-in": 5,
        "Inactive": 4,
        "Active": 3,
        "Former": 2,
        "": 1,
    }

    df["status_priority"] = (
        df["status"]
        .fillna("")
        .astype(str)
        .str.strip()
        .map(status_priority_map)
        .fillna(1)
        .astype(int)
    )

    position_lower = df["position"].fillna("").astype(str).str.lower()

    df.loc[position_lower.str.contains("stand", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=5)

    df.loc[position_lower.str.contains("inactive", na=False), "status_priority"] = df[
        "status_priority"
    ].clip(lower=4)

    df = df.sort_values(
        by=[
            "person_unique_key",
            "effective_date_parsed",
            "join_date_parsed",
            "status_priority",
            "inactive_date_parsed",
            "leave_date_parsed",
            "team_title",
        ],
        ascending=[
            True,
            False,
            False,
            False,
            False,
            False,
            True,
        ],
        na_position="last"
    )

    unique_df = df.drop_duplicates(
        subset=["person_unique_key"],
        keep="first"
    ).reset_index(drop=True)

    unique_df = unique_df.drop(
        columns=[
            "join_date_clean",
            "inactive_date_clean",
            "leave_date_clean",
            "join_date_parsed",
            "inactive_date_parsed",
            "leave_date_parsed",
            "effective_date_parsed",
            "status_priority",
        ],
        errors="ignore"
    )

    return unique_df

In [129]:
warcraft_people_unique_df = make_unique_people_latest_from_raw(warcraft_people_raw_df)

print("팀 페이지 raw row 수:", len(warcraft_people_raw_df))
print("팀 페이지 최신 기준 unique 인원 수:", len(warcraft_people_unique_df))

display(warcraft_people_unique_df[[
    "game_title",
    "team_title",
    "status",
    "id",
    "name",
    "race",
    "position",
    "join_date",
    "inactive_date",
    "leave_date",
    "new_team",
    "nationality",
    "person_url"
]].head(100))

팀 페이지 raw row 수: 70
팀 페이지 최신 기준 unique 인원 수: 64


,game_title,team_title,status,id,name,race,position,join_date,inactive_date,leave_date,new_team,nationality,person_url
0,Warcraft III,Legend of,Former,Agatha,Han Seung-hak,Human,Former Player,,,,,South Korea,https://liquipedia.net/warcraft/Agatha
1,Warcraft III,WeRRa,Former,Anyppi,Im Hyo-jin,Night Elf,Former Player,,,,,South Korea,https://liquipedia.net/warcraft/Anyppi
2,Warcraft III,Legend of,Former,Azure,Jo Sung-hoon,Orc,Former Player,,,,,South Korea,https://liquipedia.net/warcraft/Azure
3,Warcraft III,Legend of,Former,Bany,Oh Je-min,Night Elf,Former Player,,,,,South Korea,https://liquipedia.net/warcraft/Bany
4,Warcraft III,Legend of,Former,Berzerker,Ho Young-seo,Orc,Former Player,,,,,South Korea,https://liquipedia.net/warcraft/Berzerker
...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,Warcraft III,Legend of,Former,rEN,Jun Sang-yool,,Former Player,,,,,South Korea,https://liquipedia.net/warcraft/index.php?titl...
60,Warcraft III,Legend of,Active,Rupture,Kim Hong-yul,,Manager,,,,,South Korea,https://liquipedia.net/warcraft/index.php?titl...
61,Warcraft III,ARirang,Active,sOOnMaNiAc,,,Organizer,,,,,,https://liquipedia.net/warcraft/index.php?titl...
62,Warcraft III,WeRRa,Former,Star,Choi Won-il,Undead,Former Player,,,,Retired,South Korea,https://liquipedia.net/warcraft/index.php?titl...


In [131]:
warcraft_teams_final_df.to_csv(
    "/content/warcraft_teams_final_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

warcraft_people_raw_df.to_csv(
    "/content/warcraft_people_raw_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

warcraft_people_unique_df.to_csv(
    "/content/warcraft_people_unique_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

warcraft_errors_df.to_csv(
    "/content/warcraft_extract_errors_v4.csv",
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")
print("/content/warcraft_teams_final_v4.csv")
print("/content/warcraft_people_raw_v4.csv")
print("/content/warcraft_people_unique_v4.csv")
print("/content/warcraft_extract_errors_v4.csv")

저장 완료
/content/warcraft_teams_final_v4.csv
/content/warcraft_people_raw_v4.csv
/content/warcraft_people_unique_v4.csv
/content/warcraft_extract_errors_v4.csv
